# 🧪 LabQA-RAG —《实验室智能问答系统项目全景说明书》

> **生成方式**：对代码仓库全量扫描 + Git 历史分析 + 联网检索高频面试题后自动生成
> **仓库**：`Lapmind/`（本地路径：`/Users/wuhang/Desktop/简历项目经历复盘总结/Lapmind`，pyproject name=`labqa-rag`）
> **版本**：v1.0 ｜ **日期**：2026-08-08

---

## 📌 超链接规则说明

本项目有 **Git 远程仓库**（4 个 commit，分支 `main`），因此全部源码引用统一采用 **GitHub 链接格式**：

```text
[文件](https://github.com/BLYHFL/labqa-rag/blob/main/{path}#L{行号})
```

- 指向 **类 / 函数 / 关键逻辑** 时保留 `#L行号`，例如：
  [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L71)
- 指向 **整个文件** 时省略 `#L行号`，例如：
  [src/labqa/retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py)
- **行号已经主控逐一核验**，文中全部行号可直接信任，无需重新核对
- **Gitee 镜像**：同一仓库同步在 [https://gitee.com/WZTWH/lab-qa.git](https://gitee.com/WZTWH/lab-qa.git)（国内访问加速用，内容与 GitHub 一致）

---

## 📑 目录

| 部分 | 内容 |
|------|------|
| 第一部分 | 项目概览 |
| 第二部分 | 技术栈与环境 |
| 第三部分 | 项目架构与目录结构 |
| 第四部分 | 源码导航索引 |
| 第五部分 | 本地运行与部署指南 |
| 第六部分 | 核心逻辑深度解析 |
| 第七部分 | 已知问题与调优经验 |
| 第八部分 | 待优化点与未来规划 |
| 第九部分 | 面试问题智能生成 |

> ⚠️ **注意**：本文档中的 Code 单元格，除标注「可直接运行」的以外，其余为**结构说明性代码**（依赖真实环境：飞书凭证、Ollama 本地模型、API Key、已构建的索引文件等），请勿在无环境时直接执行。


# 第一部分：项目概览

## 1.1 项目名称与简介

| 项目 | 内容 |
|------|------|
| **项目名称** | LabQA-RAG v2.0 — 实验室智能问答系统（`labqa-rag`） |
| **一句话定位** | 基于 LangChain + FAISS + BM25 + RRF 混合检索的 RAG 知识库问答系统，专为实验室（~20 人）场景设计 |
| **目标用户** | 实验室成员（设备管理、项目进度、知识文档三类高频诉求） |
| **核心功能** | ① 混合检索 RAG 问答（FAISS 语义 + BM25 关键词 + RRF 融合）② Multi-Agent 意图路由（Device / Project / Knowledge）③ 双轨 LLM（Ollama 本地免费 / 云端 DeepSeek、OpenAI）④ 飞书 WebSocket 长连接机器人（无需公网 IP）⑤ CLI 交互 / 单次问答 / 离线模式兜底 |
| **仓库远端** | `github=git@github.com:BLYHFL/labqa-rag.git`（main 分支）｜ `origin=https://gitee.com/WZTWH/lab-qa.git`（镜像） |

来源：[README.md](https://github.com/BLYHFL/labqa-rag/blob/main/README.md)、[pyproject.toml](https://github.com/BLYHFL/labqa-rag/blob/main/pyproject.toml)

### Git 历史（4 commits，v1 → v2 演进主线）

| commit | 日期 | 说明 |
|--------|------|------|
| `07bb9b1` | 2026-05-21 | **Initial commit（v1）**：关键词匹配 + `context_store.py` + `llm.py` + `webhook.py`，`labqa/` 包 |
| `e40c346` | 2026-06-05 | 删除博客模板.md |
| `757df23` | 2026-06-05 | **feat: LabQA-RAG v2.0 全面重构为混合检索 RAG 架构**（`labqa/` → `src/labqa/`，新增 ingest.py / retriever.py / generator.py / prompts.py / tests，18 个测试用例） |
| `48720ed` | 2026-06-05 | docs: WORKFLOW.md（506 行问答流程详解） |

> 💡 v1 → v2 重构动机：v1 纯关键词匹配召回差 → v2 全面重构为 **FAISS（稠密）+ BM25（稀疏）+ RRF 融合** 的混合检索 RAG 架构（见 `757df23`）。

## 1.2 系统架构概述

```text
┌──────────────────────────────────────────────────────────────────────────┐
│  交互层  │  CLI（python3 main.py cli / ask）  飞书客户端（WS 长连接）        │
│          │  OpenCode 斜杠命令（/查设备 · /查项目 · /查文档）                 │
└──────────────┬───────────────────────────────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────────────────────────────┐
│  Agent 层  │  Orchestrator 主调度器（process）                             │
│            │    └→ Router 关键词意图识别（DEVICE / PROJECT / KNOWLEDGE /   │
│            │       UNKNOWN，含 MIXED 混合意图）                            │
│            │    └→ DeviceAgent / ProjectAgent / KnowledgeAgent（category   │
│            │       过滤 → 检索 → LLM 生成）                                 │
└──────────────┬───────────────────────────────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────────────────────────────┐
│  检索层  │  FAISS 稠密检索（语义）  +  BM25 稀疏检索（jieba 中文分词）        │
│          │         ↓                                                      │
│          │   RRF 融合（1/(60+rank)） → 去重 → Top-K 文档                    │
│          │   （BM25 无结果时降级为纯向量检索）                                │
└──────────────┬───────────────────────────────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────────────────────────────┐
│  知识库层  │  .opencode/context/*.md（8 个文档，YAML frontmatter）           │
│           │  faiss_index/index.faiss + bm25_index.pkl（运行时 ingest 生成）  │
└──────────────────────────────────────────────────────────────────────────┘
```

**分层职责**：

| 层 | 职责 | 关键组件 |
|----|------|----------|
| 交互层 | CLI 交互、单次问答、飞书消息收发、OpenCode 斜杠命令 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L46)、[cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62)、[feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93) |
| Agent 层 | 意图识别、Agent 路由、检索调度、Prompt 构建、LLM 生成 | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L15)、[router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150)、[agents/](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L15) |
| 检索层 | 稠密/稀疏双路检索 + RRF 融合 + 分类过滤 | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175) |
| 知识库层 | 知识文档存储、索引文件持久化 | [.opencode/context/](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/context/navigation.md)、[ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L200) |

### 双执行面架构（重要卖点）

| 执行面 | 位置 | 入口 | 说明 |
|--------|------|------|------|
| ① Python 运行时 | `src/labqa/` | `main.py`（cli / ask / feishu / ingest / stats） | 独立可运行的 CLI / 飞书 WS / 单次问答 |
| ② OpenCode Agent 定义 | `.opencode/agent/` | `lab-orchestrator` + 3 个 subagents + `commands/` + `workflows/` | OpenCode 会话内的斜杠命令（/查设备 /查项目 /查文档） |

**两个执行面共享同一知识库**（`.opencode/context/`），任何一面对知识文档的修改对双方同时生效。

## 1.3 核心模块组成

| 模块 | 文件 | 功能描述 |
|------|------|----------|
| **统一入口** | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L114) | 5 种运行模式：`cli` L46 / `ingest` L52 / `ask` L58 / `feishu` L91 / `stats` L97；手写 .env 加载器 L26-L41 |
| **主调度器** | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L15) | `initialize` L25-53 启动装配、`process` L71-130 意图→Agent→LLM→回答、`stats` L167-176、单例 `get_orchestrator` L191-196 |
| **意图路由** | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150) | 关键词打分 `recognize_intent` L150-207；DEVICE/PROJECT/KNOWLEDGE/UNKNOWN 四类意图（Intent L16-21），次要意图分数 ≥ 主意图 50% 时判 MIXED |
| **混合检索** | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175) | `dense_search` L19-52（FAISS 语义）+ `sparse_search` L57-101（BM25 关键词）+ `reciprocal_rank_fusion` L106-142（RRF 融合去重）+ `hybrid_search` L175-229 |
| **索引构建** | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L200) | 文档加载 L26-83、中文分块 L86-110（RecursiveCharacterTextSplitter 500/50）、FAISS 索引 L113-144、BM25 索引 L147-197、`check_indexes` L222-235 |
| **LLM 适配** | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99) | `OllamaEmbeddings` L31-43 / `OllamaChat` L48-74（本地轨）、`create_llm` L99-118（云端轨）、`generate_answer` L123-199 |
| **Prompt 模板** | [prompts.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L6) | `DEFAULT_RAG_PROMPT` L6 / `DEVICE_PROMPT` L23 / `PROJECT_PROMPT` L42 / `KNOWLEDGE_PROMPT` L62 / `UNKNOWN_INTENT_PROMPT` L81 / `get_agent_prompt` L93-100 |
| **CLI** | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62) | 交互式命令行（BANNER L18-27、HELP_TEXT L29-59），支持 `/help` L108 / `/stats` L110 / `/reload` L120 / `/ingest` L123 / `/debug` L131 |
| **飞书机器人** | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93) | lark-oapi WebSocket 长连接 `start` L93-180；两层去重 `_is_duplicate` L28-56；消息 19900 字符截断 L83 |
| **配置管理** | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25) | LLM 双轨 L25、API Key 三级回退 L32-35、检索参数 L42-46、飞书凭证 L49-51、`validate` L57-70 |
| **Agent 层** | [agents/](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/__init__.py#L14) | `BaseAgent` L15-121（search_context L35-69 / answer L71-107）；DeviceAgent L10-15（category="devices"）、ProjectAgent L9-14（category="projects"）、KnowledgeAgent L9-14（category=None 全量）；`AGENT_REGISTRY` L14-18 |
| **测试** | [tests/test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L17) | pytest，7 个 Test 类（TestConfig L17 / TestIngest L39 / TestRouter L89 / TestRRF L157 / TestPrompts L203 / TestGenerator L247 / TestAgents L284），约 18 用例 |

## 1.4 核心指标

| 指标 / 特性 | 数值 / 说明 | 来源 |
|-------------|-------------|------|
| **混合检索** | FAISS 稠密（语义）+ BM25 稀疏（关键词，jieba 分词）+ RRF（1/(60+rank)）融合去重 | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106) |
| **检索降级** | BM25 无结果时自动降级为纯向量检索 | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L212) |
| **v1 → v2 演进** | 关键词匹配 → 混合检索 RAG 全面重构（commit `757df23`） | [README.md](https://github.com/BLYHFL/labqa-rag/blob/main/README.md) |
| **双执行面** | Python 运行时（src/labqa/）+ OpenCode Agent 定义（.opencode/agent/）共享同一知识库 | [AGENTS.md](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md) |
| **飞书 WS 长连接** | lark-oapi WebSocket 模式，**无需公网 IP**（Webhook 需公网回调，WS 模式规避） | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93) |
| **消息去重** | 两层去重补偿 SDK 重试：message_id 集合 + 30 秒内容哈希窗口 | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28) |
| **中文优化分块** | RecursiveCharacterTextSplitter，chunk 500 / overlap 50 | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L86) |
| **双轨 LLM** | Ollama 本地 `deepseek-r1:1.5b`（免费）/ 云端 DeepSeek、OpenAI | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L48) |
| **API Key 三级回退** | `LABQA_LLM_API_KEY` → `DEEPSEEK_API_KEY` → `OPENAI_API_KEY` | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32) |
| **Multi-Agent 路由** | Device / Project / Knowledge 三类 Agent 按 category 过滤检索范围 | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L72) |
| **混合意图识别** | 次要意图分数 ≥ 主意图 50% → MIXED | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L176) |
| **CLI 离线模式** | 无 API Key 时仅关键词检索，不调 LLM（哨兵 `LABQA_LLM_API_KEY="offline-mode"`） | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62) |
| **索引运行时生成** | `python3 main.py ingest` 构建 `faiss_index/index.faiss` + `bm25_index.pkl`，已存在则加载不重建 | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L222) |
| **知识库规模** | `.opencode/context/` 8 个 Markdown 文档（devices 4 / projects 3 / knowledge 1 / guides 2），`navigation.md` 被检索排除 | [.opencode/context/navigation.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/context/navigation.md) |
| **测试覆盖** | tests/test_main.py 320 行，7 个 Test 类，约 18 用例（pytest）；无 CI | [tests/test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py) |


# 第二部分：技术栈与环境

## 2.1 核心依赖表（来自 [pyproject.toml](https://github.com/BLYHFL/labqa-rag/blob/main/pyproject.toml)，uv 管理）

| 库名称 | 版本 | 用途 |
|--------|------|------|
| Python | 3.9+ | 解释器（README badge） |
| langchain | 0.3+ | RAG 框架：文档分块（RecursiveCharacterTextSplitter）、Embedding 封装、Prompt 模板 |
| faiss-cpu | 1.8+ | FAISS 稠密向量索引（`build_faiss_index` / `dense_search`） |
| rank-bm25 | 0.2.2+ | BM25 稀疏检索（`build_bm25_index` / `sparse_search`），配合 jieba 中文分词 |
| ollama | 0.4+ | 本地 LLM / Embedding 调用（`OllamaChat` / `OllamaEmbeddings`） |
| lark-oapi | 1.6+ | 飞书开放平台 SDK：WebSocket 长连接客户端（`feishu_ws.py`） |
| httpx | 0.25+ | HTTP 客户端：云端 LLM API 调用（OpenAI 兼容 /v1/chat/completions） |
| jieba | - | BM25 中文分词（稀疏检索关键词切分） |
| python-dotenv | - | 配置加载（`config.py` 读取 .env；`main.py` 另有一套手写加载器 L26-L41） |
| pytest | dev | 测试框架（`tests/test_main.py`，约 18 用例） |
| python-docx / python-pptx / openpyxl | 可选 | `scripts/convert.sh` 文档转换（raw-docs/ → context/，仅在运行转换脚本时需要） |

**包管理**：`uv`（`uv venv --python 3.9` + `uv pip install -e ".[dev]"`，见 [README.md](https://github.com/BLYHFL/labqa-rag/blob/main/README.md) 快速开始）。

## 2.2 双轨 LLM 模式

系统支持 **本地 Ollama** 与 **云端 API** 两条 LLM 轨道，通过 `LABQA_LLM_MODE=ollama|cloud` 切换（[config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25)）。

| 维度 | Ollama 本地模式 | 云端模式 |
|------|----------------|----------|
| 环境变量 | `LABQA_LLM_MODE=ollama`（默认） | `LABQA_LLM_MODE=cloud` |
| LLM 模型 | 本地 `deepseek-r1:1.5b`（`ollama pull deepseek-r1:1.5b`） | DeepSeek / OpenAI 兼容 API（`deepseek-chat` 等） |
| Embedding | 本地 `nomic-embed-text`（`ollama pull nomic-embed-text`） | OpenAI 兼容 Embedding 接口 |
| 成本 | 免费、离线可用、无数据外发 | 按量计费，回答质量更高 |
| 适用场景 | 开发调试、内网环境 | 生产部署、高质量回答 |
| 源码 | [generator.py L31-L43](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L31)（Embeddings）、[L48-L74](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L48)（Chat） | [generator.py L99-L118](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99)（create_llm） |

**API Key 三级回退**（[config.py L32-L35](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32)）：`LABQA_LLM_API_KEY` → `DEEPSEEK_API_KEY` → `OPENAI_API_KEY`，任意一个存在即可。

**离线兜底**：CLI 模式下若无 API Key，自动进入离线模式（哨兵值 `LABQA_LLM_API_KEY="offline-mode"`），仅做关键词检索、不发起任何 LLM 调用（见 [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62)）。

## 2.3 环境变量表

模板见 [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example)（34 行：`LABQA_LLM_MODE=ollama` 默认 L6-8 / cloud 模式 L11-16 / 检索参数 L19-23 / 飞书 L26-31）。`main.py` 使用**手写加载器**读取 `.env`（[main.py L26-L41](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L26)），shell 环境变量优先于 `.env` 值；`config.py` 则使用 python-dotenv。

| 变量 | 必填 | 用途 | 敏感 |
|------|------|------|------|
| `LABQA_LLM_MODE` | 否（默认 ollama） | LLM 双轨切换：`ollama`（本地）/ `cloud`（云端） | 否 |
| `LABQA_LLM_API_KEY` | cloud 模式 ✅ | 云端 LLM API Key（三级回退第一级） | 🔒 是 |
| `DEEPSEEK_API_KEY` | 回退 | DeepSeek API Key（三级回退第二级） | 🔒 是 |
| `OPENAI_API_KEY` | 回退 | OpenAI API Key（三级回退第三级） | 🔒 是 |
| `LABQA_LLM_API_BASE` | 否 | API Base URL（默认 `https://api.deepseek.com/v1`；若已含 `/v1` 则不再重复拼接） | 否 |
| `OLLAMA_LLM_MODEL` | ollama 模式 ✅ | 本地 LLM 模型名（默认 `deepseek-r1:1.5b`） | 否 |
| OLLAMA_*（Embedding 模型等） | ollama 模式 | 本地 Embedding 模型（README 推荐 `nomic-embed-text`）等配置，见 [config.py L28-L29](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L28) | 否 |
| `FEISHU_APP_ID` | feishu 模式 ✅ | 飞书应用凭证 App ID（WS 长连接认证） | 否 |
| `FEISHU_APP_SECRET` | feishu 模式 ✅ | 飞书应用凭证 App Secret | 🔒 是 |
| `RETRIEVE_K` | 否 | 单路检索返回文档数（dense / sparse 各取） | 否 |
| `FUSION_K` | 否 | RRF 融合后最终取 Top-K 文档数 | 否 |
| `RRF_K` | 否 | RRF 平滑常数（`1/(K+rank)` 中的 K） | 否 |
| `CHUNK_SIZE` | 否 | 中文分块大小（默认 500，配合 overlap 50） | 否 |
| `CHUNK_OVERLAP` | 否 | 分块重叠窗口（默认 50） | 否 |
| `LABQA_DEBUG` | 否 | 调试日志开关（`true` 时输出 router / 检索 / LLM 详细日志） | 否 |

> 检索参数集中在 [config.py L42-L46](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L42)，飞书凭证在 [config.py L49-L51](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L49)，启动时 `validate`（[L57-L70](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L57)）校验必填项。


In [ ]:
# 环境检查示例（可直接运行，验证 Python 版本 + 核心依赖 + 索引 + .env 完整性）
import sys
from pathlib import Path

print("Python 版本:", sys.version.split()[0])
print()

# 1. 检查核心依赖（langchain / faiss / rank_bm25 / lark_oapi / ollama / httpx）
deps = ["langchain", "faiss", "rank_bm25", "lark_oapi", "ollama", "httpx", "jieba", "dotenv"]
for pkg in deps:
    try:
        mod = __import__(pkg)
        print(f"  ✅ {pkg:14s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  ❌ {pkg:14s} 未安装")

# 2. 检查索引文件（需要先运行 python3 main.py ingest 生成）
print()
print("索引文件检查:")
for f in [Path("faiss_index/index.faiss"), Path("bm25_index.pkl")]:
    print(f"  {'✅' if f.exists() else '❌'} {f}")

# 3. 检查 .env 与关键配置（敏感值脱敏显示）
print()
if Path(".env").exists():
    print("  ✅ .env 存在")
    import os
    for var in ["LABQA_LLM_MODE", "LABQA_LLM_API_KEY", "OLLAMA_LLM_MODEL",
                "FEISHU_APP_ID", "FEISHU_APP_SECRET"]:
        v = os.getenv(var)
        shown = "已配置" if v else "❌ 未配置"
        if v and var in ("FEISHU_APP_SECRET", "LABQA_LLM_API_KEY"):
            shown = "已配置（敏感，隐藏）"
        print(f"     {var}: {shown}")
else:
    print("  ❌ .env 不存在，请先执行 cp .env.example .env")


# 第三部分：项目架构与目录结构

LabQA-RAG v2.0 采用**双执行面架构**：一个 **Python 运行时**（`src/labqa/`，命令行 / 飞书 / 单次问答）与一套 **OpenCode Agent 定义**（`.opencode/agent/`，会话内斜杠命令）共享**同一个文件型知识库** `.opencode/context/`。知识库即 Markdown 文件，**无需数据库**——这是整个项目在"架构简洁性"上最核心的设计决定。

## 3.1 目录结构树

```text
Lapmind/  (LabQA-RAG v2.0, pyproject name = labqa-rag)
│
├── main.py                       # 🚀 统一入口：mode_cli / mode_ingest / mode_ask / mode_feishu / mode_stats（main L114-149）
├── pyproject.toml                # 📦 项目元数据 + pytest 配置（name = labqa-rag）
├── requirements.txt              # 运行时依赖：httpx、lark-oapi（文档转换依赖被注释，见第七部分工程债）
├── README.md                     # 项目总览 + 快速开始 + 扩展方向
├── QUICK-START.md                # 5 分钟上手：安装 → ingest → ask
├── AGENTS.md                     # Agent 指南（v1 时代的架构描述，部分内容已落后于 v2，见工程债）
├── .env.example                  # 配置模板：LLM 模式 / 检索参数 / 飞书凭据（34 行）
├── .gitignore                    # 忽略 .env、faiss_index/、bm25_index.pkl、.venv/、node_modules/ 等
│
├── src/labqa/                    # 🐍 Python 运行时（v2 包，自 757df23 起由 labqa/ 迁入）
│   ├── __init__.py
│   ├── config.py                 #   全部配置：路径常量 / 双轨 LLM / 检索参数 / 飞书（validate L57-70）
│   ├── router.py                 #   关键词意图识别：Intent 枚举 + 打分路由（recognize_intent L150-207）
│   ├── orchestrator.py           #   总调度器：initialize → process → 意图分发 → 响应（process L71-130）
│   ├── ingest.py                 #   知识库入库：加载 → 中文分块 → FAISS + BM25 索引构建（main L239-273）
│   ├── retriever.py              #   混合检索：dense + sparse + RRF 融合（hybrid_search L175-229）
│   ├── generator.py              #   双轨 LLM：Ollama / 云端 + 按 Agent 选 Prompt（generate_answer L123-199）
│   ├── prompts.py                #   RAG / 设备 / 项目 / 知识 / 兜底 5 套 Prompt 模板
│   ├── cli.py                    #   终端交互界面：BANNER + /help /stats /reload /ingest /debug
│   ├── feishu_ws.py              #   飞书 WebSocket 长连接（lark-oapi，免公网 IP，两层去重）
│   └── agents/                   #   Multi-Agent 后端（v2 新增）
│       ├── __init__.py           #     AGENT_REGISTRY 注册表 + get_agent 工厂
│       ├── base.py               #     BaseAgent：检索依赖注入 + answer 模板方法
│       ├── device_agent.py       #     DeviceAgent（category="devices"）
│       ├── project_agent.py      #     ProjectAgent（category="projects"）
│       └── knowledge_agent.py    #     KnowledgeAgent（category=None，全量检索）
│
├── tests/                        # 🧪 pytest 测试（v2 新增，7 个 Test 类约 18 用例）
│   ├── __init__.py
│   └── test_main.py              #   Config / Ingest / Router / RRF / Prompts / Generator / Agents（320 行）
│
├── scripts/                      # 🛠 运维脚本
│   ├── convert.sh                #   raw-docs/ → context/ 批量转换：docx/pptx/xlsx/pdf + frontmatter 自动补全
│   └── feishu-webhook.py         #   ⚠️ legacy 独立脚本：自带关键词匹配，绕过 labqa/，不用于新开发
│
├── docs/                         # 📄 设计文档
│   ├── ARCHITECTURE.md           #   架构说明（270 行，含 ADR-001 Markdown 统一格式 / ADR-002 Agent 体系）
│   ├── WORKFLOW.md               #   问答流程详解（506 行：Router 打分 / 混合意图 / 完整示例走读）
│   └── TESTING.md                #   手动测试清单（152 行，8 大节）
│
├── .opencode/                    # 🤖 OpenCode Agent 定义层（第二个执行面）
│   ├── agent/
│   │   ├── lab-orchestrator.md   #   主调度器：意图识别 + 路由规则 + workflow（128 行）
│   │   └── subagents/
│   │       ├── device-agent.md   #   设备查询 Agent（读 context/devices/）
│   │       ├── project-agent.md  #   项目查询 Agent（读 context/projects/）
│   │       └── knowledge-agent.md#   知识查询 Agent（读 context/knowledge/ + guides/）
│   ├── commands/                 #   💬 斜杠命令（查设备.md / 查项目.md / 查文档.md / 更新知识库.md）
│   ├── workflows/                #   🔄 流程定义（answer-query.md / update-knowledge.md）
│   ├── context/                  #   📚 共享知识库（文件即数据库，两个执行面共用）
│   │   ├── navigation.md         #     知识库总索引（60 行，ingest 按文件名跳过，不参与检索）
│   │   ├── devices/              #     设备资产：设备总览 / GPU-A100-01 / GPU-H100-01 / 自动驾驶测试车（4 个）
│   │   ├── projects/             #     项目管理：项目总览 / LLM对齐优化项目 / 自动驾驶感知路测（3 个）
│   │   ├── knowledge/论文笔记/   #     论文笔记：DPO论文笔记.md（+ README.md 辅助说明）
│   │   └── guides/               #     流程规范：新人入职指南 / 代码提交规范（2 个）
│   ├── navigation.md             #   系统导航（50 行：组件 / 命令 / 工作流索引，面向 Agent 会话）
│   └── package.json / node_modules/  # OpenCode 插件运行时（node_modules 已 gitignore）
│
├── data/                         # 📥 预留数据目录（当前为空，可放导出/统计产物）
├── raw-docs/                     # 📄 文档入库输入区（convert.sh 的源目录，当前为空）
│
├── faiss_index/                  # 🔍 运行时生成（gitignore）：index.faiss 稠密向量索引
├── bm25_index.pkl                # 🔍 运行时生成（gitignore）：BM25 稀疏索引 pickle
└── .venv/                        # 🐍 Python 虚拟环境
```

**目录树速读**：整个仓库只分四类东西——

1. **执行面**：`src/labqa/`（Python 运行时）+ `.opencode/`（OpenCode Agent 定义）
2. **知识层**：`.opencode/context/`（唯一的领域事实来源，**无数据库**）
3. **构建产物**：`faiss_index/` + `bm25_index.pkl`（`python3 main.py ingest` 从知识库现场生成，已 gitignore）
4. **支撑**：`scripts/`（文档转换）、`docs/`（设计文档）、`tests/`（回归测试）


## 3.2 各模块职责说明

### 3.2.1 Python 运行时（src/labqa/）

| 模块 | 关键行 | 职责 | 上下游 |
|------|--------|------|--------|
| `config.py` | [validate](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L57) L57-70 | 全部配置入口：`PROJECT_ROOT`/`CONTEXT_DIR`/索引路径常量、`LABQA_LLM_MODE` 双轨切换、`LLM_API_KEY` 三级回退（`LABQA_LLM_API_KEY` → `DEEPSEEK_API_KEY` → `OPENAI_API_KEY`，[L32-35](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32)）、检索参数（CHUNK_SIZE/CHUNK_OVERLAP/RETRIEVE_K，[L42-46](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L42)）、飞书凭据；启动即校验 | 被所有模块 import |
| `router.py` | [recognize_intent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150) L150-207 | 关键词打分意图识别：`DEVICE_KEYWORDS`/`PROJECT_KEYWORDS`/`KNOWLEDGE_KEYWORDS` 三组硬编码关键词（[L35-69](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L35)）+ 分组打分 `_score_intent`（[L132-147](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L132)）；输出 `Intent` 枚举（DEVICE/PROJECT/KNOWLEDGE/MIXED/UNKNOWN）+ `RoutingDecision`；**零 LLM 调用、零成本** | orchestrator 调用 |
| `orchestrator.py` | [process](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L71) L71-130 | 总调度器：`initialize`（[L25-53](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L25)）装载索引与 Agent → `process` 按意图查 Agent → 拼装响应；`_unknown_intent_response` 兜底（[L132-165](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L132)）；`get_orchestrator` 全局单例（[L191-196](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L191)） | 入口唯一面向 CLI/飞书的调度层 |
| `ingest.py` | [main](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L239) L239-273 | 知识库入库：`load_markdown_files`（[L26-83](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L26)）递归扫描 + 手写 frontmatter 解析 + **按文件名跳过 navigation.md**（L39-40）→ `split_documents`（[L86-110](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L86)，RecursiveCharacterTextSplitter 500/50 + 中文分隔符）→ FAISS（[L113-144](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L113)）/ BM25（[L147-197](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L147)，jieba 分词可选，缺失退化为字符级）；索引已存在则加载不重建（L125、L158）；支持 `--check`/`--stats`/`--rebuild` | 依赖 generator.create_embeddings |
| `retriever.py` | [hybrid_search](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175) L175-229 | 混合检索核心：FAISS 稠密（[dense_search L19-52](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L19)）+ BM25 稀疏（[sparse_search L57-101](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L57)）→ RRF `1/(60+rank)` 融合（[L106-142](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106)）；**BM25 无结果时降级为纯向量检索**（L212-216）；`search_by_category` 支持按 category 过滤（[L232-245](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L232)） | Agent 检索后端 |
| `generator.py` | [generate_answer](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L123) L123-199 | 双轨 LLM：Ollama（本地免费，[L31-74](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L31)）/ 云端 OpenAI 兼容（DeepSeek/OpenAI）；`create_llm` 按 `LABQA_LLM_MODE` 切换（[L99-118](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99)）；按 Agent 类型选用 `AGENT_PROMPTS`（[L222-274](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L222)） | Agent 生成后端 |
| `prompts.py` | [get_agent_prompt](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L93) L93-100 | 5 套 Prompt：`DEFAULT_RAG_PROMPT`（L6）/ `DEVICE_PROMPT`（L23）/ `PROJECT_PROMPT`（L42）/ `KNOWLEDGE_PROMPT`（L62）/ `UNKNOWN_INTENT_PROMPT`（L81） | 被 generator 引用 |
| `cli.py` | [main](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62) L62-149 | 终端交互 UI：BANNER 启动画（[L18-27](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L18)）、内置命令 `/help` `/stats` `/reload` `/ingest` `/debug`；**离线模式兜底**（无 API Key 时纯关键词检索） | 由 main.py mode_cli 拉起 |
| `feishu_ws.py` | [start](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93) L93-180 | 飞书 WebSocket 长连接（lark-oapi SDK，**免公网 IP**）：`on_message` 回调（[L124-160](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L124)）；**两层去重**补偿 SDK 重试——message_id 集合 + 30 秒内容哈希窗口（[_is_duplicate L28-56](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28)）；消息 19900 字符截断（L83）；WS 客户端构建（[L162-169](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L162)） | main.py mode_feishu；`webhook` 别名同样指向此 WS |
| `agents/base.py` | [BaseAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L15) L15-121 | Agent 基类：`set_dependencies` 注入检索/生成器（[L29-33](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L29)）→ `search_context`（[L35-69](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L35)）→ `answer` 模板方法（[L71-107](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L71)）→ `_no_result_response` 无结果兜底（[L109-121](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L109)） | 三个 Agent 的父类 |
| `agents/device_agent.py` | [DeviceAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/device_agent.py#L10) L10-15 | 设备意图的"专业后端"：`category="devices"`，检索只命中设备域 | 被 orchestrator 实例化 |
| `agents/project_agent.py` | [ProjectAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/project_agent.py#L9) L9-14 | 项目意图后端：`category="projects"` | 被 orchestrator 实例化 |
| `agents/knowledge_agent.py` | [KnowledgeAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/knowledge_agent.py#L9) L9-14 | 知识意图后端：`category=None`（全量检索 knowledge + guides） | 被 orchestrator 实例化 |
| `agents/__init__.py` | [AGENT_REGISTRY](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/__init__.py#L14) L14-18 | Agent 注册表 + `get_agent` 工厂（[L21-25](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/__init__.py#L21)），新增 Agent 只需注册一处 | orchestrator 依赖 |

### 3.2.2 入口层（main.py）

| 模式 | 行号 | 说明 |
|------|------|------|
| 手写 .env 加载器 | [L26-41](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L26) | 不依赖 python-dotenv（config.py 内另有 dotenv 实现，属双实现工程债）；shell 环境变量优先 |
| `sys.path` 注入 | [L24](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L24) | 免安装直接运行 `src/labqa` |
| `mode_cli` | [L46](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L46) | 终端交互模式（开发推荐） |
| `mode_ingest` | [L52](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L52) | 构建/重建索引 |
| `mode_ask` | [L58](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L58) | 单次问答（脚本化） |
| `mode_feishu` | [L91](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L91) | 飞书 WS 长连接（生产推荐，免公网 IP） |
| `mode_stats` | [L97](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L97) | 索引/知识库统计 |
| `main` | [L114-149](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L114) | 子命令分发 |

### 3.2.3 OpenCode Agent 定义层（.opencode/）

| 定义 | 关键行 | 职责 |
|------|--------|------|
| `agent/lab-orchestrator.md` | [intent_recognition L31-51](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/lab-orchestrator.md#L31) / [routing_rules L53-61](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/lab-orchestrator.md#L53) / [workflow L96-123](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/lab-orchestrator.md#L96) | 与 Python `orchestrator.py` 对应的 Agent 版总调度：意图识别 → 路由 → 分发 subagent → 汇总 |
| `agent/subagents/device-agent.md` | — | 设备域 Agent（读 `context/devices/`），对应 `device_agent.py` |
| `agent/subagents/project-agent.md` | — | 项目域 Agent（读 `context/projects/`），对应 `project_agent.py` |
| `agent/subagents/knowledge-agent.md` | — | 知识域 Agent（读 `context/knowledge/` + `guides/`），对应 `knowledge_agent.py` |
| `commands/查设备.md` 等 4 个 | — | OpenCode 斜杠命令：`/查设备` `/查项目` `/查文档` `/更新知识库` |
| `workflows/answer-query.md` | — | 问答流程定义（OpenCode 版） |
| `workflows/update-knowledge.md` | — | 知识库更新流程定义（对应 `scripts/convert.sh` + `main.py ingest` 链路） |
| `navigation.md` | — | 系统导航（组件/命令/工作流索引），供 Agent 会话快速定位 |

### 3.2.4 支撑层（tests / scripts / docs）

| 模块 | 关键行 | 职责 |
|------|--------|------|
| `tests/test_main.py` | [TestConfig L17](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L17) / [TestIngest L39](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L39) / [TestRouter L89](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L89) / [TestRRF L157](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L157) / [TestPrompts L203](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L203) / [TestGenerator L247](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L247) / [TestAgents L284](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L284) | pytest 回归：配置校验、入库加载/分块、意图识别、RRF 融合、Prompt 选择、LLM 构建、Agent 行为 |
| `scripts/convert.sh` | [主循环 L227-295](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L227) | 文档批量转换：docx（L41）/ pptx（L73）/ xlsx（L109）/ pdf（L141，占位）/ markdown 直拷（L152）；按文件名关键词自动分类到 devices/projects/guides/knowledge（[classify_target L163-187](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L163)）；缺失时自动补 frontmatter（[add_frontmatter L191-216](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L191)） |
| `scripts/feishu-webhook.py` | — | **legacy 独立脚本**：自带关键词匹配、绕过 `labqa/`，仅作历史参考，不用于新开发 |
| `docs/ARCHITECTURE.md` | [ADR-001 L221](https://github.com/BLYHFL/labqa-rag/blob/main/docs/ARCHITECTURE.md#L221) / [ADR-002 L235](https://github.com/BLYHFL/labqa-rag/blob/main/docs/ARCHITECTURE.md#L235) | 架构说明 + ADR 记录（Markdown 统一格式、.opencode Agent 体系） |
| `docs/WORKFLOW.md` | [Router 打分 L147-182](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L147) / [混合意图 L176-182](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L176) / [完整示例 L186-416](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L186) | 506 行问答流程详解，含真实示例走读 |
| `docs/TESTING.md` | — | 152 行手动测试清单（8 大节） |

### 3.2.5 知识库层（.opencode/context/）——文件即数据库

| 领域目录 | 文档 | 说明 |
|----------|------|------|
| `devices/`（4 个） | 设备总览 / GPU-A100-01 / GPU-H100-01 / 自动驾驶测试车 | 设备资产域，DeviceAgent 专属检索范围 |
| `projects/`（3 个） | 项目总览 / LLM对齐优化项目 / 自动驾驶感知路测 | 项目域，ProjectAgent 专属检索范围 |
| `knowledge/论文笔记/` | DPO论文笔记.md | 知识域（附 README.md 辅助文件，rglob 扫描时同样入库） |
| `guides/`（2 个） | 新人入职指南 / 代码提交规范 | 流程规范域，KnowledgeAgent 全量检索范围 |
| `navigation.md` | 知识库总索引（60 行） | **被检索排除**：ingest 按文件名跳过（[ingest.py L39-40](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L39)），仅供人/Agent 导航 |

### 3.2.6 双执行面架构详解

```
                        ┌───────────────────────────────┐
                        │   .opencode/context/ 知识库     │  ← 唯一事实来源，文件即数据库
                        │   (Markdown + frontmatter)     │     devices/ projects/
                        │                                │     knowledge/ guides/
                        └───────────┬───────────────────┘     (navigation.md 被检索排除)
                                    │
              ┌─────────────────────┴──────────────────────┐
              │ 执行面 A：Python 运行时                    │ 执行面 B：OpenCode Agent 定义
              │ src/labqa/ + main.py                      │ .opencode/agent/ + commands/ + workflows/
              │                                           │
              │ main.py mode_ask/cli/feishu              │ lab-orchestrator.md
              │   └→ orchestrator.py 调度                 │   └→ subagents/{device,project,knowledge}-agent.md
              │        └→ router.py 意图识别（关键词打分）│        └→ 直接读 context/*.md
              │        └→ agents/* 按 category 检索       │ commands//查设备 /查项目 /查文档 /更新知识库
              │        └→ retriever.py 混合检索（RRF）    │ workflows/answer-query.md update-knowledge.md
              │        └→ generator.py 双轨 LLM 生成      │
              │                                           │
              │ 索引运行时生成（faiss_index/ + bm25_index.pkl）│
              │ python3 main.py ingest ←── 同一知识库 ────→ 文件变更即时生效（无需重建）
              └───────────────────────────────────────────┘
```

**两个执行面如何联动**（设计要点）：

1. **同一知识库，两种消费方式**：Python 侧通过 `ingest.py` 把 `.opencode/context/*.md` 分块后转成 FAISS/BM25 索引（[ingest.py L26-83](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L26)）；OpenCode 侧由 Agent 定义**直接引用文件**（`lab-orchestrator.md` 的 workflow 指示 subagent 读取 `context/` 对应目录）。改一个 Markdown，两个执行面同时受益。
2. **知识库即文件，无需数据库**：领域事实以 Markdown + frontmatter（`type/tags/updated`）存放，手写解析（[ingest.py L48-59](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L48)），无 MySQL/MongoDB 等外部依赖；飞书消息、对话历史也不落库（无多轮对话，见第七部分）。
3. **`navigation.md` 被检索排除**：它是知识库总索引（人/Agent 导航用），不是可检索内容，ingest 按文件名硬跳过（[ingest.py L39-40](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L39)）。
4. **知识更新链路统一**：`raw-docs/` 放入原始文档 → `bash scripts/convert.sh`（分类 + 补 frontmatter，[L227-295](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L227)）→ `python3 main.py ingest` 重建/增量加载索引 → 两个执行面均可查询。
5. **代码重复风险（工程债）**：两个执行面的意图识别与路由逻辑各自维护（Python `router.py` 关键词 vs Agent md 的 routing_rules），新增意图需双处修改（详见第七部分）。


## 3.3 关键设计决策表

| # | 设计决策 | 动机 / 解决的问题 | 实现方式 | 源码证据 |
|---|----------|-------------------|----------|----------|
| 1 | **Router = 轻量前端，Agent = 专业后端** | 意图识别只是"分流器"，不该为每条消息付 LLM 费用；专业检索+生成放后端 | Router 纯关键词打分（零成本），Agent 负责 RAG 检索 + LLM 生成 | [router.py L150-207](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150) + [orchestrator.py L71-130](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L71) |
| 2 | **Agent category 过滤保精准** | 设备问题只搜设备域，避免跨域噪音命中 | `DeviceAgent(category="devices")` / `ProjectAgent(category="projects")` / `KnowledgeAgent(category=None)` 全量 | [device_agent.py L10-15](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/device_agent.py#L10) / [project_agent.py L9-14](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/project_agent.py#L9) / [knowledge_agent.py L9-14](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/knowledge_agent.py#L9) |
| 3 | **FAISS 语义 + BM25 关键词互补 + RRF 融合** | 纯向量丢精确术语、纯关键词丢同义表达；用 RRF 无参融合两路排序 | dense_search + sparse_search → `reciprocal_rank_fusion`（`1/(60+rank)`） | [retriever.py L106-142](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106) |
| 4 | **BM25 无结果降级纯向量** | 稀疏检索对口语化/改写问题可能零命中，不能直接返回空 | sparse 命中数为 0 时跳过 RRF，走 dense 结果（L212-216） | [retriever.py L212-216](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L212) |
| 5 | **索引持久化，存在即加载** | 知识库未变时重建索引浪费 Embedding API 费用与时间 | `build_faiss_index`/`build_bm25_index` 检测文件存在则 load，`--rebuild` 才强制重建 | [ingest.py L125](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L125) / [L158](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L158) |
| 6 | **飞书 WebSocket 长连接，免公网 IP** | 实验室无公网 IP/反向代理；HTTP webhook 需要公网可达 | lark-oapi WS 长连接（`main.py feishu`），`webhook` 仅作兼容别名 | [feishu_ws.py L93-180](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93) |
| 7 | **飞书两层消息去重** | 飞书 SDK 重试会重复投递，产生重复回答 | message_id 集合 + 30 秒内容哈希窗口（chat 维度） | [feishu_ws.py L28-56](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28) |
| 8 | **双轨 LLM：Ollama 本地 / 云端 DeepSeek-OpenAI** | 本地免费可离线、云端质量高；用 `LABQA_LLM_MODE` 一键切换，Key 三级回退 | `create_llm` 按模式分支（[generator.py L99-118](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99)）；配置见 [.env.example L6-16](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L6) | [config.py L25-35](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25) |
| 9 | **混合意图自动发现（MIXED）** | 一句"看下 GPU 项目和 A100 的进度"同时涉设备+项目，不能二选一 | 次要意图分数 ≥ 主意图 50% → MIXED，两路 Agent 并行回答 | [router.py L178-181](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L178) |
| 10 | **中文优化分块（500/50）** | 中文按字符计长，需中文分隔符保证语义连贯 | RecursiveCharacterTextSplitter，separators 含 `。！？；`（[ingest.py L95-101](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L95)）；BM25 优先 jieba 分词（[L169-180](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L169)） | [config.py L42-46](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L42) |
| 11 | **CLI 离线模式兜底** | 无 API Key 时系统仍可用（纯关键词检索），用 `LABQA_LLM_API_KEY="offline-mode"` 哨兵 | cli.py 检测哨兵跳过 LLM 调用 | [cli.py L62-149](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62) |

### v1 → v2 重构演进（commit `757df23`）

> `feat: LabQA-RAG v2.0 全面重构为混合检索 RAG 架构`（2026-06-05，[查看 commit](https://github.com/BLYHFL/labqa-rag/commit/757df23)）

| 维度 | v1（commit 07bb9b1，2026-05-21） | v2（commit 757df23，2026-06-05） |
|------|----------------------------------|----------------------------------|
| 检索方式 | 关键词匹配（context_store.py 扫描 + 打分） | **FAISS 稠密 + BM25 稀疏 + RRF 融合** |
| 包结构 | `labqa/`（context_store.py / llm.py / webhook.py） | `src/labqa/`（ingest / retriever / generator / prompts / agents 分层） |
| Agent 体系 | 单 Agent 直接检索 | **Multi-Agent 路由**：Device / Project / Knowledge 三专用 Agent |
| 意图识别 | 简易关键词匹配 | `router.py` 打分路由 + MIXED 混合意图 |
| 测试 | 无 | tests/test_main.py（pytest，7 个 Test 类约 18 用例） |
| 文档 | — | docs/WORKFLOW.md（506 行流程详解）、docs/ARCHITECTURE.md（含 ADR） |
| 重构动机 | 关键词匹配**召回差**：同义表达、语义查询（"哪台机器空闲"）命中不了 | 混合检索覆盖语义 + 精确术语两类查询 |

**演进主线**：单文件直搜（v1）→ 索引化混合检索 + 多 Agent 专业化（v2）；知识库始终是 `.opencode/context/` 文件目录，这是两次架构共用、未变的部分。


## 3.4 可运行验证：打印目录结构

> 需在**项目根目录**运行（`/Users/wuhang/Desktop/简历项目经历复盘总结/Lapmind`）。仅用标准库 os.walk，跳过 `.git` / `.venv` / `node_modules` / `__pycache__` 等运行时目录。


In [ ]:
import os

SKIP = {".git", ".venv", "node_modules", "__pycache__", ".DS_Store", ".idea", ".vscode"}
INDEX_ARTIFACTS = {"faiss_index": "🔍 索引（运行时生成，gitignore）", "bm25_index.pkl": "🔍 索引（运行时生成，gitignore）"}

def tree(root: str, prefix: str = "") -> int:
    """递归打印目录树，返回文件计数"""
    entries = sorted(
        e for e in os.listdir(root)
        if e not in SKIP and not e.startswith(".")
    )
    files = 0
    for i, name in enumerate(entries):
        path = os.path.join(root, name)
        is_last = i == len(entries) - 1
        branch = "└── " if is_last else "├── "
        if os.path.isdir(path):
            note = INDEX_ARTIFACTS.get(name, "")
            print(f"{prefix}{branch}{name}/ {note}")
            files += tree(path, prefix + ("    " if is_last else "│   "))
        else:
            print(f"{prefix}{branch}{name}")
            files += 1
    return files

root = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."
print(f"📁 项目目录结构（{root}，跳过运行时目录）：\n")
total = tree(root)
print(f"\n共 {total} 个文件（不含 .git/.venv/node_modules 等）")


# 第四部分：源码导航索引

> 本部分为 LabQA-RAG v2.0（Lapmind）全项目源码导航索引，按七个模块分小节：**4.1 主入口与调度 → 4.2 意图识别与路由 → 4.3 混合检索 → 4.4 数据入库与生成 → 4.5 Agent 层 → 4.6 交互层与集成 → 4.7 命令与工作流**。每节以「内容 | 位置」超链接表列出全部关键符号。
>
> 链接格式：`https://github.com/BLYHFL/labqa-rag/blob/main/{path}#L{行号}`；所有行号均已在仓库中核验，可直接跳转。

## 4.1 主入口与调度

### 4.1.1 主入口 `main.py`（项目根目录）

**定位**：命令行入口，五种 mode（cli/ingest/ask/feishu/stats）的调度中枢；自带手写 .env 加载器，未安装 python-dotenv 也能零依赖启动。

| 内容 | 位置 |
|------|------|
| `main`：argparse 分发五种子命令（L114-L149） | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L114-L149) |
| `mode_cli`：交互式 CLI 问答模式 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L46) |
| `mode_ingest`：构建 FAISS + BM25 索引模式 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L52) |
| `mode_ask`：单次问答模式（`--question` 参数） | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L58) |
| `mode_feishu`：飞书 WebSocket 长连接模式 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L91) |
| `mode_stats`：知识库统计模式 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L97) |
| 手写 .env 加载器（兼容 dotenv 之外的零依赖启动） | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L26-L41) |
| `sys.path` 注入（保证根目录直接运行时可 import labqa） | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L24) |

**设计要点**

- **双轨配置来源**：main.py 手写 .env 加载器与 config.py 的 python-dotenv 为双实现（4.6 详述），保证入口在任何环境下都能读到环境变量。
- **五种模式互不耦合**：每个 `mode_*` 函数独立构建所需组件；`feishu` 模式通过 `get_orchestrator` 复用全局单例。
- **兼容别名**：`main.py webhook` 实际启动的是 WebSocket 长连接（历史别名保留，见工程债部分）。

### 4.1.2 调度中枢 `src/labqa/orchestrator.py`

**定位**：RAG 问答编排的核心，持有 retriever + generator + 三个 Agent，完成「意图识别 → 上下文检索 → Agent 生成」全链路。

| 内容 | 位置 |
|------|------|
| `Orchestrator` 类定义（编排器主体） | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L15) |
| `initialize`：初始化检索器/生成器/Agent 依赖 | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L25-L53) |
| `_init_agents`：注册 Device/Project/Knowledge 三个专业 Agent | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L55-L68) |
| `process`：核心问答编排（路由 → 检索 → 生成 → 兜底） | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L71-L130) |
| `_unknown_intent_response`：未知意图兜底回复 | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L132-L165) |
| `stats`：知识库统计信息 | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L167-L176) |
| `reload`：热重载检索索引（CLI `/reload` 触发） | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L178-L184) |
| `get_orchestrator`：全局单例（避免重复加载索引） | [orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L191-L196) |

**设计要点**

- **Router 是「轻量级前端」，Agent 是「专业后端」**：orchestrator 先用 router 做零成本关键词打分定意图，再交给对应 Agent 做 RAG 检索 + LLM 生成。
- **单例模式**：`get_orchestrator` 保证全进程只有一个 Orchestrator，飞书长连接与 CLI 复用同一份索引内存，降低内存占用。
- **未知意图有独立兜底路径**：`_unknown_intent_response` 不进入 RAG 流程，直接返回引导文案（prompts.py 的 UNKNOWN_INTENT_PROMPT，见 4.6）。


## 4.2 意图识别与路由：`src/labqa/router.py`

**定位**：意图识别路由层，基于关键词打分的「轻量级前端」。把用户问题分类为 DEVICE / PROJECT / KNOWLEDGE / UNKNOWN / MIXED 五种意图，并抽取关键词供检索层使用；零 LLM 成本、毫秒级响应。

| 内容 | 位置 |
|------|------|
| `Intent` 枚举（DEVICE/PROJECT/KNOWLEDGE/UNKNOWN/MIXED） | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L16-L21) |
| `RoutingDecision` 数据结构（意图 + 关键词 + 分类） | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L24-L31) |
| `DEVICE_KEYWORDS` 设备意图关键词表（GPU/显卡/服务器等） | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L35-L44) |
| `PROJECT_KEYWORDS` 项目意图关键词表（对齐/路测/优化等） | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L46-L56) |
| `KNOWLEDGE_KEYWORDS` 知识意图关键词表（论文/方法/笔记等） | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L58-L69) |
| `AGENT_MAP`：意图 → Agent 名映射 | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L72-L76) |
| `INTENT_CATEGORY_MAP`：意图 → 检索分类映射 | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L79-L83) |
| `extract_keywords`：jieba 分词抽取查询关键词 | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L86-L129) |
| `_score_intent`：三类意图关键词打分 | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L132-L147) |
| `recognize_intent`：意图识别主函数（含混合意图判定） | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150-L207) |
| 混合意图判定：次要意图 ≥ 主意图 50% → MIXED | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L178-L181) |
| `get_intent_category`：意图 → 检索分类快速查询 | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L210-L212) |

**设计要点**

- **混合意图自动发现**：当次要意图得分 ≥ 主意图得分的 50% 时判定为 MIXED（L178-L181），避免"GPU 项目进度"这类跨域问题被单一路由截断。
- **零成本路由**：纯关键词打分 + jieba 分词，不调用 LLM，保证飞书高频消息下入口无延迟。
- **硬编码代价**：关键词表是手写的，新增意图需同时改 router.py 关键词与 Agent 的关键词提取方法（工程债，见后续部分）。
- **工作流文档对应**：打分机制详解见 docs/WORKFLOW.md L147-L182（4.6 索引）。


## 4.3 混合检索：`src/labqa/retriever.py`

**定位**：检索层核心。FAISS 稠密向量检索（语义）+ BM25 稀疏关键词检索（词面）+ RRF 倒数排名融合，输出去重后的 Top-K 上下文；支持按分类过滤检索。

| 内容 | 位置 |
|------|------|
| `dense_search`：FAISS 稠密向量检索 | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L19-L52) |
| `sparse_search`：BM25 稀疏检索（jieba 分词 + 关键词评分） | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L57-L101) |
| `reciprocal_rank_fusion`：RRF 融合（1/(60+rank)） | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106-L142) |
| `load_indexes`：加载 FAISS/BM25 索引（缺失时重建） | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L147-L172) |
| `hybrid_search`：混合检索主流程 | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175-L229) |
| BM25 无结果时降级为纯向量检索 | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L212-L216) |
| `search_by_category`：按分类（devices/projects/knowledge）过滤检索 | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L232-L245) |

**设计要点**

- **两路互补**：FAISS 捕获语义相似（同义改写、口语化），BM25 保证术语精确命中（型号、编号如 GPU-A100-01）。
- **RRF 融合去重**：`1/(60+rank)` 对两路排名加权，天然去重并稳定排序，无需调权重。
- **降级设计**：BM25 无命中时自动退化为纯向量检索（L212-L216），保证关键词不匹配的场景仍有结果。
- **分类过滤**：`search_by_category` 配合 Agent 的 category（4.5），实现"只搜设备目录"的精准检索。


## 4.4 数据入库与生成

### 4.4.1 索引构建 `src/labqa/ingest.py`

**定位**：知识库入库管道。递归加载 `.opencode/context/` 下 Markdown → frontmatter 解析 → 中文优化分块 → 分别构建 FAISS（稠密）与 BM25（稀疏）索引 → 持久化到磁盘。

| 内容 | 位置 |
|------|------|
| `load_markdown_files`：递归加载 .md 文件并解析 frontmatter | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L26-L83) |
| frontmatter 手写解析（非完整 YAML，简单键值对） | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L51-L59) |
| `split_documents`：RecursiveCharacterTextSplitter 分块（500/50 中文优化） | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L86-L110) |
| `build_faiss_index`：构建 FAISS 向量索引 | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L113-L144) |
| FAISS 索引持久化：已存在则加载不重建 | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L125) |
| `build_bm25_index`：jieba 分词构建 BM25 稀疏索引 | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L147-L197) |
| BM25 索引持久化：已存在则加载不重建 | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L158) |
| `build_indexes`：全量构建入口（faiss_index/ + bm25_index.pkl） | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L200-L219) |
| `check_indexes`：校验索引是否可用 | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L222-L235) |
| `main`：CLI 入口（`python3 main.py ingest` 触发） | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L239-L273) |

### 4.4.2 生成层 `src/labqa/generator.py`

**定位**：双轨 LLM 抽象 + 检索增强生成。支持 Ollama 本地免费 / 云端 DeepSeek/OpenAI 两种模式；提供 Agent 专属提示词模板。

| 内容 | 位置 |
|------|------|
| `OllamaEmbeddings`：本地 Ollama embedding 封装（LangChain 兼容） | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L31-L43) |
| `OllamaChat`：本地 Ollama chat 封装 | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L48-L74) |
| `create_embeddings`：按 LLM_MODE 创建 embedding 模型 | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L79-L96) |
| `create_llm`：双轨 LLM 工厂（ollama / cloud） | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99-L118) |
| `generate_answer`：检索增强生成主流程（上下文 + 提示词 → 回答） | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L123-L199) |
| `PROMPT_TEMPLATE`：通用 RAG 提示词模板 | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L204-L218) |
| `AGENT_PROMPTS`：三个 Agent 的专属提示词集 | [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L222-L274) |

**设计要点**

- **索引运行时生成**：`faiss_index/index.faiss` + `bm25_index.pkl` 由 ingest 命令构建，运行时 `load_indexes` 直接加载。
- **幂等构建**：索引已存在则跳过重建（L125、L158），支持增量场景下安全重跑。
- **双轨切换**：`LABQA_LLM_MODE=ollama` 用本地免费模型，`cloud` 走 OpenAI 兼容接口；离线哨兵 `LABQA_LLM_API_KEY="offline-mode"` 触发 CLI 离线兜底（见工程债部分）。


## 4.5 Agent 层：`src/labqa/agents/`

**定位**：专业后端 Agent 层。`BaseAgent` 提供「检索上下文 → 生成回答」模板方法；三个子类通过 category 限定检索范围，实现多 Agent 分工。

### 4.5.1 基类 `agents/base.py`

| 内容 | 位置 |
|------|------|
| `BaseAgent` 类定义（模板方法：search_context → answer） | [base.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L15-L121) |
| `set_dependencies`：注入 retriever/generator 依赖 | [base.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L29-L33) |
| `search_context`：RAG 上下文检索（hybrid_search） | [base.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L35-L69) |
| `answer`：生成回答（Agent 专属提示词 + 上下文） | [base.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L71-L107) |
| `_no_result_response`：检索无结果时的兜底文案 | [base.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L109-L121) |

### 4.5.2 三个专业 Agent 与注册表

| 内容 | 位置 |
|------|------|
| `DeviceAgent`（category="devices"，只检索设备目录） | [device_agent.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/device_agent.py#L10-L15) |
| `ProjectAgent`（category="projects"，只检索项目目录） | [project_agent.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/project_agent.py#L9-L14) |
| `KnowledgeAgent`（category=None，全量检索 knowledge+guides） | [knowledge_agent.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/knowledge_agent.py#L9-L14) |
| `AGENT_REGISTRY`：Agent 注册表（name → class） | [__init__.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/__init__.py#L14-L18) |
| `get_agent`：Agent 工厂函数 | [__init__.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/__init__.py#L21-L25) |

**设计要点**

- **category 过滤保证精准度**：DeviceAgent 只搜 devices/，ProjectAgent 只搜 projects/，KnowledgeAgent 全量搜 knowledge + guides——路由意图与检索范围严格对应。
- **模板方法模式**：子类只需声明 category，继承基类的检索与生成逻辑，新增意图 Agent 成本极低。
- **注册表解耦**：orchestrator 通过 `AGENT_REGISTRY` + `get_agent` 按 router 的 `AGENT_MAP` 动态取 Agent，新增 Agent 无需改编排代码。


## 4.6 交互层与集成

### 4.6.1 CLI 交互 `src/labqa/cli.py`

**定位**：命令行交互界面（BANNER + 系统命令），`/help /stats /reload /ingest /debug` 五个内置命令。

| 内容 | 位置 |
|------|------|
| `BANNER`：启动横幅 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L18-L27) |
| `HELP_TEXT`：帮助文案 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L29-L59) |
| `main`：CLI 主循环（读输入 → 路由 → 输出） | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62-L149) |
| `/help`：帮助命令 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L108) |
| `/stats`：知识库统计命令 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L110) |
| `/reload`：热重载索引命令 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L120) |
| `/ingest`：重新构建索引命令 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L123) |
| `/debug`：调试信息命令 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L131) |

### 4.6.2 飞书 WebSocket 集成 `src/labqa/feishu_ws.py`

**定位**：飞书长连接运行时（无需公网 IP/回调地址），含消息去重、分发、发送全链路。

| 内容 | 位置 |
|------|------|
| `_is_duplicate`：两层去重（message_id 集合 + 30 秒内容哈希窗口） | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28-L56) |
| `_handle_message`：消息处理分发（路由到 orchestrator） | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L59-L75) |
| `_send_message`：回复消息发送 | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L78-L90) |
| 超长回复截断（19900 字符，飞书限制） | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L83) |
| `start`：WS 长连接主循环 | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93-L180) |
| `on_message`：WS 事件回调 | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L124-L160) |
| WS 客户端构建（lark-oapi） | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L162-L169) |

### 4.6.3 配置层 `src/labqa/config.py`

| 内容 | 位置 |
|------|------|
| `PROJECT_ROOT`：项目根目录 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L14) |
| `CONTEXT_DIR`：知识库目录（.opencode/context/） | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L17) |
| `FAISS_INDEX_DIR`：FAISS 索引目录 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L21) |
| `BM25_INDEX_PATH`：BM25 索引路径 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L22) |
| `LLM_MODE`：ollama / cloud 双轨切换 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25) |
| `OLLAMA_*`：本地 Ollama 配置（地址/模型） | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L28-L29) |
| `LLM_API_KEY`：三级回退（LABQA_LLM_API_KEY → DEEPSEEK_API_KEY → OPENAI_API_KEY） | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32-L35) |
| `LLM_API_BASE`：云端 API 地址 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L36) |
| 检索参数（top_k / 阈值等） | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L42-L46) |
| `FEISHU_*`：飞书配置（App ID/Secret 等） | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L49-L51) |
| `DEBUG`：调试开关 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L54) |
| `validate`：配置校验 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L57-L70) |
| `print_config`：打印当前配置 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L73-L89) |

### 4.6.4 提示词管理 `src/labqa/prompts.py`

| 内容 | 位置 |
|------|------|
| `DEFAULT_RAG_PROMPT`：默认 RAG 提示词 | [prompts.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L6) |
| `DEVICE_PROMPT`：设备 Agent 提示词 | [prompts.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L23) |
| `PROJECT_PROMPT`：项目 Agent 提示词 | [prompts.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L42) |
| `KNOWLEDGE_PROMPT`：知识 Agent 提示词 | [prompts.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L62) |
| `UNKNOWN_INTENT_PROMPT`：未知意图兜底提示词 | [prompts.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L81) |
| `get_agent_prompt`：按 Agent 名取提示词 | [prompts.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L93-L100) |

### 4.6.5 文档转换脚本 `scripts/convert.sh`

**定位**：把 docx/pptx/xlsx/pdf 转换为带 frontmatter 的 Markdown 后归入知识库（主循环 + 分类 + 幂等）。

| 内容 | 位置 |
|------|------|
| `convert_docx`：Word 转 Markdown | [convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L41) |
| `convert_pptx`：PPT 转 Markdown | [convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L73) |
| `convert_xlsx`：Excel 转 Markdown | [convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L109) |
| `convert_pdf`：PDF 转 Markdown（仅占位实现） | [convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L141) |
| `copy_markdown`：纯 Markdown 直接复制 | [convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L152) |
| `classify_target`：按路径分类目标（devices/projects/knowledge/guides） | [convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L163-L187) |
| `add_frontmatter`：补写 frontmatter 元数据 | [convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L191-L216) |
| 主循环（遍历输入 → 转换 → 归档） | [convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L227-L295) |
| `feishu-webhook.py`：legacy 独立脚本（自带关键词匹配，绕过 labqa/，不用于新开发） | [feishu-webhook.py](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/feishu-webhook.py) |

### 4.6.6 OpenCode Agent 体系 `.opencode/agent/`

**定位**：双执行面架构的第二面——OpenCode Agent 定义，与 Python 运行时共享同一知识库（.opencode/context/）。

| 内容 | 位置 |
|------|------|
| `lab-orchestrator.md`：主 Agent 定义（128 行） | [lab-orchestrator.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/lab-orchestrator.md) |
| intent_recognition：意图识别章节 | [lab-orchestrator.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/lab-orchestrator.md#L31-L51) |
| routing_rules：路由规则章节 | [lab-orchestrator.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/lab-orchestrator.md#L53-L61) |
| workflow：工作流章节 | [lab-orchestrator.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/lab-orchestrator.md#L96-L123) |
| 设备子 Agent `device-agent.md` | [device-agent.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/subagents/device-agent.md) |
| 项目子 Agent `project-agent.md` | [project-agent.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/subagents/project-agent.md) |
| 知识子 Agent `knowledge-agent.md` | [knowledge-agent.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/subagents/knowledge-agent.md) |
| 知识库总索引 `context/navigation.md`（60 行，被检索排除） | [navigation.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/context/navigation.md) |

### 4.6.7 文档与测试 `docs/` + `tests/`

**定位**：项目文档（架构/测试/工作流）与 pytest 测试套件（320 行，7 个 Test 类，约 18 用例）。

| 内容 | 位置 |
|------|------|
| 架构设计文档 ARCHITECTURE.md（270 行） | [ARCHITECTURE.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/ARCHITECTURE.md) |
| ADR-001：知识库统一 Markdown 格式 | [ARCHITECTURE.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/ARCHITECTURE.md#L221) |
| ADR-002：.opencode Agent 体系 | [ARCHITECTURE.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/ARCHITECTURE.md#L235) |
| 测试手册 TESTING.md（152 行，手动测试清单 8 大节） | [TESTING.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/TESTING.md) |
| 问答流程详解 WORKFLOW.md（506 行） | [WORKFLOW.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md) |
| Router 打分机制详解 | [WORKFLOW.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L147-L182) |
| 混合意图判定详解 | [WORKFLOW.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L176-L182) |
| 完整示例走读（端到端问答） | [WORKFLOW.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L186-L416) |
| `TestConfig`：配置加载测试类 | [test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L17) |
| `TestIngest`：入库管道测试类 | [test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L39) |
| `TestRouter`：意图识别测试类 | [test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L89) |
| `TestRRF`：RRF 融合算法测试类 | [test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L157) |
| `TestPrompts`：提示词测试类 | [test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L203) |
| `TestGenerator`：生成层测试类 | [test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L247) |
| `TestAgents`：Agent 层测试类 | [test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L284) |

### 4.6.8 环境变量样例 `.env.example`

| 内容 | 位置 |
|------|------|
| `.env.example`（34 行配置样例） | [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example) |
| `LABQA_LLM_MODE=ollama` 默认模式 | [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L6-L8) |
| cloud 模式配置 | [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L11-L16) |
| 检索参数配置 | [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L19-L23) |
| 飞书配置 | [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L26-L31) |


## 4.7 命令与工作流速查

### 4.7.1 main.py 五种命令（入口分发）

| 命令 | 功能 | 位置 |
|------|------|------|
| `python3 main.py cli` | 交互式 CLI 问答 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L46) |
| `python3 main.py ingest` | 构建 FAISS + BM25 索引 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L52) |
| `python3 main.py ask --question "..."` | 单次问答（适合脚本调用） | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L58) |
| `python3 main.py feishu` | 启动飞书 WebSocket 长连接 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L91) |
| `python3 main.py stats` | 知识库统计 | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L97) |

### 4.7.2 CLI 内置系统命令（交互模式内）

| 命令 | 功能 | 位置 |
|------|------|------|
| `/help` | 显示帮助 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L108) |
| `/stats` | 查看知识库统计 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L110) |
| `/reload` | 热重载检索索引 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L120) |
| `/ingest` | 重新构建索引 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L123) |
| `/debug` | 调试信息 | [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L131) |

### 4.7.3 OpenCode 斜杠命令（.opencode/commands/）

| 命令 | 功能 | 位置 |
|------|------|------|
| `/查设备` | 查询设备信息（OpenCode Agent 执行面） | [查设备.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/commands/%E6%9F%A5%E8%AE%BE%E5%A4%87.md) |
| `/查项目` | 查询项目信息 | [查项目.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/commands/%E6%9F%A5%E9%A1%B9%E7%9B%AE.md) |
| `/查文档` | 查询知识库文档 | [查文档.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/commands/%E6%9F%A5%E6%96%87%E6%A1%A3.md) |
| `/更新知识库` | 更新知识库索引 | [更新知识库.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/commands/%E6%9B%B4%E6%96%B0%E7%9F%A5%E8%AF%86%E5%BA%93.md) |

### 4.7.4 OpenCode 工作流（.opencode/workflows/）

| 工作流 | 功能 | 位置 |
|--------|------|------|
| `answer-query.md` | 问答工作流（检索 → 回答） | [answer-query.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/workflows/answer-query.md) |
| `update-knowledge.md` | 知识库更新工作流 | [update-knowledge.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/workflows/update-knowledge.md) |

### 4.7.5 全链路调用关系（速览）

```
用户提问（飞书 WS / CLI / OpenCode 斜杠命令）
        │
        ▼
router.recognize_intent ── 关键词打分 + jieba 分词 ──► Intent + keywords
        │
        ▼
orchestrator.process ──► AGENT_MAP 选 Agent（Device/Project/Knowledge）
        │
        ▼
BaseAgent.search_context ──► retriever.hybrid_search
                                 ├── dense_search（FAISS 语义）
                                 ├── sparse_search（BM25 词面）
                                 └── reciprocal_rank_fusion（1/(60+rank)）
        │
        ▼
BaseAgent.answer ──► generator.generate_answer（PROMPT_TEMPLATE / AGENT_PROMPTS）
        │
        ▼
回复（CLI 打印 / 飞书 _send_message）
```

**设计要点**

- **双执行面入口等价**：Python 运行时（main.py + cli.py + feishu_ws.py）与 OpenCode Agent 体系（lab-orchestrator.md + commands/ + workflows/）共享同一知识库，用户可任选入口。
- **索引生命周期**：`ingest` 构建 → `load_indexes` 加载 → `/reload` 热重载 → `/ingest` 重建，闭环完整。
- **完整示例走读**：端到端问答的逐步走读见 [WORKFLOW.md L186-L416](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L186-L416)。


# 第五部分：本地运行与部署指南

> **启动顺序总览**：安装依赖 → 配置 .env → ingest 建索引 → cli / ask / feishu
>
> 本项目的"部署"极轻：无 Docker、无数据库、无公网服务器——一切都在本地 Python 运行时内完成（FAISS 索引落盘 + 飞书 WebSocket 长连接）。整个启动链路只需 4 步：

```
┌─────────────┐   ┌─────────────┐   ┌─────────────┐   ┌──────────────────────┐
│  ① 环境准备  │ → │ ② 配置 .env │ → │ ③ ingest    │ → │ ④ cli / ask / feishu │
│  uv venv    │   │  双轨 LLM   │   │  构建索引    │   │  三种入口 + 运维命令  │
└─────────────┘   └─────────────┘   └─────────────┘   └──────────────────────┘
```

| 步骤 | 命令 | 产物 | 位置 |
|------|------|------|------|
| ① 环境准备 | `uv venv --python 3.9 && source .venv/bin/activate` + `uv pip install -e ".[dev]"` | 虚拟环境 + 依赖 | `.venv/` |
| ② 配置 .env | `cp .env.example .env` | 双轨 LLM 配置 | 项目根目录 `.env` |
| ③ 构建索引 | `python3 main.py ingest` | FAISS 稠密索引 + BM25 稀疏索引 | `faiss_index/index.faiss` + `bm25_index.pkl` |
| ④ 启动入口 | `python3 main.py cli` / `ask` / `feishu` | 交互式问答服务 | — |

> 入口分发逻辑见 [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L114-L149)（`main` L114-L149），各模式分支：`mode_cli` L46 / `mode_ingest` L52 / `mode_ask` L58 / `mode_feishu` L91 / `mode_stats` L97。

---

## 5.1 环境准备

### ① 创建虚拟环境（推荐 uv，Python 3.9）

```bash
# 方式 A：uv（推荐，速度快）
uv venv --python 3.9
source .venv/bin/activate

# 方式 B：python3 自带 venv
python3 -m venv .venv
source .venv/bin/activate
```

> 项目 pyproject 声明 `name = "labqa-rag"`（v2.0），依赖 LangChain 生态（FAISS 向量库、jieba 分词、lark-oapi SDK 等），Python 3.9 为最低兼容版本。

### ② 安装依赖（二选一）

```bash
# 方式 A：从 pyproject 安装（含 dev extras，推荐开发）
uv pip install -e ".[dev]"

# 方式 B：从 requirements.txt 安装（仅运行时核心依赖）
pip install -r requirements.txt
```

**依赖清单对照**（注意两处不一致，以 pyproject 为准）：

| 依赖 | pyproject（editable + dev） | requirements.txt | 用途 |
|------|------|------|------|
| langchain / langchain-community | ✅ | ✅ | RAG 框架（分块、FAISS、向量化接口） |
| faiss-cpu | ✅ | ✅ | 稠密向量索引 |
| jieba | ✅ | ✅ | BM25 中文分词 |
| lark-oapi | ✅ | ✅（未注释） | 飞书 SDK（长连接/发消息） |
| httpx | ✅ | ✅（未注释） | HTTP 客户端（云端 LLM API） |
| python-dotenv | ✅ | ✅ | .env 加载（config.py 使用） |
| pytest | ✅（dev extras） | ❌ 未列出 | 测试框架 |
| 文档转换依赖（python-docx / python-pptx / openpyxl / pypdf） | ✅（dev extras） | ⚠️ **被注释** | `scripts/convert.sh` 转换管道 |

> ⚠️ **已知工程债**：`requirements.txt` 仅保留 httpx / lark-oapi，文档转换依赖被注释；pyproject 的 dev extras 与之不一致。**以 `uv pip install -e ".[dev]"` 为准**可同时覆盖测试与转换管道。若仅需跑问答，`requirements.txt` 已够用。

### ③ 可选：本地 Ollama 模型（ollama 模式才需要）

```bash
# 拉取 LLM 与 Embedding 模型（ollama 免费本地模式）
ollama pull deepseek-r1:1.5b     # 轻量推理模型（生成回答）
ollama pull nomic-embed-text     # 768 维 Embedding 模型（向量化）

# 验证
ollama list
```

> 双轨 LLM 架构详见 5.2：不装 Ollama 也可以直接用 cloud 模式（DeepSeek/OpenAI 云端 API），Embedding 由 [generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L31-L43) 的 `OllamaEmbeddings`（L31-L43）/ [OllamaChat](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L48-L74)（L48-L74）封装。

---

## 5.2 配置 .env（双轨 LLM 模式）

```bash
cp .env.example .env
# 然后编辑 .env：选择模式 + 填必填项
```

**核心开关 `LABQA_LLM_MODE`**（[config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25) L25，默认值见 [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L6-L8) L6-L8）：

| 模式 | 配置值 | LLM | Embedding | 成本 | 适用场景 |
|------|--------|-----|-----------|------|----------|
| **ollama**（默认） | `LABQA_LLM_MODE=ollama` | 本地 `deepseek-r1:1.5b` | 本地 `nomic-embed-text` | 免费 | 内网/无外网、隐私敏感 |
| **cloud** | `LABQA_LLM_MODE=cloud` | DeepSeek / OpenAI 云端 API | 同上 Ollama 本地（或云端 API） | 按 token 计费 | 追求回答质量、有 API Key |

**必填项清单**（标注 ★ 为必须）：

| 变量 | 模式 | 必填 | 说明 |
|------|------|------|------|
| `LABQA_LLM_MODE` | 双轨 | ★ | `ollama`（默认）或 `cloud`，见 [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L6-L16) L6-L16 |
| `LABQA_OLLAMA_BASE_URL` | ollama | — | Ollama 服务地址，默认 `http://localhost:11434` |
| `LABQA_OLLAMA_CHAT_MODEL` | ollama | — | 聊天模型名，默认 `deepseek-r1:1.5b` |
| `LABQA_OLLAMA_EMBED_MODEL` | ollama | — | Embedding 模型名，默认 `nomic-embed-text` |
| `LABQA_LLM_API_KEY` | cloud | ★ | 云端 API Key（**三级回退**，见下） |
| `LABQA_LLM_API_BASE` | cloud | — | API 地址，默认 DeepSeek/OpenAI 官方端点 |
| `LABQA_LLM_MODEL` | cloud | — | 云端模型名 |
| 检索参数（`LABQA_TOP_K` / `LABQA_FAISS_WEIGHT` / `LABQA_BM25_WEIGHT` 等） | 双轨 | — | 混合检索调参，见 [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L19-L23) L19-L23 |
| `LABQA_FEISHU_APP_ID` / `LABQA_FEISHU_APP_SECRET` | feishu | ★（仅飞书模式） | 飞书应用凭证，见 [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L26-L31) L26-L31 |

**API Key 三级回退**（[config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32-L35) L32-L35）——按序取第一个非空值：

```
LABQA_LLM_API_KEY → DEEPSEEK_API_KEY → OPENAI_API_KEY
```

> 即：即使只配置了旧的 `DEEPSEEK_API_KEY` 也能直接跑 cloud 模式，无需迁移。配置加载为"手写 .env 加载器 + python-dotenv 双实现"（[main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L26-L41) L26-L41），启动时 `sys.path` 注入项目根（[main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L24) L24）。

**验证配置**：`python3 main.py stats` 会调用 [config.print_config](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L73-L89)（L73-L89）打印当前生效配置（含校验结果，[validate](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L57-L70) L57-L70）。

---

## 5.3 知识库准备（.opencode/context/ 目录体系）

### 现状：8 个文档 + 1 个总索引

知识库位于 `.opencode/context/`，Markdown + frontmatter 格式（主控核验统计：**8 个可检索文档**；`navigation.md` 为知识库总索引、**被检索排除**）：

```
.opencode/context/
├── navigation.md                          # 知识库总索引（60 行，检索时排除）
├── devices/                               # 设备域（DeviceAgent 检索范围）
│   ├── 设备总览.md
│   ├── GPU-A100-01.md
│   ├── GPU-H100-01.md
│   └── 自动驾驶测试车.md
├── projects/                              # 项目域（ProjectAgent 检索范围）
│   ├── 项目总览.md
│   ├── LLM对齐优化项目.md
│   └── 自动驾驶感知路测.md
├── knowledge/
│   └── 论文笔记/DPO论文笔记.md             # 知识域（KnowledgeAgent 全量检索）
└── guides/                                # 指南域（KnowledgeAgent 全量检索）
    ├── 新人入职指南.md
    └── 代码提交规范.md
```

| 目录 | 文档数 | 检索范围（Agent category 过滤） | 说明 |
|------|--------|------|------|
| `devices/` | 4 | DeviceAgent（category=`devices`） | 设备台账类 |
| `projects/` | 3 | ProjectAgent（category=`projects`） | 项目进展类 |
| `knowledge/` | 1 | KnowledgeAgent（category=`None` 全量） | 论文笔记等 |
| `guides/` | 2 | KnowledgeAgent（category=`None` 全量） | 流程/规范类 |
| `navigation.md` | 1 | ❌ 排除 | 手工维护的总索引，仅作目录导航 |

> Agent 的 category 过滤定义见 [agents/__init__.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/__init__.py#L14-L18)（`AGENT_REGISTRY` L14-L18），每个 Agent 的检索入口在 [agents/base.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L35-L69)（`search_context` L35-L69）。

### 新增文档：raw-docs/ → convert.sh 转换管道

若需要从 Word/PPT/Excel/PDF 导入文档，走 `scripts/convert.sh` 一键转换（依赖 5.1 中 dev extras 的转换库）：

```bash
# 把原始文档放入 raw-docs/ 后执行
bash scripts/convert.sh
```

转换管道内部步骤（[convert.sh](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh)）：

| 阶段 | 函数 | 行号 | 说明 |
|------|------|------|------|
| Word 转换 | `convert_docx` | [L41](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L41) | .docx → .md |
| PPT 转换 | `convert_pptx` | [L73](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L73) | .pptx → .md |
| Excel 转换 | `convert_xlsx` | [L109](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L109) | .xlsx → .md |
| PDF 转换 | `convert_pdf` | [L141](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L141) | ⚠️ 目前为占位实现（已知工程债） |
| Markdown 直通 | `copy_markdown` | [L152](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L152) | .md 直接复制 |
| 目标分类 | `classify_target` | [L163-L187](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L163-L187) | 按路径规则归入 devices/projects/... |
| 补 frontmatter | `add_frontmatter` | [L191-L216](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L191-L216) | 自动生成 title/date 等元信息 |
| 主循环 | — | [L227-L295](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L227-L295) | 遍历 raw-docs/ 全量处理 |

### navigation.md：手工索引

`.opencode/context/navigation.md`（60 行）是知识库总索引，需**手工维护**目录与链接（新文档入库后手动追加条目）。ingest 阶段会将其排除出检索（不参与建索引），仅作为人工导航入口。

> 文档解析由 [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L26-L83) 的 `load_markdown_files`（L26-L83）完成，frontmatter 为手写解析（非完整 YAML，嵌套结构会损坏——已知工程债，ingest.py L51-L59）。

---

## 5.4 构建索引（ingest 建库）

```bash
# 首次运行必须先建索引
python3 main.py ingest
```

**流程**（[ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py) `main` L239-L273）：

```
load_markdown_files (L26-83)          读取 .opencode/context/ 下全部 md（排除 navigation.md）
        ↓
split_documents (L86-110)             中文优化分块（RecursiveCharacterTextSplitter 500/50）
        ↓
build_faiss_index (L113-144)          稠密向量 → faiss_index/index.faiss（幂等：已存在则跳过 L125）
        ↓
build_bm25_index (L147-197)           稀疏向量 → bm25_index.pkl（jieba 分词；已存在则加载 L158）
        ↓
check_indexes (L222-235)              校验两个索引文件完整性
```

**产物与持久化策略**：

| 索引文件 | 格式 | 内容 | 是否可重建 |
|----------|------|------|-----------|
| `faiss_index/index.faiss` | FAISS 二进制 | 全部 chunk 的 768 维稠密向量 | ✅ `ingest` 重建 |
| `bm25_index.pkl` | pickle | jieba 分词后的稀疏词频统计 | ✅ `ingest` 重建 |

> **关键机制——索引持久化**：`build_faiss_index` 在索引已存在时直接加载不重建（[ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L113-L144) L125），`build_bm25_index` 同理（[ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L147-L197) L158）。因此**知识库新增文档后，需显式重跑 `python3 main.py ingest` 才会把新内容纳入检索**（索引是运行时生成、落盘持久化，非每次启动自动重建）。启动时由 [retriever.load_indexes](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L147-L172)（L147-L172）加载。


## 5.5 五种使用方式

入口统一在 [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L114-L149)（`main` L114-L149），按子命令分发：

| # | 子命令 | 模式函数 | 行号 | 用途 | 交互方式 |
|---|--------|----------|------|------|----------|
| 1 | `cli` | `mode_cli` | [L46](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L46) | 交互式问答 | 终端循环问答 |
| 2 | `ask` | `mode_ask` | [L58](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L58) | 单次查询 | 命令行传参即问即答 |
| 3 | `feishu` | `mode_feishu` | [L91](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L91) | 飞书机器人 | WebSocket 长连接，持续服务 |
| 4 | `ingest` | `mode_ingest` | [L52](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L52) | 重建索引 | 见 5.4 |
| 5 | `stats` | `mode_stats` | [L97](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L97) | 运行统计 | 打印检索/调用计数 |

```bash
# 1. CLI 交互模式（终端持续问答，/help /stats /reload /ingest /debug 内置命令）
python3 main.py cli

# 2. ask 单次查询（适合脚本化/管道）
python3 main.py ask "GPU-A100-01 的显存是多少"

# 3. feishu 长连接（后台常驻，见 5.6）
python3 main.py feishu

# 4. ingest 重建索引（知识库变更后执行）
python3 main.py ingest

# 5. stats 查看统计
python3 main.py stats
```

> CLI 内置命令（[cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62-L149) `main` L62-L149）：`/help` L108、`/stats` L110、`/reload` L120（热重载索引，对应 [orchestrator.reload](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L178-L184) L178-L184）、`/ingest` L123、`/debug` L131。交互 BANNER 见 [cli.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L18-L27) L18-L27，帮助文本 L29-L59。

---

## 5.6 飞书接入步骤（WebSocket 长连接，无需公网 IP）

### ① 创建飞书应用

1. 打开 [飞书开放平台](https://open.feishu.cn/) → 开发者后台 → 创建企业自建应用
2. 记录 **App ID** 与 **App Secret**，填入 `.env`：
   ```
   LABQA_FEISHU_APP_ID=cli_xxxxxx
   LABQA_FEISHU_APP_SECRET=xxxxxx
   ```
3. 在「权限管理」中开通以下权限：

| 权限代码 | 用途 |
|----------|------|
| `im:message:send_as_bot` | 以机器人身份发送消息（回复用户） |
| `im:message:read` | 读取用户发来的消息 |

### ② 启动长连接

```bash
python3 main.py feishu
```

连接逻辑见 [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py)：

| 组件 | 行号 | 说明 |
|------|------|------|
| `start` 主循环 | [L93-L180](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93-L180) | 建立 WS 长连接 + 自动重连 |
| `on_message` 事件回调 | [L124-L160](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L124-L160) | 收到消息 → 处理 → 回复 |
| `_handle_message` | [L59-L75](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L59-L75) | 去重过滤后走 Orchestrator |
| `_send_message` | [L78-L90](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L78-L90) | 发送回复（超长文本 19900 字符截断，L83） |
| WS 客户端构建 | [L162-L169](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L162-L169) | 长连接握手参数 |

### ③ 与旧 Webhook 方式的对比

| 维度 | **WebSocket 长连接（v2 现行）** | 旧 Webhook（legacy） |
|------|------|------|
| 网络要求 | **无需公网 IP、无需域名**（客户端主动出站连接） | 需要公网可访问的回调 URL（内网需 ngrok 穿透） |
| 事件获取 | SDK 长连接推送 | 飞书 HTTP POST 回调 |
| 实现位置 | [feishu_ws.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93-L180) | `scripts/feishu-webhook.py`（独立脚本，**自带关键词匹配、绕过 labqa/ 包**，不用于新开发） |
| 消息去重 | 内置两层去重（见 5.8） | 无 |
| 兼容别名 | `python3 main.py webhook` 实际启动的也是 WS 长连接（兼容别名） | — |

> 实验室内网场景（无公网 IP）用 WS 长连接即可；旧 `scripts/feishu-webhook.py` 仅作历史遗留参考。

---

## 5.7 验证与测试

### ① 自动化测试（pytest，18 个用例）

```bash
pytest tests/ -v
```

测试文件 [tests/test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py)（320 行，7 个 Test 类，约 18 个用例）：

| Test 类 | 行号 | 覆盖范围 |
|---------|------|----------|
| `TestConfig` | [L17](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L17) | .env 配置加载与三级回退 |
| `TestIngest` | [L39](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L39) | 文档加载、分块、frontmatter 解析 |
| `TestRouter` | [L89](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L89) | 意图识别与关键词打分路由 |
| `TestRRF` | [L157](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L157) | RRF 融合算法（1/(60+rank)） |
| `TestPrompts` | [L203](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L203) | 各 Agent Prompt 模板装配 |
| `TestGenerator` | [L247](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L247) | LLM/Embedding 初始化与离线降级 |
| `TestAgents` | [L284](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L284) | 三类 Agent 的检索与回答 |

> ⚠️ **文档不一致说明**：`AGENTS.md` L73 仍写"no test framework"，但 v2.0 重构（commit `757df23`）已引入 pytest 用例（pyproject 已配置 pytest、README 有测试说明）。**以实际代码为准**：`tests/test_main.py` 存在且可运行。

### ② 手动测试清单（docs/TESTING.md，8 大节）

[TESTING.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/TESTING.md)（152 行）给出 8 大节手工验收清单，覆盖：

| 节 | 验收内容 |
|----|----------|
| 1 | 环境启动：venv 激活、依赖安装、ingest 建索引成功 |
| 2 | CLI 交互：启动 BANNER、`/help`、提问有回复 |
| 3 | ask 单次查询：答案含检索来源 |
| 4 | 路由准确性：设备/项目/知识类问题被正确分流 |
| 5 | 混合检索质量：语义近似词命中（FAISS）与精确词命中（BM25） |
| 6 | 飞书端到端：长连接启动、发消息收回复、@机器人 场景 |
| 7 | 降级与兜底：LLM 离线 → 离线模式响应；BM25 无结果 → 纯向量降级 |
| 8 | 统计与热重载：`stats` 计数、知识库更新后 `/reload` 生效 |

### ③ WORKFLOW.md 示例走读

[docs/WORKFLOW.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md)（506 行）包含完整问答流程走读，可用于人工验证链路：Router 打分细节（[L147-L182](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L147-L182)）与端到端示例（[L186-L416](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L186-L416)）。按示例逐条提问，比对预期输出即可完成冒烟测试。

---

## 5.8 常见问题排查表

| 现象 | 排查步骤 | 对应代码/文档 |
|------|----------|---------------|
| **启动即报索引不存在** | ① 是否先跑过 `python3 main.py ingest`？② 检查 `faiss_index/index.faiss` 与 `bm25_index.pkl` 是否落盘 ③ 重新执行 ingest 后重启 | [retriever.load_indexes](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L147-L172) L147-L172 |
| **知识库改了但回答没变化** | 索引持久化机制：已存在则加载不重建。需显式重跑 `python3 main.py ingest`（或 CLI 内 `/ingest` + `/reload`） | [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L113-L144) L125 / [ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L147-L197) L158 |
| **LLM 初始化失败（ollama 模式）** | ① `ollama list` 确认模型已拉取 ② 确认 `ollama serve` 在运行 ③ 设置 `LABQA_LLM_API_KEY="offline-mode"` 进入**离线模式**（不调 LLM，直接返回检索原文）兜底 | [generator.create_llm](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99-L118) L99-L118 |
| **cloud 模式 401/超时** | ① 检查 `LABQA_LLM_API_KEY` 是否配置（三级回退：LABQA_LLM_API_KEY → DEEPSEEK_API_KEY → OPENAI_API_KEY）② 检查 `LABQA_LLM_API_BASE` 端点 ③ 确认网络可达 | [config.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32-L35) L32-L35 |
| **BM25 无结果导致回答空** | 这是**设计内降级**：BM25 无结果时自动降级为纯 FAISS 向量检索，无需处理 | [retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L212-L216) L212-L216 |
| **飞书消息重复回复** | 内置两层去重：message_id 集合 + 30 秒内容哈希窗口（补偿 SDK 重试），正常无需干预；若仍重复检查是否多实例同时跑 `feishu` | [feishu_ws._is_duplicate](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28-L56) L28-L56 |
| **飞书收不到消息** | ① 确认 `.env` 已填 `LABQA_FEISHU_APP_ID/SECRET` ② 确认应用已发布/启用机器人 ③ 确认权限 `im:message:send_as_bot` + `im:message:read` 已开通 ④ 确认用 WS 长连接（无需公网 IP，勿再配 webhook 回调） | [.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L26-L31) L26-L31 |
| **飞书回复被截断** | 单条消息超过 19900 字符会被截断（飞书消息长度限制），属设计行为 | [feishu_ws._send_message](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L78-L90) L83 |
| **意图路由不准** | 关键词路由是硬编码：需同时改 [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L35-L44) 的 `DEVICE_KEYWORDS`（L35-L44）/`PROJECT_KEYWORDS`（L46-L56）/`KNOWLEDGE_KEYWORDS`（L58-L69）与对应 Agent 的关键词提取方法 | [router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L35-L69) L35-L69 |
| **转换管道不生效** | 依赖 dev extras 中的 python-docx/python-pptx/openpyxl/pypdf，若 `pip install -r requirements.txt` 安装则缺失（被注释）——改用 `uv pip install -e ".[dev]"` | `scripts/convert.sh` |
| **pytest 报模块找不到** | 从项目根运行 `pytest tests/ -v`；[main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L24) L24 有 `sys.path` 注入，勿在子目录执行 | [tests/test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py) |


In [ ]:
# ============================================================
# 5.x 常用命令速查（注释形式，按需在终端执行，本单元格仅作说明）
# ============================================================

# ── ① 环境准备 ──────────────────────────────────────────────
#   uv venv --python 3.9                 # 创建虚拟环境
#   source .venv/bin/activate            # 激活
#   uv pip install -e ".[dev]"           # 安装依赖（含 dev extras，推荐）
#   pip install -r requirements.txt      # 备选：仅运行时依赖

# ── ② 配置 ──────────────────────────────────────────────────
#   cp .env.example .env                 # 生成配置
#   # 编辑 .env：LABQA_LLM_MODE=ollama|cloud
#   # cloud 模式必填：LABQA_LLM_API_KEY（或复用 DEEPSEEK_API_KEY）

# ── ③ 可选：本地 Ollama ─────────────────────────────────────
#   ollama pull deepseek-r1:1.5b         # 推理模型
#   ollama pull nomic-embed-text         # Embedding 模型
#   ollama list                          # 验证

# ── ④ 构建索引（知识库变更后必须重跑）───────────────────────
#   python3 main.py ingest               # 生成 faiss_index/ + bm25_index.pkl

# ── ⑤ 使用入口 ──────────────────────────────────────────────
#   python3 main.py cli                  # 交互式问答（/help /stats /reload /ingest /debug）
#   python3 main.py ask "问题文本"       # 单次查询
#   python3 main.py feishu               # 飞书长连接（常驻，无需公网 IP）
#   python3 main.py stats                # 运行统计

# ── ⑥ 验证与测试 ────────────────────────────────────────────
#   pytest tests/ -v                     # 18 个用例（7 个 Test 类）
#   # 手动验收：docs/TESTING.md（8 大节）+ docs/WORKFLOW.md 示例走读


# 第六部分：核心逻辑深度解析

> 本部分从源码反推 LabQA-RAG v2.0 的核心实现原理：意图识别如何零成本路由、FAISS + BM25 + RRF 混合检索为何互补、RRF 数学上为什么有效、双轨 LLM 如何适配、Multi-Agent 如何处理混合意图、飞书长连接如何免公网 IP。所有行号均已对照源码核验。

## 6.1 完整问答链路流程图

一条用户提问从飞书/CLI 进入，到返回带来源标注的答案，完整链路如下：

```
用户提问（飞书 @机器人 / CLI ask / 一次性问答）
   │
   ▼
① Router 意图识别  recognize_intent()
   │  三路关键词打分：device / project / knowledge
   │  max_score==0 → UNKNOWN（走全量兜底）
   │  次要意图 ≥ 主意图×50% → MIXED
   ▼
② Orchestrator 分发  Orchestrator.process()
   │  UNKNOWN → _unknown_intent_response() 全量混合检索兜底
   │  单意图 → 路由到对应 Agent（Device/Project/Knowledge）
   │  MIXED  → 主 Agent 回答 + 次要 Agent 回答拼接
   ▼
③ Agent 混合检索  BaseAgent.search_context()
   │  category 过滤：DeviceAgent→devices/  ProjectAgent→projects/  KnowledgeAgent→全量
   │  ├─ FAISS 稠密检索（语义相似）    dense_search()
   │  ├─ BM25  稀疏检索（关键词精确）  sparse_search()
   │  └─ RRF 融合（1/(60+rank) 累加）  reciprocal_rank_fusion()
   │     BM25 无结果 → 降级纯向量检索
   ▼
④ Prompt 构建  get_agent_prompt() + generate_answer()
   │  上下文拼接（[source] 前缀 + 分隔符）→ 填入 Agent 专用 Prompt 模板
   ▼
⑤ LLM 生成  generate_answer()
   │  ollama 模式 → OllamaChat.predict()（本地免费）
   │  cloud  模式 → ChatOpenAI.invoke()（DeepSeek/OpenAI 兼容）
   ▼
⑥ 来源标注  generate_answer() 提取 sources + BaseAgent.answer() 去重拼接
   │  📎 来源: devices/GPU-A100-01.md, ...
   ▼
返回给用户（飞书文本消息 / CLI 终端 / 函数返回值）
```

**各环节源码位置对照表**：

| # | 环节 | 核心函数 | 源码位置 |
|---|------|---------|---------|
| ① | 意图识别 | `recognize_intent()` / `extract_keywords()` / `_score_intent()` | [router.py#L150-L207](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150-L207) / [router.py#L86-L129](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L86-L129) / [router.py#L132-L147](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L132-L147) |
| ② | 主调度分发 | `Orchestrator.process()` / `_unknown_intent_response()` | [orchestrator.py#L71-L130](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L71-L130) / [orchestrator.py#L132-L165](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L132-L165) |
| ③ | 混合检索 | `hybrid_search()` / `dense_search()` / `sparse_search()` / `reciprocal_rank_fusion()` | [retriever.py#L175-L229](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175-L229) / [retriever.py#L19-L52](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L19-L52) / [retriever.py#L57-L101](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L57-L101) / [retriever.py#L106-L142](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106-L142) |
| ③′ | Agent 检索入口 | `BaseAgent.search_context()` | [agents/base.py#L35-L69](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L35-L69) |
| ④ | Prompt 构建 | `get_agent_prompt()` / `generate_answer()` | [prompts.py#L93-L100](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L93-L100) / [generator.py#L123-L199](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L123-L199) |
| ⑤ | LLM 生成 | `create_llm()` / `generate_answer()` | [generator.py#L99-L118](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99-L118) / [generator.py#L123-L199](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L123-L199) |
| ⑥ | 来源标注 | `generate_answer()` sources 提取 + `BaseAgent.answer()` 拼接 | [generator.py#L188-L190](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L188-L190) / [agents/base.py#L101-L104](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L101-L104) |
| 入口 | 飞书消息接入 | `_handle_message()` / `on_message()` | [feishu_ws.py#L59-L75](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L59-L75) / [feishu_ws.py#L124-L160](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L124-L160) |

> **设计要点**：Router 是"轻量级前端"（纯关键词打分，零 LLM 成本、毫秒级），Agent 是"专业后端"（RAG 检索 + LLM 生成）。意图识别从不调用 LLM，这是 20 人实验室场景下"快"的关键——一次问答只有最后一步 LLM 生成消耗算力。

---


## 6.2 意图识别与路由（Router）

Router 的职责：判断"用户想问设备 / 项目 / 知识库 / 混合 / 未知"，输出 `RoutingDecision`（意图 + 置信度 + 目标 Agent + 匹配词 + 次要意图）。三个关键词库定义于 [router.py#L35-L69](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L35-L69)（`DEVICE_KEYWORDS` / `PROJECT_KEYWORDS` / `KNOWLEDGE_KEYWORDS`）。

### 6.2.1 extract_keywords：中文口语的分词策略

[extract_keywords()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L86-L129) 不用 jieba（那属于检索侧），而是针对"意图识别只需要粗粒度词面"设计的三步流水线：

```python
# ① 中文标点替换 + 切分（router.py L116-119）
for part in text.replace("，", " ").replace("？", " ").replace("。", " ").split():
    if len(part) >= 2:            # 过滤单字虚词：的/吗/呢/啊
        words.add(part.lower())   # 统一小写：GPU 与 gpu 同词

# ② 设备型号实体正则（router.py L125-126）
model_pattern = re.findall(r'[a-zA-Z]\d+', text)   # "A100" "H800" ✅，纯数字 "4090" ❌
words.update(m.lower() for m in model_pattern)
```

**原理拆解**：

1. **中文没有空格**，不能像英文 `text.split()` 直接分词。先把中文逗号/问号/句号替换成空格，再用 `split()` 切出"词块"。注意只处理了三种标点（`,` `?` `。`），感叹号、冒号、顿号不在其列——这是刻意取舍：意图识别只需粗粒度匹配，漏切几个词块影响不大。
2. **`len(part) >= 2` 过滤单字**：中文里"的/吗/呢/啊"等单字虚词无信息量，直接丢弃。
3. **正则 `[a-zA-Z]\d+` 补捉设备型号**：`"GPU-A100-01 还有空闲卡吗？"` 切分后得到 `["gpu-a100-01", "还有空闲卡吗"]`，`"a100"` 被包在长词块里，若不做实体提取，`A100` 就永远匹配不上 `DEVICE_KEYWORDS` 里的独立 `"a100"`。正则从原始文本中额外捞出 `["A100"]` → 转小写 `"a100"` 补进集合。**`4090` 不会被匹配**（要求字母开头），这是注释中明确记录的已知限制。
4. 全程用 `set` 去重，返回无序列表。

### 6.2.2 _score_intent：长短关键词双通道加权

[_score_intent()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L132-L147) 对每个意图计算整数分数，**每个关键词有两条得分通道，可叠加**：

```python
for kw in intent_keywords:
    if kw_lower in text_lower:          # 通道A：子串包含
        score += 1 if len(kw) <= 3 else 2   # 短词+1，长词+2
    if kw_lower in keywords:            # 通道B：extract_keywords 分词精确命中
        score += 1
```

| 规则 | 分值 | 设计意图 |
|------|------|---------|
| 短关键词（≤3 字）子串命中 | +1 | 短词信息量低（"gpu"、"卡"），容易误命中 |
| 长关键词（>3 字）子串命中 | +2 | 长词信息量高（"激光雷达"、"提交规范"），命中即强证据 |
| 关键词在分词结果中精确出现 | +1 | 说明该词是独立的"实义词块"，而非长串的偶然包含 |

**双通道为什么叠加**：`"gpu"` 同时满足子串命中（+2，长度 3）和分词精确命中（+1），合计 +3——两个证据互相印证，信号更强。以 WORKFLOW.md 示例 `"GPU-A100-01 还有空闲卡吗？"` 走读：`"gpu"` 子串+2、分词+1，`"a100"` 子串+2、分词+1，`"空闲"` 子串+1 → **device 总分 7**（WORKFLOW 文档示例记 6 分，因当时分词结果略有差异；此处以源码逻辑为准）。

### 6.2.3 recognize_intent：三路打分 + 混合意图判定

[recognize_intent()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150-L207) 的完整决策逻辑：

```
① 三路打分：scores = {DEVICE: _, PROJECT: _, KNOWLEDGE: _}          （L161-165）
② max_score == 0 → UNKNOWN，confidence=0.0，target_agent="none"     （L167-173）
③ primary = 第一个达到 max_score 的意图                            （L175）
④ secondary = 其他意图中 score>0 且 score ≥ max_score×0.5 的       （L178-181）
⑤ confidence = min(max_score / 5.0, 1.0)                            （L191）
⑥ 有 secondary → intent=MIXED，否则 intent=primary                 （L202）
```

**三个关键参数的设计理由**：

1. **混合意图阈值 50%**（`v >= max_score * 0.5`）：分数由"词面命中"构成，次要意图能拿到主意图一半以上的分数，说明问题在词面上真实横跨两个领域（如"GPU 服务器和 LLM 对齐项目哪个更紧急"——`gpu/服务器` 命中 device，`项目/对齐/llm` 命中 project）。若阈值设太低（如 30%），噪声词（如"测试"同时存在于 DEVICE 和 PROJECT 关键词库）会频繁触发伪混合；设太高则漏掉真正的跨域问题。50% 是"有实际词面重叠"的合理经验值。
2. **置信度 `min(max_score/5, 1)`**：5 分即满置信度。设计含义是"≥5 个独立证据词（或等权组合）即可 100% 信任路由结果"。低分如 1-2 分时置信度只有 0.2-0.4，为后续兜底策略留出解释空间。它是**启发式软信号**，当前版本 Orchestrator 未用置信度做分支（只用于日志与扩展），但保留了未来做"低置信度 → 全量检索"改造的钩子。
3. **三路同源关键词**：`DEVICE/PROJECT/KNOWLEDGE` 之间有交叉词（如 `dpo`、`rlhf` 同时出现在 PROJECT 与 KNOWLEDGE 关键词库）——这正是需要"混合意图"机制的原因，交叉词天然产生双高分数。

**路由决策输出**（[RoutingDecision](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L24-L31)）：意图 + 置信度 + `AGENT_MAP` 映射出的目标 Agent（[router.py#L72-L76](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L72-L76)）+ 匹配到的关键词 + 次要意图列表。`get_intent_category()`（[router.py#L210-L212](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L210-L212)）将意图映射到知识库分类目录（device→devices、project→projects、knowledge→None 全量）。

> **为什么关键词路由而非 LLM 路由**：LLM 路由准确但每次问答多一次模型调用（成本 + 延迟 + 抖动）。实验室 ~20 人、问题域固定（设备/项目/论文），手工维护三张关键词表即可覆盖绝大多数查询。router.py 顶部注释明确写了演进路径："保留关键词匹配作为轻量级前置路由，未来可扩展为 LLM 路由"。

---


## 6.3 混合检索详解（FAISS + BM25 + RRF）

### 6.3.1 FAISS 稠密检索：语义匹配

[dense_search()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L19-L52) 调用 LangChain FAISS 的 `similarity_search_with_relevance_scores` 做向量相似度检索：

```python
if category_filter:
    docs_with_scores = vectorstore.similarity_search_with_relevance_scores(query, k=k * 3)  # L32-34
# 遍历结果，过滤 category 不匹配的文档，凑满 k 条（L40-46）
```

- **无 category 过滤**：直接取 Top-K。
- **有 category 过滤**：先取 `k*3` 条再后过滤。**为什么多取**：FAISS 索引按全部 chunk 构建，无法在索引内部做 metadata 过滤，只能"取多了再筛"。取 `k*3` 是为了过滤后仍能凑满 k 条（devices 目录只有 4 个文档时尤其必要——多取能覆盖同一文档的多个分块）。
- **relevance score**：LangChain 对 FAISS 的 `similarity_search_with_relevance_scores` 返回的是归一化相关性（0~1，越大越相关），与裸余弦距离方向相反——代码在 `float(score)` 处保留原样传递，RRF 阶段根本不用这个数值（见 6.3.3）。

**FAISS 语义检索的长处**：同义改写。用户问"显卡还能用吗"，向量上能命中写着"GPU-A100-01 运行正常"的文档块。**短处**：精确专有名词（`GPU-A100-01`、`A800`）在 embedding 空间被语义稀释，容易与相似设备混淆排名。

### 6.3.2 BM25 稀疏检索：关键词精确匹配

[sparse_search()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L57-L101) 基于 rank_bm25 的 `BM25Okapi` 做倒排检索：

```python
import jieba
tokenized_query = list(jieba.cut(query.lower()))          # L76  中文分词
# ImportError 时回退：re.findall(r'[一-鿿]|[a-zA-Z]+|\d+', ...)  # L78-79 字符级
scores = bm25.get_scores(tokenized_query)                  # L84  对全语料打分
indexed.sort(key=lambda x: x[1], reverse=True)             # L87-88
# score<=0 直接丢弃；同样 k*3 多取再按 category 过滤（L91-99）
```

- **jieba 中文分词**：构建索引时（见 6.4）与查询时用**同一套分词器**，保证倒排索引的 term 对齐。无 jieba 时退化为"单字符 + 英文单词 + 数字"的正则分词（`[一-鿿]` 匹配单个汉字），召回变差但不崩溃——降级友好的工程风格。
- **`score <= 0` 过滤**：BM25 对未命中的文档得分可能为负或 0，直接丢弃，避免无意义条目进入 RRF。
- **BM25 分数无界**：它本质是词频 TF × 逆文档频率 IDF 的加权和，数值可达几十——这正是不敢拿 BM25 原始分数与 FAISS 相关性分数直接加权平均的原因（见 6.3.3）。

**BM25 的长处**：`GPU-A100-01` 这种精确标识符在倒排索引中一击即中；对"空闲/坏了/维修"等强领域词也敏感。**短处**：无法处理同义改写（"显卡"≠"GPU"），且未在文档中出现过的词（如问"显存"而文档只写"内存"）直接失配。

### 6.3.3 RRF 融合：为什么它数学上有效

[reciprocal_rank_fusion()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106-L142) 的核心只有三行：

```python
for retriever_results in results_list:
    for rank, item in enumerate(retriever_results, start=1):
        doc_key = item["doc"].page_content                      # 以 page_content 为唯一标识
        doc_scores[doc_key]["rrf_score"] += 1.0 / (k + rank)    # k = RRF_K = 60
```

**数学原理**：

```
RRF score(doc) = Σ_{r ∈ 各检索器}  1 / (k + rank_r(doc))
```

其中 `rank_r(doc)` 是 doc 在第 r 路检索结果中的排名（从 1 开始）。**关键设计：只用排名，不用原始分数**。

| 特性 | FAISS relevance score | BM25 score | RRF score |
|------|----------------------|------------|-----------|
| 取值范围 | 0~1（归一化） | 0~∞（无界） | 有界（每路 ≤ 1/61，两路 ≤ 0.0328） |
| 跨路可比性 | ❌ 与 BM25 不可比 | ❌ 与 FAISS 不可比 | ✅ 各路径天然同量纲 |
| 对分数标定的敏感性 | 受 embedding 模型影响 | 受语料词频分布影响 | 无（只看次序） |

**RRF 为什么有效**：两路检索器的分数分布完全不同（一个 0~1 相关性、一个无界 TF-IDF 权重），直接加权平均需要调权重且脆弱；RRF 把"分数"问题转化为"排名"问题——排名的含义是跨检索器一致的（第 1 名就是第 1 名）。**只排名、不比分**，因此对异构检索器鲁棒。

**为什么 k=60**：k 是平滑常数，控制"排名差异"的影响力：

```
k=1 时：rank1 贡献 1/2=0.5，rank5 贡献 1/6≈0.167  → 3 倍差距，强烈偏向各自榜单前列
k=60 时：rank1 贡献 1/61≈0.0164，rank5 贡献 1/65≈0.0154 → 仅 1.065 倍，几乎无差别
```

- k 越小，越信任"单路检索器内部的排名顺序"；
- k 越大，越倾向"只要被任一路排在较前即可"，融合结果主要由**跨路共识**决定。
- 每路只取 5 条（`RETRIEVE_K=5`，见 [config.py#L42-L46](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L42-L46)）时，k=60 让 rank1~rank5 的权重几乎相等——相当于把"两路都命中"作为最强信号。实证上 60 是 RRF 论文与业界常用的经验值（原始论文建议 60，兼顾鲁棒性与区分度）。

**数值例子**（与 6.3 末尾可运行演示一致）：

```
文档 X 在 FAISS 排第1、BM25 排第1 → RRF = 1/61 + 1/61 ≈ 0.0328  ← 两路共识，最高
文档 Y 在 FAISS 排第3、BM25 排第2 → RRF = 1/63 + 1/62 ≈ 0.0320  ← 两路共识，紧随
文档 Z 仅在 FAISS 排第2           → RRF = 1/62 ≈ 0.0161          ← 单路命中，被腰斩
```

**融合的胜负手是"跨路共识"而非"单路最高分"**——这正是一个鲁棒检索器该有的行为：单路的高排名可能是偶然，两路同时看好才是真相关。

**以 `page_content` 为唯一标识**：同一文档的不同 chunk 内容不同，天然区分；FAISS 与 BM25 共享同一批 chunk（入库时同源），page_content 完全一致，因此能精确跨路匹配累加。代价是内容完全相同的两个 chunk 会被合并——在实验室知识库场景（Markdown 分块，内容基本唯一）是合理的。

### 6.3.4 hybrid_search：完整编排与降级

[hybrid_search()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175-L229) 编排整个混合检索：

```
Step 1  两路并行检索：dense_search() + sparse_search()，每路 k=RETRIEVE_K=5   （L205-206）
Step 2  BM25 无结果 → 降级为纯向量检索，直接返回 dense_results[:top_k]         （L212-216）
Step 3  两路结果进 RRF，k=RRF_K=60 融合                                       （L219）
Step 4  返回 fused[:top_k]（top_k=FUSION_K=5）的 Document 列表                （L229）
```

**BM25 无结果为何降级纯向量**（L212-216）：jieba 切出的查询词在语料中完全不存在时（如罕见缩写、纯英文型号），BM25 全部分数为 0，那一路是纯噪音。若仍强行进 RRF，等于把"一个全是 0 的榜单"混入融合，会稀释稠密路的有效信号。直接返回向量结果，保证用户永远拿到至少一路的检索结果——**可用性优先的降级哲学**。

**search_by_category()**（[retriever.py#L232-L245](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L232-L245)）是 Agent 分类检索的入口：把 `category_filter` 透传给 `hybrid_search`，两路检索内部各自做分类过滤（各自 k*3 多取 → 过滤 → 取 k），保证 Agent 只看到自己领域的文档。

---


## 6.4 知识库入库管道（Ingestion Pipeline）

入口：`python -m labqa.ingest`（[ingest.py#L239-L273](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L239-L273)），或 `main.py ingest`。完整管道：

```
.opencode/context/*.md（8 个文档，frontmatter）
   ↓ load_markdown_files()   扫描 + 解析 frontmatter + 定 category
Document 列表（整篇文档为一条）
   ↓ split_documents()       RecursiveCharacterTextSplitter（500/50 字符）
Chunk 列表
   ├→ build_faiss_index()    FAISS.from_documents → faiss_index/index.faiss
   └→ build_bm25_index()     jieba 分词 → BM25Okapi → bm25_index.pkl（pickle）
```

### 6.4.1 load_markdown_files：扫描、frontmatter、分类

[load_markdown_files()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L26-L83) 三步：

1. **扫描**：`base_dir.rglob("*.md")` 递归找全部 Markdown；**跳过 `navigation.md`**（[L39](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L39)）——它是给人/Agent 看的总索引，不是可检索内容。
2. **frontmatter 手写解析**（L51-59）：不用 YAML 库，`raw.split("---", 2)` 切出首尾 `---` 之间文本，逐行 `partition(":")` 拆 `key: value`。**这是刻意的轻量实现**：知识库 frontmatter 只有 `type/tags/updated` 三个简单字段，够用且零依赖。代价是嵌套 YAML（列表、对象）会解析错——已记录为工程债。
3. **分类**：`relative.split("/")[0]` 取**一级目录名**作 category（`devices/GPU-A100-01.md` → `devices`）。知识库目录结构即分类体系，无需手工标注。
4. **metadata 6 字段**：`source`（相对路径，也是来源标注的显示名）/ `filename` / `category` / `type` / `tags` / `updated`（后三者来自 frontmatter，缺省为空串）。

### 6.4.2 split_documents：中文友好的递归分块

[split_documents()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L86-L110) 用 `RecursiveCharacterTextSplitter`，分隔符链是中文优化的关键：

```python
RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "！", "？", "；", " ", ""],   # 按优先级尝试
    chunk_size=500, chunk_overlap=50,
    length_function=len,          # 按字符数而非 token 计长
    add_start_index=True,
)
```

**为什么这个分隔符链有效**：`RecursiveCharacterTextSplitter` 从链头开始，用第一个能切的分隔符切——即**优先在段落边界（`\n\n`）切，其次是行（`\n`），再退到句子边界（`。！？；`）**，最后才用空格和"硬切"兜底。中文不像英文有天然空格分词，若分隔符链只含 `\n`，长段落会被硬切在任意字符处，破坏语义；把 `。！？；` 放进链中，保证 chunk 边界大概率落在完整句子上。500 字符约合 250 个中文词（中文 1 字 ≈ 0.5~1 token），是 embedding 模型（nomic-embed-text，8192 上下文）完全能消化的长度；overlap 50 字符让相邻 chunk 共享上下文，缓解"句子被切成两半导致的信息断裂"。

### 6.4.3 双索引构建与持久化

**FAISS**（[build_faiss_index()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L113-L144)）：

```
已存在 faiss_index/index.faiss 且未 force_rebuild → FAISS.load_local() 直接加载（L125-131）
否则 → FAISS.from_documents(chunks, embed_model) → save_local() 持久化（L138-142）
```

- `allow_dangerous_deserialization=True`：LangChain 新版本对 pickled FAISS 的反序列化安全要求，本地单机场景可接受。
- **幂等设计**：索引已存在就加载、不重建。知识库更新后需显式 `--rebuild` 或运行 `reload` 命令（[orchestrator.py#L178-L184](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L178-L184)）。

**BM25**（[build_bm25_index()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L147-L197)）：

```python
tokenized_corpus = [list(jieba.cut(text.lower())) for text in doc_texts]   # L183
bm25 = BM25Okapi(tokenized_corpus)                                          # L184
bm25_data = {"bm25": bm25, "documents": chunks, "doc_texts": ..., "tokenized_corpus": ...}  # L186-191
pickle.dump(bm25_data, open(BM25_INDEX_PATH, "wb"))                         # L193-194
```

- **pickle 整个 dict 而非只存模型**：`documents`（chunk 对象含 metadata）与 `tokenized_corpus` 一并持久化，运行时 `sparse_search()` 直接取用（[retriever.py#L70-L71](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L70-L71)），省去重新分词。
- **分词器降级**：jieba 不可用时回退字符级正则分词（L174-180），索引与查询用同一回退逻辑，保证两侧一致。
- **为什么双索引缺一不可**（v2 重构动机，commit `757df23`）：v1 仅关键词匹配，召回差；只上 FAISS 则精确标识符（`GPU-A100-01`、`A800`）语义检索易失配；只上 BM25 则同义改写全丢。两者互补覆盖"语义"与"词面"两个维度，RRF 负责合并。

**索引加载**（[retriever.py#L147-L172](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L147-L172)）：`load_indexes()` 分别检查两个索引文件是否存在并加载，任一失败打印警告但不中断——Orchestrator 据此进入"离线模式"（仅意图识别、无检索），由 `_unknown_intent_response` 等兜底路径处理（[orchestrator.py#L38-L41](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L38-L41)）。

---


## 6.5 双轨 LLM 与生成（Generator）

双轨架构的开关是 `LABQA_LLM_MODE`（[config.py#L25](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25)）：`ollama`（本地免费，默认）或 `cloud`（DeepSeek/OpenAI 兼容 API）。

### 6.5.1 Embedding 双轨

[create_embeddings()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L79-L96)：

| 模式 | 实现 | 模型 | 特点 |
|------|------|------|------|
| ollama | `OllamaEmbeddings`（自写适配器） | `nomic-embed-text` | 本地推理，零 API 成本，维度 768，中文效果够用 |
| cloud | `langchain_openai.OpenAIEmbeddings` | `LABQA_LLM_MODEL`（deepseek-chat 同配置） | 调云端兼容 API |

[OllamaEmbeddings](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L31-L43) 是 12 行的轻量适配器：继承 LangChain `Embeddings` 抽象基类，`embed_documents` 循环复用 `embed_query`，后者调 `ollama.embeddings()`。**用最小的胶水代码让本地模型接入 LangChain 生态**。

注意 `create_embeddings` 中 cloud 分支的 base_url 处理（[L94](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L94)）：`rstrip("/") + "/v1" if "/v1" not in LLM_API_BASE else LLM_API_BASE`——防止 `https://api.deepseek.com/v1` 被拼成 `/v1/v1`（DeepSeek 官方 base 常自带 `/v1`）。

### 6.5.2 LLM 双轨

[create_llm()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99-L118)：

| 模式 | 实现 | 默认模型 | 调用方式 |
|------|------|---------|---------|
| ollama | `OllamaChat`（自写适配器） | `deepseek-r1:1.5b` | `ollama.chat()` 本地推理 |
| cloud | `langchain_openai.ChatOpenAI` | `deepseek-chat` | OpenAI 兼容协议，天然兼容 DeepSeek/OpenAI |

[OllamaChat](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L48-L74) 的两个设计点：

1. **`invoke(messages)` 做 LangChain → Ollama 消息格式转换**（L54-69）：LangChain 的 `SystemMessage/HumanMessage/AIMessage` 有 `.type` 属性，据此映射到 Ollama 的 `system/user/assistant` role。
2. **`predict(text)` 简化接口**（L71-74）：包一层 `HumanMessage` 直接发文本。`generate_answer` 中通过 `isinstance(llm, OllamaChat)` 分支（[generator.py#L175-L185](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L175-L185)）：Ollama 走 `predict(完整prompt)`（整个 prompt 作为一条 user 消息）；ChatOpenAI 走标准的 `SystemMessage(prompt) + HumanMessage(question)` 两条消息——**Ollama 小模型对"系统指令 + 用户问题"分条消息的遵循度不如一条完整 prompt**，这是针对 1.5B 小模型的务实选择。

### 6.5.3 generate_answer：RAG 生成流水线

[generate_answer()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L123-L199) 四步：

```
Step 1  上下文拼接（L148-151）：
        context = "[{source}]\n{page_content}" 用 "\n\n---\n\n" 连接所有 docs
        → 每段都带来源路径，LLM 可据此回答"引用来源放最后一行"
Step 2  Prompt 构建（L154-163）：system_prompt_template（Agent 专用）或默认 PROMPT_TEMPLATE
Step 3  调 LLM（L175-185）：OllamaChat.predict() / ChatOpenAI.invoke(SystemMessage+HumanMessage)
Step 4  来源提取（L188-190）：sources = list(set(doc.metadata["source"] for doc in docs))
        → set 去重，保证同一文档的多个分块只出现一次
```

返回 `{"answer": str, "sources": List[str], "docs": List}`——answer 是 LLM 原文，sources 是去重后的来源列表，由 Agent 层负责最终展示（见 6.6.2 的"来源去重拼接"）。

### 6.5.4 三套 Agent 专用 Prompt 的差异设计

[AGENT_PROMPTS](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L222-L274) 为三种 Agent 定制了回答格式，与默认 [PROMPT_TEMPLATE](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L204-L218) 的"通用知识库问答"定位形成分工：

| Prompt | 角色设定 | 强制格式 | 典型输出 |
|--------|---------|---------|---------|
| `device` | 实验室设备管理员 | 单设备一行：`名称 \| 位置 \| 状态 \| 负责人`；清单用简表 | `GPU-A100-01 \| 2号机房 \| 空闲 \| 张三` |
| `project` | 项目管理助理 | 人员查询一行一项目：`项目名 \| 角色 \| 状态 \| 关键节点`；项目查询含阶段+节点+风险 | 项目进度三段式 |
| `knowledge` | 知识管理员 | 论文：一句话总结+核心方法+关键结论；流程：编号步骤+⚠️注意点；对比：简表 | DPO 论文结构化摘要 |
| 默认 | 通用知识库助手 | ≤150 字、不确定说"未查到" | 通用问答 |

**共同约束**：都要求"禁止开场白、禁止'好的'、禁止末尾总结、引用来源放最后一行"——这是飞书群聊场景的产品化要求：答案要一眼读完、来源可追溯。Prompt 差异即产品差异：**设备答案求"快查"，项目答案求"状态"，知识答案求"结构"**。

---


## 6.6 Multi-Agent 与混合意图（Orchestrator）

### 6.6.1 BaseAgent：检索 + 生成的标准模板

[BaseAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L15-L121) 是三个子 Agent 的抽象基类，子类只覆盖 4 个类属性：

| 子类 | category | agent_type | 检索范围 |
|------|---------|-----------|---------|
| [DeviceAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/device_agent.py#L10-L15) | `"devices"` | `device` | 仅 `devices/` 目录 |
| [ProjectAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/project_agent.py#L9-L14) | `"projects"` | `project` | 仅 `projects/` 目录 |
| [KnowledgeAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/knowledge_agent.py#L9-L14) | `None` | `knowledge` | 全量（knowledge/ + guides/ + 全部） |

**`search_context()`**（[base.py#L35-L69](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L35-L69)）：

```python
if self.category:    # 有分类 → 分类限定检索
    docs = search_by_category(self.vectorstore, self.bm25_data, query, self.category, top_k=top_k)
else:                # KnowledgeAgent → 全量混合检索
    docs = hybrid_search(self.vectorstore, self.bm25_data, query, top_k=top_k)
```

**为什么 category 过滤能保证精准**：用户问"GPU-A100-01 还有空闲卡吗"，若不做分类过滤，混合检索可能命中 `projects/LLM对齐优化项目.md` 里"训练用到 A100"的句子——词面相关但答非所问。DeviceAgent 把检索域锁死在 `devices/`，**从源头杜绝跨域串扰**。这是 Router 粗路由 + Agent 细过滤的两级精度保障。

**`answer()`**（[base.py#L71-L107](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L71-L107)）完整流程：

```
llm 未初始化 → "⚠️ LLM 未初始化，请稍后再试"
search_context() 检索失败 → "⚠️ 检索失败"
无检索结果 → _no_result_response() 兜底
generate_answer(专用 Prompt) → answer + sources
sources 非空且 answer 里没有"来源"字样 → 追加 "📎 来源: {', '.join(sources[:3])}"
```

**来源去重拼接**（L102-103）：`sources` 已经过 `set()` 去重（generator.py L188），再截取前 3 个展示，并检查 `"来源" not in answer`——防止 LLM 已经在正文里写过来源时重复追加。**去重 → 截断 → 防重**三层处理保证来源标注干净。

**`_no_result_response()`**（[base.py#L109-L121](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L109-L121)）：检索零命中时的兜底话术——告知"未找到 + 换关键词建议 + 支持的问题类型"，而不是给用户一句空泛的"不知道"。**把检索失败转化为引导**。

### 6.6.2 Orchestrator.process：混合意图的二次路由

[Orchestrator.process()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L71-L130) 的完整决策树：

```
process(query)
 ├─ 未初始化 → initialize()（加载索引 + LLM + Agent）                    L77-78
 ├─ recognize_intent(query)                                            L81
 ├─ intent == UNKNOWN → _unknown_intent_response(query)（全量混合检索兜底） L84-85
 ├─ 取 primary_agent（AGENT_MAP 映射）                                  L88-90
 ├─ response = primary_agent.answer(query)                              L99
 └─ intent == MIXED 且 secondary_intents 非空 →                          L104-128
     对每个次要意图（排除与主 Agent 相同者）：
       sec_agent.answer(query) → 以 "**📎 {sec_agent.description}**:\n{...}" 拼接
     response += "\n\n---\n\n" + 拼接结果
```

**混合意图处理的设计细节**：

1. **主 Agent 先答，次要 Agent 补答**：主答案永远在最前，次要答案以"📎 描述"标题分隔追加——主次分明，用户先看到核心答案。
2. **排除与主 Agent 重复的次要意图**（L114）：防重复查询。
3. **次要 Agent 异常静默跳过**（L124-125）：`except Exception: pass`——次要查询失败不影响主答案，**降级不阻断**。
4. **UNKNOWN 兜底**（[_unknown_intent_response](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L132-L165)）：索引与 LLM 都可用时，走**不带分类过滤的全量混合检索**（top_k=5）再生成；失败则回退到"引导用户换说法"的文案——三档降级：全量检索 → 引导话术 → 兜底。

**Agent 初始化与依赖注入**（[orchestrator.py#L25-L68](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L25-L68)）：`initialize()` 先加载 embedding → 加载双索引 → 创建 LLM → `_init_agents()` 实例化三个 Agent 并 `set_dependencies()` 注入索引与 LLM（[base.py#L29-L33](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L29-L33)）。任一环节失败（索引缺失/LLM 失败）不中断启动，以"离线模式"继续服务。`get_orchestrator()` 单例（[orchestrator.py#L191-L196](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L191-L196)）保证飞书长连接与 CLI 复用同一份状态。

---


## 6.7 飞书长连接与消息去重

### 6.7.1 WebSocket 长连接：为什么无需公网 IP

传统 IM 机器人常做 HTTP webhook：飞书把事件 POST 到你的公网 URL，你必须暴露公网 IP + 端口映射 + HTTPS 证书，内网环境（实验室）极难满足。

LabQA 用的是 **WebSocket 长连接模式**（[feishu_ws.py#L93-L180](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93-L180)）：

```
飞书服务器 ◄────wss:// 长连接（客户端主动建立）──── 本机进程
     │                                                      
     └── 消息事件 push（服务端沿已建立的连接推送）────────────────► on_message()
```

**原理**：由**客户端主动**向飞书服务器发起 WebSocket 连接（`lark_oapi.ws.Client`，[L163-169](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L163-L169)），连接建立后飞书服务端把事件沿这条**反向通道**推送回来。客户端不需要任何入站端口——出站连接在绝大多数内网（NAT 后）都允许。这就是"无需公网 IP"的底层原因：**通信方向反转了**。

事件注册（[L166-168](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L166-L168)）：

```python
EventDispatcherHandler.builder()
    .register_v2("im.message.receive_v1", on_message)   # 注册消息接收事件
```

`on_message()`（[L124-160](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L124-L160)）的解析链路：取 `chat_id/message_id/msg_type` → 非文本直接忽略 → `json.loads(content)` 取 text → **剥掉 `@机器人` 前缀**（L148，`content.strip().lstrip("@").strip()`，连续多个 @ 也能处理）→ `_handle_message()` → 有响应则 `_send_message()` 回发。

### 6.7.2 两层去重：补偿 SDK 重试

飞书消息投递存在重试机制（断线重连后的补偿推送、服务端偶发重复投递），同一消息可能收到多次。[_is_duplicate()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28-L56) 用两层防线：

```python
# 层1：message_id 集合（L31-38）
if message_id and message_id in _seen_message_ids:
    return True
_seen_message_ids.add(message_id)
if len(_seen_message_ids) > 10000:   # 内存保护：超限清空（宁可误收，不可 OOM）
    _seen_message_ids.clear()

# 层2：内容哈希，30 秒窗口，chat 维度（L41-55）
content_hash = hashlib.md5(content.encode()).hexdigest()
# chat_id → {hash: timestamp}；过期(>30s)哈希定期清理；窗口内同哈希 → 重复
```

| 层 | 键 | 窗口 | 防什么 |
|----|----|------|--------|
| 层1 | `message_id` | 永久（集合，上限 10000 条） | 同一事件的重试/补偿推送——message_id 唯一 |
| 层2 | `md5(content)`，按 `chat_id` 隔离 | 30 秒 | 不同 message_id 但内容相同（如断线重连后 SDK 以新 id 重发；或用户快速重复发送） |

**为什么层2要 chat-scoped**：不同群聊里发相同内容（"在吗"）是正常行为，不能全局去重；`_chat_content_hashes[chat_id]` 把哈希表按会话隔离，只在同一群聊内 30 秒窗口生效。**为什么 30 秒**：飞书重试窗口在秒级~分钟级，30 秒覆盖绝大多数重试，又不会误杀 1 分钟后用户真实的重复提问。

**内存保护**：message_id 集合超 10000 清空（L37-38，清空后可能短暂误收，但避免了无界增长）；内容哈希表每次调用顺手清理过期条目（L47-50）。两个数据结构都是惰性清理，零定时器开销。

### 6.7.3 发送与 19900 截断

[_send_message()](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L78-L90)：

```python
# 飞书文本消息最长 20000 字符（L83）
truncated = content[:19900] + ("..." if len(content) > 19900 else "")
```

**19900 = 20000 - 100 余量**：飞书限制单条文本消息 20000 字符，留 100 字符缓冲防止边界波动，超长时截断并加 `...` 提示。RAG 答案通常几百字，此截断主要防御 LLM 抽风输出超长文本。

**完整服务启动**（`start()`，L93-180）：校验 `FEISHU_APP_ID/SECRET` → `get_orchestrator().initialize()`（复用单例）→ 构建发送客户端 + WS 客户端 → `ws_client.start()` 阻塞运行（Ctrl-C 优雅退出）。一次启动 = 长连接 + RAG 全链路，单进程部署。

---


## 6.8 可运行演示 1：RRF 融合算法（纯函数，零依赖）

与 [retriever.py#L106-L142](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106-L142) 的实现一致的最小复刻，展示"跨路共识 > 单路高排名"的核心行为。可直接运行，无需任何第三方库：


In [ ]:
def reciprocal_rank_fusion(results_list, k=60):
    """
    RRF（Reciprocal Rank Fusion）最小演示实现
    score(doc) = Σ 1/(k + rank_r(doc))，k 为平滑常数
    """
    doc_scores = {}
    for retriever_results in results_list:
        for rank, item in enumerate(retriever_results, start=1):
            doc_key = item["doc"]                       # 源码中以 page_content 为唯一标识
            if doc_key not in doc_scores:
                doc_scores[doc_key] = {"doc": doc_key, "rrf_score": 0.0}
            doc_scores[doc_key]["rrf_score"] += 1.0 / (k + rank)
    return sorted(doc_scores.values(), key=lambda x: x["rrf_score"], reverse=True)


# ===== 模拟两路检索结果 =====
# FAISS 稠密检索 Top-3（语义相似排序）
faiss_results = [
    {"doc": "devices/GPU-A100-01.md", "score": 0.92},
    {"doc": "devices/GPU-H100-01.md", "score": 0.78},
    {"doc": "devices/设备总览.md",     "score": 0.65},
]
# BM25 稀疏检索 Top-2（关键词精确匹配："GPU-A100-01" 直接命中倒排索引）
bm25_results = [
    {"doc": "devices/GPU-A100-01.md", "score": 12.5},
    {"doc": "devices/设备总览.md",     "score": 3.1},
]

fused = reciprocal_rank_fusion([faiss_results, bm25_results], k=60)

print(f"{'文档':<24} {'FAISS rank':>10} {'BM25 rank':>10} {'RRF 分数':>10}")
print("-" * 58)
for item in fused:
    faiss_rank = next((i + 1 for i, r in enumerate(faiss_results) if r["doc"] == item["doc"]), "-")
    bm25_rank  = next((i + 1 for i, r in enumerate(bm25_results) if r["doc"] == item["doc"]), "-")
    print(f"{item['doc']:<24} {str(faiss_rank):>10} {str(bm25_rank):>10} {item['rrf_score']:.5f}")

print("\n关键结论：")
print("  1. GPU-A100-01.md 双路都排第1 → RRF ≈ 1/61 + 1/61 = 0.03279（最高）")
print("  2. 设备总览.md 双路都命中（第3/第2）→ ≈ 1/63 + 1/62 = 0.03200（紧随）")
print("  3. GPU-H100-01.md 仅 FAISS 命中 → 1/62 = 0.01613（被腰斩）")
print("  → 融合结果由『跨路共识』主导，而非单路最高分——这是 RRF 鲁棒性的来源")


**实验**：把 `k` 改成 1 再运行，观察排名第 1 的文档权重被放大 3 倍（1/2 vs 1/6）——体会 k=60 为何让两路更"均衡"。

---


## 6.9 可运行演示 2：意图识别打分（需项目环境）

直接调用项目源码 `labqa.router`，复现 6.2 的完整打分与混合意图判定。**依赖**：`python-dotenv`（router → config 导入链）+ 项目 `src/` 路径。需在 Lapmind 仓库环境运行：


In [ ]:
import sys
from pathlib import Path

# 项目根目录（按你的实际克隆路径调整）
ROOT = Path("/Users/wuhang/Desktop/简历项目经历复盘总结/Lapmind")
sys.path.insert(0, str(ROOT / "src"))

# 依赖：pip install python-dotenv（router.py 导入链需要 config.py）
from labqa.router import recognize_intent, extract_keywords

cases = [
    "GPU-A100-01 还有空闲卡吗？",          # 纯设备
    "LLM对齐项目进展如何？",                # 纯项目
    "DPO论文的核心方法是什么？",            # 纯知识
    "GPU服务器和LLM对齐项目哪个更紧急？",   # 混合意图
    "今天天气怎么样？",                     # 未知意图
]

for q in cases:
    keywords = extract_keywords(q)
    decision = recognize_intent(q)
    print(f"Q: {q}")
    print(f"   关键词: {keywords}")
    print(f"   意图={decision.intent.value:<8} "
          f"置信度={decision.confidence:.2f} "
          f"目标Agent={decision.target_agent}")
    print(f"   匹配词={decision.matched_keywords}")
    if decision.secondary_intents:
        print(f"   次要意图={[s.value for s in decision.secondary_intents]}  "
              f"→ 触发 MIXED 二次路由")
    print()

# 预期观察：
#   1. "GPU-A100-01" → extract_keywords 用正则补捉出 "a100"（词块里的型号实体）
#   2. 前 3 例分别高分路由 device/project/knowledge，置信度 = min(分数/5, 1)
#   3. 第 4 例 device 与 project 同时高分 → intent=mixed，Orchestrator 会双 Agent 回答
#   4. 第 5 例三路全 0 → UNKNOWN，走全量混合检索兜底（orchestrator._unknown_intent_response）


> **源码对照**：`extract_keywords` 的标点替换 + 单字过滤 + 型号正则见 [router.py#L86-L129](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L86-L129)；长短词双通道加权见 [_score_intent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L132-L147)；三路打分、50% 混合阈值、`min(分数/5, 1)` 置信度见 [recognize_intent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150-L207)。


# 第七部分：已知问题与调优经验

> **说明**：本部分基于对代码仓库的全量扫描 + git 历史分析（4 个 commit，其中 `757df23` 为 v1→v2 重构）+ `AGENTS.md` 的 Key Gotchas 章节整理，所有问题均已通过源码逐一核验。标注 `来源：AGENTS.md` 的条目以文档为准并注明与现状的差异；标注 `来源：源码核验` 的条目为本说明书独立确认。**特别注意**：`AGENTS.md` 多个 Gotcha 描述的是 v1（`labqa/` 包）行为，v2（`src/labqa/`）重构后部分已不再成立——这本身就是一个重要的文档债（见 7.1 P-04）。

---

## 7.1 架构层已知问题表

| 编号 | 问题 | 影响 | 位置 / 建议 |
|:---:|------|------|------------|
| **P-01** | **双执行面代码重复维护成本**：同一套知识库（`.opencode/context/`）被两套独立执行面消费——Python 运行时（`src/labqa/`，关键词路由 + 混合检索）与 OpenCode Agent 定义（`.opencode/agent/` 的 lab-orchestrator + 3 个 subagents，直接在会话内读 context 文件）。两边的"意图识别规则、检索逻辑、回答风格"各自实现，**知识库变更影响两边，但逻辑变更需要分别维护** | 新增/修改一个功能往往要改两处；两边的行为会逐渐漂移（例如 Python 侧已升级为 FAISS+BM25 混合检索，Agent 侧仍是关键词/直接引用文件）；新人理解成本高 | [AGENTS.md](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L5)（Architecture 双执行面定义）。**建议**：收敛为单一事实源——要么 Agent 定义调用 Python 侧 API，要么用生成脚本从同一份配置产出两边规则；至少先在 [docs/ARCHITECTURE.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/ARCHITECTURE.md) 固化两边的职责边界（ADR-001/002 已有雏形） |
| **P-02** | **legacy 独立脚本绕过核心包**：`scripts/feishu-webhook.py` 是 Flask 版飞书 HTTP webhook，**自带 tenant token 获取、消息发送、关键词匹配逻辑，完全不经过 `labqa/` orchestrator**，与主系统形成两套并行实现 | 该脚本与主系统的检索/路由逻辑脱节（主系统已是混合检索 RAG，它仍是自研关键词匹配）；团队若误用会导致"同样的仓库、两种回答质量"；依赖 `flask requests` 也未纳入任何依赖清单 | [scripts/feishu-webhook.py](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/feishu-webhook.py#L1)、[AGENTS.md](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L50)（明确标注 "Do NOT use for new work"）。**建议**：删除或归档到 `scripts/legacy/`，并在 README 明确"飞书接入唯一入口 = `python3 main.py feishu`" |
| **P-03** | **`main.py webhook` 别名歧义**：命令名 `webhook` 实际启动的是 WebSocket 长连接客户端（`mode_feishu()`），而非 HTTP webhook | 名字与行为不符：使用者以为在起 HTTP 服务，实际是 WS 长连接；排障时按"webhook 模式"查日志会走弯路；AGENTS.md 也承认该别名仅为向后兼容保留（v1 的 HTTP webhook 路径已被替换） | [main.py](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L135)（`elif cmd in ("feishu", "webhook")`）、[AGENTS.md](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L32)。**建议**：v3 中移除别名，或改为打印提示"webhook 已废弃，请用 feishu"后退出 |
| **P-04** | **AGENTS.md 与 v2 现状不一致**：多处描述停留在 v1——①"Package Structure" 仍列 `labqa/` 下的 `context_store.py` / `llm.py` / `webhook.py`（v2 已删除，实际是 `src/labqa/` 下的 `retriever.py` / `generator.py` / `feishu_ws.py`）；②"No test framework" 声称无 pytest，但 v2 已有 [tests/test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L1)（7 个 Test 类、约 18 个用例），且 pyproject.toml 已配置 `[tool.pytest.ini_options]`；③"CLI offline mode" 的 `offline-mode` 哨兵在 v2 源码中已不存在（见 7.2-C） | 新成员按 AGENTS.md 学习会找不到文件、误以为无测试可跳过、误以为有离线哨兵逻辑；文档与代码互斥会消耗信任 | [AGENTS.md](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L54)（Package Structure）、[AGENTS.md](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L73)（No test framework，与现状不符）。**建议**：按 v2 结构重写 AGENTS.md（见 8.1-D1） |
| **P-05** | **frontmatter 手写解析限制**：`ingest.py` 对 YAML frontmatter 用 `split("---", 2)` + 逐行 `key: value` 手动解析，不是完整 YAML parser | 嵌套对象、列表、带冒号的值会解析错误或被截断（例如 `tags: [a, b]` 会被存成 `[a, b]` 字符串）；一旦知识库文档的 frontmatter 复杂度上升，元数据静默失真，检索时按 `type` 过滤可能失效；AGENTS.md 已警告"Don't put complex YAML" | [src/labqa/ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L51)（手写解析循环 L55-58）、[AGENTS.md](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L74)。**建议**：升级为 `PyYAML` 安全加载（`yaml.safe_load`），并加 frontmatter 解析单测 |
| **P-06** | **无多轮对话**：每次问答完全独立，Orchestrator 不维护 chat history，无法追问（"那 GPU-A100-02 呢？"） | 实验室用户必须把每轮问题写完整（"GPU-A100-02 还有空闲卡吗？"），对话体验生硬；无法支持"先问设备再对比"的连续查询流 | [src/labqa/orchestrator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L71)（process 无历史参数）、[README.md](https://github.com/BLYHFL/labqa-rag/blob/main/README.md#L127)（官方列为扩展方向） |
| **P-07** | **无 Re-Ranker 精排**：RRF 融合后直接取 Top-K 送入 LLM，检索结果与查询的相关性没有二次排序 | 混合检索"召回"足够但"精度"依赖 RRF 排名质量；对易混淆查询（如"测试车"与"测试"）可能出现次优片段进入上下文，稀释生成质量；无法利用 LLM 做精排的潜力 | [src/labqa/retriever.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L218)（RRF 后直接截断）、[README.md](https://github.com/BLYHFL/labqa-rag/blob/main/README.md#L129)（Re-Ranking 列为扩展方向） |
| **P-08** | **关键词路由硬编码**：意图识别完全依赖 `DEVICE_KEYWORDS` / `PROJECT_KEYWORDS` / `KNOWLEDGE_KEYWORDS` 三张手写词表 + 每类 Agent 的 `extract_search_keywords()`，**新增意图类型需要同时改 router.py 和 agent 两处**；词表间还出现了关键词重叠（dpo/rlhf 同时出现在 project 与 knowledge，见 7.2-B） | 扩展成本高、易遗漏（只改一处导致行为不一致）；重叠词引发潜在误路由；词表无法覆盖未预见的说法（口语变体），属于"白名单式"脆弱设计 | [src/labqa/router.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L35)（三张词表）、[AGENTS.md](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L77)。**建议**：短期先做词表去重与单测覆盖；中期按 README 规划演进为 LLM 路由（router.py 注释已预留，见 [L154-157](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L154)） |


## 7.2 代码细节问题

### 7.2-A v1 → v2 重构前后对比（来源：git log commit `757df23` + 源码核验）

v1（`labqa/` 包，Initial commit `07bb9b1`）是纯关键词匹配方案，v2（`src/labqa/`，commit `757df23`）全面重构为 FAISS + BM25 + RRF 混合检索 RAG。重构动机即 v1 的**召回短板**：

| 维度 | v1（关键词匹配时代） | v2（混合检索 RAG 时代） | 重构收益 |
|------|---------------------|------------------------|---------|
| 检索机制 | `context_store.py` 关键词打分搜索（AGENTS.md 描述为 "keyword search + scoring"，该文件 v2 已删除） | `ingest.py` 建索引 + `retriever.py` 混合检索：FAISS 稠密 + BM25 稀疏 + RRF 融合 | 语义相近但无关键词重合的查询（如"算力不够了找谁"）也能召回；专有名词（A100/H800）靠 BM25 兜住 |
| 索引形态 | 无持久化索引，每次搜索实时扫描文件 | `faiss_index/index.faiss` + `bm25_index.pkl` 持久化，已存在则加载不重建（[ingest.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L125)） | 启动秒级加载，查询延迟大幅下降 |
| LLM 客户端 | `llm.py` 自研 OpenAI-compatible httpx 客户端（60s 超时） | `generator.py` 双轨：OllamaChat 本地适配器 + ChatOpenAI 云端（[generator.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L48)） | 本地免费运行 + 云端高质量双轨切换 |
| 测试 | 无 pytest（AGENTS.md 原话 "No test framework"） | `tests/test_main.py` 320 行、7 个 Test 类（Config/Ingest/Router/RRF/Prompts/Generator/Agents） | 路由与 RRF 融合算法有回归保障 |
| 目录 | `labqa/`（顶层包） | `src/labqa/`（src 布局，pyproject `packages = ["src/labqa"]`） | 符合 Python 工程惯例，为 `pip install .` 打底 |

> ⚠️ **迁移遗留**：v1 文件（`context_store.py` / `llm.py` / `webhook.py`）已从 v2 移除，但 AGENTS.md 的 Package Structure 与 Two Feishu Implementations 表仍引用它们（见 7.1 P-04）——文档是重构后唯一"残留的 v1 文件"。

### 7.2-B dpo/rlhf 关键词重复 → 潜在误路由分析（来源：源码核验 + 打分演算）

`dpo`、`rlhf` 同时出现在 **PROJECT_KEYWORDS（[router.py L53](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L53)："llm", "rlhf", "dpo", "对齐"…）** 与 **KNOWLEDGE_KEYWORDS（[router.py L66](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L66)："dpo", "rlhf", "transformer", "attention"…）** 两套词表中。原因是：DPO/RLHF 既是实验室的**研究项目主题**（projects/LLM对齐优化项目），也是**论文笔记的检索词**（knowledge/论文笔记/DPO论文笔记.md）。词表重叠在直觉上"两边都能命中"，但 `_score_intent` 的打分机制会让结果跑偏：

| 查询示例 | DEVICE 分 | PROJECT 分 | KNOWLEDGE 分 | 路由结果（演算） |
|---------|:---:|:---:|:---:|------|
| "DPO论文的核心方法是什么？"（测试用例，期望 KNOWLEDGE） | 0 | 1（"dpo" 命中） | **4**（"论文"+1、"核心方法"+2、"dpo"+1） | ✅ KNOWLEDGE——靠"论文/核心方法"补分压过重复词 |
| "DPO是什么？" / "RLHF的原理？"（**无论文/方法等知识域词**） | 0 | **1**（"dpo"） | **1**（"dpo"） | ❌ **平票 → MIXED，且 primary=PROJECT**（`next()` 按 dict 顺序取首个最大值：DEVICE→PROJECT→KNOWLEDGE，见 [router.py L175](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L175)）→ 路由到 **Project-Agent**，检索 `projects/` 目录，**不会命中 knowledge/DPO论文笔记.md** |
| "dpo 和 kto 哪个更好？" | 0 | 1（"dpo"） | 3（"dpo"+1、"kto"+1、"哪个更好"+1…） | ✅ KNOWLEDGE——"对比/哪个更好"类词在 KNOWLEDGE 词表较全 |

**根因**：`_score_intent`（[router.py L132-147](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L132)）对每个意图独立计分，短查询（1-2 个词）只命中重叠词时必然平票，而平票裁决规则（dict 顺序优先）没有领域知识。同类风险词还有："测试"（PROJECT L48 vs 隐含设备语义，测试用例已用注释记录该现象，见 [test_main.py L99-100](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L99)）、"自动驾驶"（PROJECT L53 vs DEVICE 的"测试车"）。

**缓解措施（v2 已实现）**：测试用例 `test_recognize_device_intent` 明确断言"测试车谁在用"→ MIXED 且 target=Device-Agent（[test_main.py L92-107](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L92)），把平票场景固化为回归保护；但 **"DPO 是什么" 这类短查询的误路由没有测试覆盖**，建议补用例 + 词表去重（把 dpo/rlhf 从 PROJECT 或 KNOWLEDGE 中移除，靠项目名"LLM对齐优化项目"中的"对齐"承担 project 侧命中）。

### 7.2-C CLI 离线模式哨兵 hack（来源：AGENTS.md L75 记录 vs 源码核验）

AGENTS.md 记录了 v1 的离线兜底设计：无 API Key 时 CLI 进入"仅关键词搜索、不调用 LLM"的离线模式，实现方式是把 `os.environ["LABQA_LLM_API_KEY"] = "offline-mode"` 作为哨兵值。**经 grep 全库核验：v2 源码中已不存在 `offline-mode` 哨兵**——v2 用更干净的方式解决了同一问题：

- `LLM_MODE` 默认 `ollama`（[config.py L25](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25)），本地模式**根本不需要 API Key**（Ollama 本地服务）；
- `validate()` 只在 `LLM_MODE == "cloud"` 时强制要求 Key（[config.py L64-65](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L64)），ollama 模式下 `mode_ask` 直接放行（[main.py L70-73](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L70)）。

**教训**：用环境变量值当"哨兵"是典型 hack——它污染了配置语义（"offline-mode" 不是合法 Key，任何下游读到它都得特判）。v2 用"模式枚举 + 校验规则"替代哨兵是对的，但 AGENTS.md 未同步更新，读者仍以为存在哨兵逻辑（文档债，见 8.1-D1）。

### 7.2-D .env 加载双实现（来源：源码核验）

项目存在**两套 .env 加载逻辑**，行为细节还不一致：

| 加载器 | 位置 | 行为 | 差异点 |
|--------|------|------|--------|
| `config.py` 模块级 `load_dotenv()` | [src/labqa/config.py L8-11](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L8) | 无条件覆盖前加载 .env（python-dotenv 默认不覆盖已存在的环境变量） | 所有 `from labqa.config import *` 的模块都会触发 |
| `main.py` 的 try-dotenv-fallback 手写解析 | [main.py L26-41](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L26) | 优先 `load_dotenv()`，ImportError 时手写解析 `key=value` 行（跳过 `#` 注释、剥引号、**不覆盖已存在的环境变量**） | 手写解析不支持值内 `#`、多行、`export` 前缀等 dotenv 语法 |

AGENTS.md 声称 "The .env loader in main.py is hand-rolled (not python-dotenv)"（[AGENTS.md L36](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L36)）——实际是 **dotenv 优先、手写兜底**，文档描述又不准确。且手写解析器只有 main.py 入口才生效；如果直接 `python -m labqa.cli` 启动，则只有 config.py 的 dotenv 在起作用。**风险**：两条路径解析结果理论上一致（都不覆盖 shell 环境变量），但维护双实现是隐患。

### 7.2-E requirements.txt 与 pyproject.toml 依赖不一致（来源：源码核验）

| 清单 | 内容 | 问题 |
|------|------|------|
| [requirements.txt](https://github.com/BLYHFL/labqa-rag/blob/main/requirements.txt#L1) | 仅 `httpx` + `lark-oapi`；文档转换三件套（python-docx/pptx/openpyxl）被**注释掉**（L10-13） | 按此安装后 v2 运行时**必然缺 langchain / faiss-cpu / rank-bm25 / ollama / python-dotenv**，`main.py ingest` 直接 ImportError |
| [pyproject.toml L7-30](https://github.com/BLYHFL/labqa-rag/blob/main/pyproject.toml#L7) | 完整声明 14 个运行时依赖（langchain 全家桶、faiss-cpu、rank-bm25、ollama、httpx、lark-oapi、python-docx、python-pptx、openpyxl、python-dotenv、tiktoken）+ dev extras（pytest、pytest-cov） | ✅ 正确的单一事实源，README 快速开始已正确使用 `uv pip install -e ".[dev]"`（[README L32-34](https://github.com/BLYHFL/labqa-rag/blob/main/README.md#L32)）；但 **AGENTS.md 命令区（[AGENTS.md L28-29](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L28)）仍引导 `pip install httpx lark-oapi` 的旧方式**，照做必缺依赖 |

**结论**：requirements.txt 是 v1 遗留，v2 应统一走 pyproject（`pip install -e ".[dev]"`，README 已是正确姿势）；AGENTS.md 的安装命令与 requirements.txt 需要同步收敛（见 8.1-D6）。

### 7.2-F 其他代码细节问题速览（来源：源码核验）

| # | 位置 | 问题 | 说明 |
|:---:|------|------|------|
| 1 | [KnowledgeAgent category=None](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/knowledge_agent.py#L9) | AGENTS.md L71 的 Gotcha"KnowledgeAgent 检索两个目录（两次 get_context_text 调用）"描述的是 v1 行为；v2 中 `INTENT_CATEGORY_MAP` 对 KNOWLEDGE 置 `None`（[router.py L82](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L82)），`hybrid_search` 不做 category 过滤即全库检索 | 文档过时；v2 实现反而更简单，但"全库检索"意味着 knowledge/ 与 guides/ 一起命中，**没有"只查 knowledge/"的细粒度控制** |
| 2 | [feishu_ws.py L83](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L83) | 发送消息硬截断到 19900 字符 | 飞书消息体上限防护是对的，但截断点硬编码，超长回答会被切断而无提示 |
| 3 | [main.py L24](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L24) / [cli.py L11](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L11) | `sys.path.insert(0, ...)` 手工注入 src/ | 运行期 hack；若 `pip install -e .` 已装包则冗余；两处注入路径不一致（main.py 注入根/src，cli.py 注入 parent.parent） |
| 4 | [test_main.py L14](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L14) | 测试同样靠 `sys.path.insert` 注入 src | pyproject 已配 `pythonpath = ["src"]`（[pyproject.toml L50](https://github.com/BLYHFL/labqa-rag/blob/main/pyproject.toml#L50)），测试内的注入是冗余的（可能是为了兼容裸 pytest 运行） |
| 5 | [ingest.py L37-40](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L37) | `navigation.md` 靠文件名硬编码排除 | 若知识库新增其他"索引类"文档（如 overview.md），会静默进入索引污染检索（AGENTS.md L78 记录该设计） |
| 6 | [router.py L116](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L116) | `extract_keywords` 只替换 `，？。` 三种中文标点，`！`、`：`、`；` 未处理 | "GPU服务器在哪！"会被切成一个长词块，模型号提取正则 `[a-zA-Z]\d+` 是唯一兜底；英文句点 `A100.` 会残留 |


## 7.3 调优经验与设计权衡表

> 本表回答"为什么这么做"——每个参数/设计决策背后的权衡，全部可溯源到源码。来源：源码核验 + AGENTS.md + git log。

| 决策 / 参数 | 为什么这么定（权衡分析） | 源码位置 | 来源 |
|------------|------------------------|---------|------|
| **RRF 融合用 k=60** | RRF 公式 `score = 1/(k + rank)`：k 越大，两路排名的"名次差异"被稀释，**两路权重越均衡、对单路 Top 名次不敏感（稳健）**；k 越小，高排名文档主导。项目里 FAISS（余弦相似度）与 BM25（词频分）**量纲不同、无法直接相加**，RRF 用排名而非分数融合，天然规避归一化问题；k=60 是业界常用经验值（论文/教程默认），docstring 明确写"越大两路越均衡" | [retriever.py L106-142](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106)（k 默认取自 [config.py L44](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L44)，docstring L120 注明"经验值"） | 源码核验 |
| **CHUNK_SIZE=500 / CHUNK_OVERLAP=50** | 实验室文档是"条目式"短文（设备卡、项目进度、新人指南），500 字符≈250-330 汉字，约一个自然段，语义完整度与检索粒度平衡；50 字符重叠（10%）保证跨块语义不断裂。`RecursiveCharacterTextSplitter` 按 `\n\n → \n → 。！？； → 空格` 的中文感知分隔链切分（[ingest.py L95-99](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L95)），避免把一句话劈成两半。若块过大，RRF 后 Top-5 会浪费上下文窗口；过小则语义碎片化 | [config.py L45-46](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L45)、[ingest.py L86-110](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L86) | 源码核验 |
| **BM25 无结果时降级为纯向量检索** | 中文分词（jieba）对 OOV 词（英文缩写、型号、新名词）可能零命中；若稀疏路空结果仍进 RRF，会拖累整体召回。降级策略：`sparse_results` 为空 → 直接返回 `dense_results[:top_k]`（[retriever.py L211-216](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L211)），保证"至少有一路可用"。代价：该查询丢失关键词精确匹配能力，但语义路能兜底 | [retriever.py L211-216](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L211) | 源码核验 |
| **混合意图阈值 50%** | 次意图分数 ≥ 主意图 × 50% → 判定 MIXED（[router.py L178-181](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L178)）。权衡：阈值过高 → 漏判混合查询（只按主意图走，丢失次意图信息）；阈值过低 → 大量查询被判 MIXED，所有 Agent 都跑一遍检索，浪费算力。50% 是"分母=主意图分"的相对阈值，自动适配不同量级分数；测试用例"测试车谁在用"（DEVICE=2, PROJECT=1 → 1 ≥ 2×0.5 成立 → MIXED 且主 Device-Agent）固化了该行为 | [router.py L178-181](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L178)、[test_main.py L99-100](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L99) | 源码核验 |
| **飞书用 WS 长连接而非 HTTP webhook** | 核心原因：**实验室/内网环境无公网 IP**，webhook 需要飞书回调到公网地址（还要配 SSL、反向代理）；WS 长连接由客户端主动连飞书，零公网暴露（[AGENTS.md L21-22](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L21)）。飞书 SDK（lark-oapi）原生支持 WS 事件订阅，断线自动重连。代价：进程必须常驻（CLI 里是 `while True` 事件循环），适合 7×24 跑在实验室服务器上 | [feishu_ws.py L93-180](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93)、[AGENTS.md L40](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L40) | AGENTS.md + 源码核验 |
| **默认 Ollama 本地模式** | `LLM_MODE` 默认 `"ollama"`（[config.py L25](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25)）：①实验室场景（~20 人）成本敏感，本地推理零 API 费用；②**零配置开箱即用**——无需 API Key，`validate()` 只在 cloud 模式强制 Key（[config.py L64-65](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L64)）；③数据不出内网。代价：`deepseek-r1:1.5b` 质量明显低于云端 `deepseek-chat`，所以保留双轨，质量优先场景切 cloud。**这同时是"三级 API Key 回退"（LABQA_LLM_API_KEY → DEEPSEEK_API_KEY → OPENAI_API_KEY，[config.py L32-35](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32)）存在的原因**：兼容老项目环境变量，降低迁移成本 | [config.py L25](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25)、[config.py L32-35](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32)、[generator.py L99-118](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99) | 源码核验 |
| **Router 用关键词而非 LLM 路由** | 明确的设计取舍（docstring 原文："保留关键词匹配作为轻量级前置路由（快速、零成本）"，[router.py L5-6](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L5)）：①每次问答都过 LLM 路由，延迟 + 成本（尤其 Ollama 小模型）不可接受；②关键词打分是确定性逻辑，可单测、可调试（DEBUG 模式打印三路分数，[router.py L193-199](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L193)）；③路由错误的最坏情况只是"走错 Agent 目录"，检索兜底 + LLM 生成仍能回答。代价就是 7.1 P-08 的硬编码问题，代码注释已留升级口："未来可扩展为 LLM 路由（更准确但更慢）"（[router.py L156](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L156)） | [router.py L1-7](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L1)、[router.py L150-157](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150) | 源码核验 |
| **Agent category 过滤保证精准度** | 路由定意图 → Agent 定检索范围：DeviceAgent 只搜 `devices/`（category="devices"）、ProjectAgent 只搜 `projects/`、KnowledgeAgent 全库（category=None）。**检索域收窄 = 语义检索不容易跨域污染**（"GPU 项目"不会在设备库里搜到无关内容），同时 RRF 融合对 category 过滤生效（dense/sparse 两路都带 filter）。代价：跨域查询（"项目用的 GPU 还有空吗"）要靠 MIXED 意图 + 主 Agent 检索，次意图信息不参与检索 | [agents/device_agent.py L10-15](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/device_agent.py#L10)、[router.py L79-83](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L79)、[retriever.py L175-229](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175) | 源码核验 |
| **飞书消息两层去重** | 飞书事件推送存在 SDK 重试，同一消息可能重复触发。方案：①按 `message_id` 集合去重（精确）；②30 秒窗口内按内容哈希去重（chat-scoped，补偿 message_id 缺失/变化的场景）。（[feishu_ws.py L28-56](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28)）两层互补：message_id 防"同一事件重推"，内容哈希防"同内容不同 id"。代价：哈希窗口内相同内容的两条独立消息会被误杀（30 秒内几乎不会发生，可接受） | [feishu_ws.py L28-56](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28)、[AGENTS.md L76](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L76) | AGENTS.md + 源码核验 |


# 第八部分：待优化点与未来规划

## 8.1 短期工程债（建议 1-2 个迭代内清理）

| # | 工程债 | 现状与风险 | 建议动作 | 优先级 |
|:---:|--------|-----------|---------|:---:|
| D1 | **AGENTS.md 全面对齐 v2** | 多处描述停留在 v1：包结构列了已删除的 `context_store.py`/`llm.py`/`webhook.py`（[AGENTS.md L54-67](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L54)）；声称无测试框架但已有 18 个 pytest 用例（[AGENTS.md L73](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L73)）；offline 哨兵已不存在（[AGENTS.md L75](https://github.com/BLYHFL/labqa-rag/blob/main/AGENTS.md#L75)）。新成员按文档学习会直接踩坑 | 按 v2 重写 Package Structure（src/labqa/ + tests/）；补测试运行方式 `pytest tests/ -v`；更新 Two Feishu Implementations 表（webhook.py 已删）；把"双执行面"章节写成维护指南 | **P0** |
| D2 | **接入 CI** | 有 18 个测试用例但无 CI 卡点，回归全靠自觉；pyproject 已配 `[tool.pytest.ini_options]`（[pyproject.toml L48-50](https://github.com/BLYHFL/labqa-rag/blob/main/pyproject.toml#L48)）和 dev extras（[L32-36](https://github.com/BLYHFL/labqa-rag/blob/main/pyproject.toml#L32)），接入成本极低 | GitHub Actions 最小 workflow：`pip install -e ".[dev]"` → `pytest tests/ -v`；后续加 lint（ruff）与 frontmatter 解析用例 | **P0** |
| D3 | **convert.sh PDF 转换占位完善** | `convert_pdf` 是占位实现（[convert.sh L141](https://github.com/BLYHFL/labqa-rag/blob/main/scripts/convert.sh#L141)），PDF 无法入库；Word/PPT/Excel 转换三件套虽已在 pyproject 声明（[pyproject.toml L24-26](https://github.com/BLYHFL/labqa-rag/blob/main/pyproject.toml#L24)），但 requirements.txt 将其注释（[L10-13](https://github.com/BLYHFL/labqa-rag/blob/main/requirements.txt#L10)）、AGENTS.md 安装命令也未包含 | 接入 `pypdf` / `pdfplumber` 完成 PDF→Markdown；删除 requirements.txt 中的注释残留，安装指引统一走 pyproject | P1 |
| D4 | **frontmatter 解析升级为 YAML** | 手写 `key: value` 解析（[ingest.py L51-59](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L51)），嵌套 YAML/列表会静默失真（见 7.1 P-05） | 引入 `PyYAML`，`yaml.safe_load` 解析；metadata 中 tags 存为 list；补 frontmatter 边界单测（空 frontmatter、嵌套、带冒号的值） | P1 |
| D5 | **统一 .env 加载实现** | main.py 手写兜底 loader（[main.py L26-41](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L26)）+ config.py dotenv（[config.py L11](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L11)）双实现，手写解析器不支持完整 dotenv 语法 | 删掉 main.py 的手写解析分支，依赖 python-dotenv（已在 pyproject 声明）；若担心 ImportError，把 python-dotenv 移入运行时必装依赖即可 | P1 |
| D6 | **依赖清单收敛** | requirements.txt 与 pyproject.toml 严重不一致（见 7.2-E），`pip install -r requirements.txt` 装不出可运行系统；AGENTS.md 安装命令过时 | 删除或精简 requirements.txt（仅留注释指向 pyproject）；AGENTS.md 安装命令改为 `pip install -e ".[dev]"`（README 已用 uv 安装，[README L32-34](https://github.com/BLYHFL/labqa-rag/blob/main/README.md#L32)，无需改动） | P1 |
| D7 | **legacy 清理** | `scripts/feishu-webhook.py` 自带关键词匹配绕过主系统（见 7.1 P-02）；`main.py webhook` 别名名不副实（见 7.1 P-03） | 归档 legacy 脚本到 `scripts/legacy/`；webhook 别名改为打印废弃提示；统一"飞书唯一入口 = `main.py feishu`" | P2 |
| D8 | **词表去重与短查询误路由修复** | dpo/rlhf 等词在 PROJECT 与 KNOWLEDGE 双词表重复，短查询平票时按 dict 顺序误路由（见 7.2-B 演算） | ① 移除 KNOWLEDGE_KEYWORDS 中的 dpo/rlhf（保留论文域专属词），project 侧靠"对齐/项目"词命中；② 平票裁决改为"置信度相等时按 KNOWLEDGE > PROJECT > DEVICE 的领域常识顺序"或引入加权；③ 补 "DPO是什么"、"RLHF 是什么" 的 KNOWLEDGE 断言用例 | P1 |

## 8.2 README 扩展方向落地表（来源：README.md L124-130 + 工程化要点）

README 官方列出的 5 个扩展方向（[README.md L124-130](https://github.com/BLYHFL/labqa-rag/blob/main/README.md#L124)），结合现有架构给出落地要点与优先级：

| 方向 | 说明 | 落地要点（结合现有代码） | 优先级 |
|------|------|------------------------|:---:|
| **接入更多文档源（PDF/Word/Notion/数据库）** | 目前知识库仅 `.opencode/context/` Markdown；convert.sh 已有 Word/PPT/Excel→Markdown 骨架，PDF 是占位 | ① PDF：补 `convert_pdf`（pypdf/pdfplumber）；② Notion/数据库：走导出 Markdown 再入库的"间接路线"最省事（复用 ingest.py 全链路）；③ 每个文档源封装为 `SourceAdapter`，输出统一 Document 结构 | P1 |
| **多轮对话（聊天历史 + 追问）** | 当前每次问答独立（7.1 P-06） | ① Orchestrator `process` 增加 `history: List[Message]` 参数；② 飞书场景按 `chat_id` 维护会话窗口（复用 feishu_ws 的消息去重结构）；③ 追问改写（"那 02 呢？" → 补全实体）可先用规则（替换上轮设备名），再演进 LLM 改写；④ 注意上下文窗口预算，只保留最近 N 轮 | P0 |
| **Web UI（Streamlit / Gradio）** | 实验室成员浏览器直接问答 | Streamlit 最小实现：`st.chat_message` 流式展示 + 调 `get_orchestrator().process()`；复用 config 校验与索引加载；**建议先做 Streamlit（与 Python 生态同构），Gradio 适合快速 demo** | P1 |
| **Re-Ranking 精排** | RRF 融合后直接 Top-K，无二次排序（7.1 P-07） | ① 轻量：bge-reranker-base（本地，`FlagEmbedding`）对 RRF Top-20 精排取 Top-5；② 云端：Cohere Rerank / OpenAI 兼容接口；③ 插在 `hybrid_search` 的 RRF 之后（[retriever.py L218-229](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L218)），做成可开关的 `reranker=None` 参数，默认关（保零成本） | P1 |
| **LLM 路由替代关键词路由** | 关键词路由硬编码（7.1 P-08），router.py 注释已预留升级口（[router.py L156](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L156)） | ① 双轨方案：关键词打分保底（快速通道），低置信度（confidence < 阈值）时才升级 LLM 路由——**兼顾速度与准确率**；② LLM 输出用 JSON 结构化（intent/confidence/entities），校验后落入 RoutingDecision；③ 保留现有 `recognize_intent` 接口不变，替换内部实现，测试兼容 | P2 |

## 8.3 中长期演进（架构级）

| 演进方向 | 现状基线 | 目标态 | 关键动作 |
|---------|---------|--------|---------|
| **双执行面收敛** | Python 运行时（src/labqa/）+ OpenCode Agent 定义（.opencode/agent/）各自实现意图/检索逻辑，共享知识库但规则漂移（7.1 P-01） | 单一事实源：Agent 定义只做"提示词与工具调用编排"，检索与路由统一走 Python 侧 API | ① 把 Agent 的意图规则抽取为共享 YAML/JSON 配置，Python 与 Agent 定义共同消费；② 或在 OpenCode agent 中用 command 调用 `main.py ask` 作为统一后端；③ 在 ARCHITECTURE.md 新增 ADR 记录收敛决策 |
| **RAG 评估体系** | 仅 18 个单元测试（路由/RRF 算法级），无检索质量指标、无评测集 | 检索质量可量化、回归可拦截 | ① 建评测集：从知识库 8 个文档人工标注 30-50 条 `(query, expected_doc, category)`；② 指标：Recall@K（命中率）、MRR、Faithfulness（生成内容忠于检索片段）；③ 关键参数（RRF_K、CHUNK_SIZE、混合阈值 50%）进评测回归门禁 |
| **可观测性** | DEBUG 打印（router/generator 的 verbose 分支）+ CLI 本地日志；飞书侧无运行指标 | 生产可排查：每次问答留痕 | ① 结构化日志（JSON lines）落盘，字段：query/intent/scores/检索耗时/LLM 耗时/响应；② 统计埋点：每日问答量、意图分布、RRF 降级率（BM25 空结果次数）、LLM 失败率；③ 索引状态（文档数/版本）与知识库变更联动（convert.sh 入库后自动重建索引的通知） |
| **权限与多租户** | 无鉴权：飞书群内任何人可问全库 | 按角色可见性过滤（如"设备密码"仅管理员可见） | ① 检索层 metadata 增加 `visibility` 字段，混合检索结果按请求者角色过滤（复用 category 过滤机制）；② 飞书消息带 open_id，建用户-角色映射；③ 内部接口先行，不做复杂 OAuth |
| **LLM 分级路由** | 全量请求走同一 LLM（ollama 或 deepseek-chat） | 简单查询本地小模型、复杂推理云端大模型 | 基于 Router 意图 + 查询长度/置信度做分级；在 generator 的 `create_llm` 处扩展 `route_llm(intent)`（[generator.py L99-118](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99)），成本与延迟双降 |
| **知识库更新链路闭环** | 手工：raw-docs → convert.sh → ingest；文档变更与索引更新割裂 | 文件变更自动触发索引重建 + 变更通知 | ① convert.sh 成功后自动调 `main.py ingest`（增量模式：按文件 mtime 只重嵌入变更文档）；② 飞书群通知"知识库已更新 N 篇文档"；③ 长期可接飞书云文档/多维表格 API 做自动同步 |

> **规划小结**：短期（D1-D8）聚焦"文档债 + 依赖 + 误路由"止血，成本低、收益立竿见影；中期（8.2 的 P0/P1 项）补齐多轮对话与 Web UI，把系统从"能用的 RAG"推向"好用的产品"；长期（8.3）围绕"双执行面收敛 + 评估体系 + 可观测性"把项目升级为可持续演进、可量化验证的工程系统。


---
# 第九部分：面试问题智能生成 + 附录：自检记录与使用建议

> 本部分针对 LabQA-RAG v2.0（实验室智能问答系统）定制 20 道高频面试问题（10 维度 × 2 题），
> 每题回答均包含结构化要点 + 项目源码链接（GitHub 格式，行号为主控核验值）+ 联网检索到的外部参考来源。
> 问题全部结合本项目真实场景（GPU 设备查询、混合意图、飞书集成、双轨 LLM、三级降级等），非通用八股。

## 9.1 维度拆解表

| # | 面试维度 | 项目具体信息（提取来源） | 检索到的外部来源（2025-2026 高频题） |
|---|---|---|---|
| 1 | RAG 混合检索（FAISS+BM25+RRF）原理与选型 | `retriever.py` dense_search L19-52 / sparse_search L57-101 / hybrid_search L175-229；v1→v2 重构动机（commit 757df23） | 知乎《RAG大厂面试题汇总》、博客园《2026年RAG面试高频考点全解析》、smallyoung《RAG混合检索深度解析》 |
| 2 | RRF 融合算法数学原理（1/(k+rank)、k=60） | `retriever.py` reciprocal_rank_fusion L106-142；`config.py` 检索参数 L42-46；`tests/test_main.py` TestRRF L157 | 腾讯云《RRF倒数排序融合》、Elasticsearch RRF 官方文档、ParadeDB《What is Reciprocal Rank Fusion?》 |
| 3 | Multi-Agent 路由与意图识别（为什么不用 LLM、混合意图） | `router.py` recognize_intent L150-207 / DEVICE_KEYWORDS L35-44；orchestrator.py process L71-130；WORKFLOW.md 混合意图 L176-182 | 掘金《Agent系列（五）：意图识别与路由》、腾讯云《大模型意图识别节点》、DAMO《多意图路由与动态查询分发引擎》 |
| 4 | Agent category 过滤设计（精准度保障） | `agents/__init__.py` AGENT_REGISTRY L14-18；device_agent.py L10-15；knowledge_agent.py L9-14；`retriever.py` search_by_category L232-245 | 卡码《RAG优化思路》元数据过滤、dirjaker《RAG 系统设计》、CSDN《万字详解面试题库-LangChain 篇》 |
| 5 | 检索质量与评估（怎么测、防污染） | `tests/test_main.py` TestRRF L157 / TestRouter L89；`docs/TESTING.md`；索引运行时生成（ingest.py L200-219） | 卡码笔记《RAG评估体系：Recall/MRR/NDCG与RAGAS》、meko1《RAG评估（RAGAS 与指标体系）》、腾讯云《Recall@K、MRR、NDCG 三指标》 |
| 6 | 中文分块与分词优化（500/50、jieba、frontmatter） | `ingest.py` split_documents L86-110 / load_markdown_files L26-83 / build_bm25_index L147-197 | 掘金《RAG Chunking 全攻略（递归分块）》、51CTO《RAG 文本分块最佳实践》、掘金《混合检索实战教程》 |
| 7 | 飞书/IM 集成（WebSocket vs Webhook、免公网 IP、两层去重） | `feishu_ws.py` _is_duplicate L28-56 / start L93-180 / 19900 截断 L83；main.py mode_feishu L91 | Akamai《API、Webhook 与 WebSocket 区别》、博客园《飞书 .NET SDK 事件幂等性与去重》、阿里云《搭建 websocket 消息推送服务》 |
| 8 | 双轨 LLM 架构（Ollama vs 云端、成本权衡、降级） | `config.py` LLM_MODE L25 / LLM_API_KEY 三级回退 L32-35；`generator.py` OllamaChat L48-74 / create_llm L99-118；`.env.example` L6-16 | followbot《本地部署 vs API 调用：成本对比》、知乎《本地部署 vs 云端调用成本分析》、匠人学院《开源模型、本地部署与模型路由》 |
| 9 | 离线兜底与降级设计（LLM 不可用/索引未构建） | `cli.py` main L62-149；orchestrator.py _unknown_intent_response L132-165；ingest.py check_indexes L222-235；offline-mode 哨兵（AGENTS.md L75） | JavaGuide《高可用系统设计详解》、极客时间《从架构师角度回答容错降级》、JavaGuide《高可用系统设计面试题总结》 |
| 10 | LangChain 框架与工程化（索引持久化、双执行面、v1→v2 重构） | `ingest.py` 索引持久化 L125/L158；`.opencode/agent/lab-orchestrator.md` L96-123；docs/ARCHITECTURE.md ADR-002 L235；18 个测试用例 | CSDN《万字详解面试题库-框架篇（LangChain/LangGraph）》、博客园《2026年RAG面试高频考点全解析》、AgentGuide《RAG 面试题》 |

> 说明：外部来源均为 2026-08-08 联网检索所得的真实 URL（快照），面试前建议重新打开验证内容时效性。


## 维度 1：RAG 混合检索（FAISS + BM25 + RRF）原理与选型

**Q1. 用户问「GPU服务器在哪」，你的系统为什么能比纯向量检索更快更准地给出答案？为什么 v1 的关键词匹配会漏召回？**

- **场景走读**：用户问「GPU服务器在哪」，`router.py` 的 DEVICE_KEYWORDS（L35-44）命中「GPU」→ 意图 DEVICE → DeviceAgent（category="devices"）→ `hybrid_search` 双路检索。BM25 路对「GPU」做精确词匹配（jieba 分词后 IDF 权重高），FAISS 路对「GPU 服务器在哪」做语义匹配，两路互补。
- **v1 失败原因**：v1 是纯关键词匹配（commit `07bb9b1`），用户说「服务器」文档写「GPU-A100-01」就漏召回；v2 改混合检索后语义路兜底（commit `757df23` 全面重构）。
- **为什么不能只用向量**：精确设备名（GPU-A100-01）、型号、编号是向量检索的结构性短板（语义相近≠字面命中）；而 BM25 不懂同义改写。混合检索 = 稠密（语义）+ 稀疏（精确）互补，是 2025-2026 生产推荐形态。
- **降级设计**：BM25 无结果时自动降级为纯向量检索（`retriever.py` L212-216），保证用户永远拿到结果而非空集。
- 项目链接：`[retriever.py hybrid_search](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175)`、`[retriever.py dense_search](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L19)`、`[retriever.py sparse_search](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L57)`
- 外部来源：[知乎：RAG大厂面试题汇总（纯向量检索三个致命问题）](https://zhuanlan.zhihu.com/p/2029999895302628181) / [博客园：2026年RAG面试高频考点全解析：从BM25到Self-RAG](https://www.cnblogs.com/ycfenxi/p/20057801) / [smallyoung：RAG 混合检索深度解析：BM25 + 向量检索 + RRF](https://www.smallyoung.cn/docs/028-RAG%E6%B7%B7%E5%90%88%E6%A3%80%E7%B4%A2%E4%B8%8ERRF%E7%AE%97%E6%B3%95%E6%B7%B1%E5%BA%A6%E8%A7%A3%E6%9E%90)

**Q2. 「自动驾驶测试车」这类精确设备名和「路测遇到障碍物怎么处理」这种语义问题，为什么单路检索必然翻车？你用什么指标证明混合检索有效？**

- **精确名场景**：「自动驾驶测试车」是 `devices/自动驾驶测试车.md` 的标题，BM25 对标题词做精确命中，FAISS 却可能把它和「自动驾驶感知路测项目」混在一起（语义相似但实体不同）。
- **语义场景**：「路测遇到障碍物怎么处理」文档里可能写的是「行车安全避障流程」，字面零重合 → BM25 完全漏掉，FAISS 靠语义向量兜住。
- **实现分工**：dense_search（FAISS 余弦/内积）负责语义泛化；sparse_search（BM25 + jieba）负责术语/编号精确匹配；两者 top-k 各自召回后再由 RRF 合并去重，而不是简单拼接。
- **证明方式**：`tests/test_main.py` TestRRF（L157）+ TestRouter（L89）覆盖「同一文档双路都命中时排名上升」的融合正确性；`docs/TESTING.md` 提供 8 大节手动测试清单，含混合意图、精确名、同义改写等场景用例。
- 项目链接：`[retriever.py hybrid_search](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L175)`、`[tests/test_main.py TestRRF](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L157)`
- 外部来源：[掘金：混合检索（稀疏+稠密）实战教程](https://juejin.cn/post/7613230154211721262) / [博客园：FAISS、HNSW、BM25 向量检索引擎选型（10k 条实测表）](https://www.cnblogs.com/peacemaple/p/19745520)

## 维度 2：RRF 融合算法数学原理

**Q3. 你的 `reciprocal_rank_fusion` 为什么用 `1/(k+rank)` 而不是把 FAISS 分数和 BM25 分数加权平均？k=60 是怎么定的？**

- **量纲不可比**：FAISS 输出的是向量距离/相似度（0~1 或负无穷~0），BM25 输出的是词频加权分（可到几十），两者加权平均需要归一化、还要调权重；RRF 只看**排名**，天然免疫量纲差异——这是面试官最想听的核心。
- **公式**：`RRF_score(d) = Σ 1/(k + rank_i(d))`，文档 d 在第 i 路检索中的排名越靠前贡献越大；只有一路命中的文档分数天然低于双路命中的文档，实现「取长补短 + 去重」。
- **k 的作用**：平滑常数，控制「排名靠后文档」的贡献衰减速度。k 越小，前排优势越大、长尾干扰越强；k=60 是 RRF 论文与 Elasticsearch 8.8+ 官方实现的业界默认值，社区共识「在 20~80 之间调参」。
- **本项目落地**：`config.py` 检索参数（L42-46）集中管理 k 与 top-k；`reciprocal_rank_fusion`（L106-142）实现时用 dict 累计分数 + 排序，TestRRF（L157）用「双路命中 vs 单路命中」用例锁定行为，防止未来重构回归。
- 项目链接：`[retriever.py reciprocal_rank_fusion](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106)`、`[config.py 检索参数](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L42)`
- 外部来源：[腾讯云：大模型 RAG 中 RRF 是什么（含数学推导与示例）](https://cloud.tencent.com/developer/article/2638995) / [Elasticsearch 官方：Reciprocal rank fusion（k 默认 60）](https://www.elastic.co/docs/reference/elasticsearch/rest-apis/reciprocal-rank-fusion) / [ParadeDB：What is Reciprocal Rank Fusion?](https://www.paradedb.com/learn/search-concepts/reciprocal-rank-fusion)

**Q4. RRF 会算错吗？你怎么验证融合结果「两个引擎都说好」的文档确实排在前面？如果 k 设成 0 或 1 会发生什么？**

- **双路命中优势验证**：设文档 D 在 FAISS 路 rank=1、BM25 路 rank=1，k=60 时 D 得分 ≈ 1/61 + 1/61 ≈ 0.0328；而只在一路 rank=1 的文档只有 0.0164——正好差一倍。TestRRF 用断言锁住这类排序性质。
- **k 的边界**：k→0 时公式退化为 `1/rank`，rank=1 的文档分是 rank=2 的两倍，前排文档碾压式胜出，长尾文档几乎无机会；k 过大会让所有文档分数趋同（1/k 主导），融合失去区分度。k=60 是「前排优势与尾部容忍」的平衡点。
- **为什么不用加权**：加权方案（如 0.6×dense + 0.4×sparse）需要为每个查询调权重，且分数分布随语料漂移；RRF 零参数、零训练、可解释，是工业界混合检索融合的默认选择（ES、Weaviate、LangChain EnsembleRetriever 均支持）。
- **测试即验证**：本项目 18 个 pytest 用例中 TestRRF（L157）专门覆盖融合逻辑，`docs/TESTING.md` 列出手动验证步骤，把「算法正确性」变成可回归的工程资产。
- 项目链接：`[tests/test_main.py TestRRF](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L157)`、`[retriever.py reciprocal_rank_fusion](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L106)`
- 外部来源：[腾讯云：混合检索 RAG 多路召回 + RRF 融合](https://developer.cloud.tencent.com/article/2710906?policyId=1003) / [知乎：RRF（Reciprocal Rank Fusion）排序融合](https://zhuanlan.zhihu.com/p/1914270914654237406)


## 维度 3：意图识别与 Multi-Agent 路由

**Q5. 你的系统为什么用关键词打分做意图识别，而不是让 LLM 来判断？「轻量级前端 + 专业后端」的分工是什么？**

- **成本与延迟**：`router.py` 的 `recognize_intent`（L150-207）是纯规则打分：`_score_intent`（L132-147）遍历 DEVICE/PROJECT/KNOWLEDGE 三组关键词（L35-69），零 LLM 调用、零网络开销、毫秒级返回；如果用 LLM 做路由，每一次问答多一次推理（延迟 + 成本），且 LLM 判断可能不稳定。
- **确定性**：规则打分是确定性的——同样的输入永远同样的路由，可单测（TestRouter L89）；LLM 路由有温度随机性，且需要 prompt 调优。
- **分工设计**：Router 是「轻量级前端」只做分类，Agent 是「专业后端」做 RAG 检索 + LLM 生成（`orchestrator.py` process L71-130 按 AGENT_MAP 分发给对应 Agent）；把「要不要花钱」的决策交给免费规则，把钱花在检索和生成上。
- **升级路径**：README 已注明未来可用 LLM 路由替代关键词路由——20 人实验室意图有限（3 类），规则足够；意图变多、表达变杂时再上 LLM，这是刻意的演进式设计。
- 项目链接：`[router.py recognize_intent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L150)`、`[router.py _score_intent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L132)`、`[orchestrator.py process](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L71)`
- 外部来源：[掘金：Agent系列（五）意图识别与路由——让 Agent 听懂用户在说什么](https://juejin.cn/post/7643818760442363910) / [腾讯云：大模型意图识别节点（单选/多选模式设计）](https://cloud.tencent.com/document/product/1759/115747)

**Q6. 用户问「王五在忙什么？GPU 用完了吗」——这是一个查询还是两个查询？你的混合意图自动发现是怎么实现的？**

- **问题本质**：一句口语同时含两个意图——「王五在忙什么」是 PROJECT/人员状态类、「GPU 用完了吗」是 DEVICE 状态类。单纯路由到任一 Agent 都会丢一半信息，这是真实实验室场景的高频形态。
- **自动发现机制**：`recognize_intent` 对每个意图分别打分，当**次要意图分数 ≥ 主意图分数的 50%** 时判定为 MIXED（`router.py` L178-181）；MIXED 走多 Agent 聚合路径，而不是写死「如果包含两个词就……」，阈值可调、可解释。
- **为什么用分数比而不是硬编码组合**：用户表达千变万化（「显卡还够吗」「A100 有空的吗」「老王那项目进展如何」），关键词组合穷举不完；打分制天然对「部分命中」敏感，50% 阈值就是「明显次意图」的量化定义。
- **完整走读**：`docs/WORKFLOW.md` L176-182 专门讲解混合意图打分，L186-416 有完整示例走读；`orchestrator.py` process（L71-130）处理多意图分支，`_unknown_intent_response`（L132-165）兜底无法识别的输入。
- 项目链接：`[router.py 混合意图判定](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L178)`、`[docs/WORKFLOW.md 混合意图](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L176)`、`[orchestrator.py _unknown_intent_response](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L132)`
- 外部来源：[DAMO：AI Agent 架构核心——如何构建多意图路由与动态查询分发引擎](https://damodev.csdn.net/6960aa90ea53844658f5a31d.html) / [代码随想录：2026 最全大模型面经汇总（Agent/RAG 面试题）](https://programmercarl.com/qita/0022.llminterview.html)

## 维度 4：Agent category 过滤设计

**Q7. DeviceAgent 只搜 `devices/`、KnowledgeAgent 搜全量，这个 category 过滤解决了什么问题？为什么不做全库检索？**

- **问题**：用户问「A100 显存多大」如果全库检索，`projects/LLM对齐优化项目.md` 里提到 A100 的段落也可能被召回，噪声混入上下文会干扰 LLM 生成（检索信噪比下降）。
- **设计**：三个 Agent 各自声明检索范围——`DeviceAgent`（category="devices"，device_agent.py L10-15）、`ProjectAgent`（category="projects"，project_agent.py L9-14）、`KnowledgeAgent`（category=None 全量，knowledge_agent.py L9-14）；`retriever.py` 的 `search_by_category`（L232-245）在混合检索结果上做类别过滤。
- **本质**：这是 RAG 的**元数据过滤（Metadata Pre-filtering）**——先用路由把问题分到「该搜哪个目录」，再在目录内做精检索，等于把「全库 Top-5」变成「相关库 Top-5」，精度和效率双赢。
- **为什么不全部用 None**：KnowledgeAgent 检索 knowledge/（论文笔记）+ guides/（新人指南）是开放域；设备/项目是强实体域，过滤能防止「王五的项目」检索到「GPU 状态」的串扰。
- 项目链接：`[device_agent.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/device_agent.py#L10)`、`[knowledge_agent.py](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/knowledge_agent.py#L9)`、`[retriever.py search_by_category](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L232)`
- 外部来源：[卡码笔记：RAG 优化思路（Query 改写、混合检索、元数据过滤）](https://notes.kamacoder.com/llm/app/rag_evaluation.html) / [dirjaker：LLM & Agent 面试题全集 05-RAG 系统设计（元数据设计章节）](https://dirjaker.github.io/llm_agent_interview/02-%E6%A0%B8%E5%BF%83%E6%8A%80%E8%83%BD/05-RAG%E7%B3%BB%E7%BB%9F%E8%AE%BE%E8%AE%A1.html)

**Q8. 现在要新增一个「论文阅读」意图和对应 Agent，你需要改哪几处？这暴露了什么工程债？**

- **改动清单（5 处）**：① `router.py` 新增 `PAPER_KEYWORDS`（L58-69 附近）并注册到 AGENT_MAP（L72-76）与 INTENT_CATEGORY_MAP（L79-83）；② 新增 `src/labqa/agents/paper_agent.py`（继承 BaseAgent，声明 category）；③ `agents/__init__.py` 的 AGENT_REGISTRY（L14-18）注册新类；④ `generator.py` 的 AGENT_PROMPTS（L222-274）增加论文专用 prompt；⑤ 补充测试用例（TestRouter/TestAgents）。
- **工程债**：`AGENTS.md` L77 明确指出「关键词路由是硬编码——新增意图需同时改 router.py 关键词 + agent 关键词提取方法」；`router.py` 的 `extract_keywords`（L86-129）中每个 Agent 还有各自的提取逻辑，改一处漏一处就会路由漂移。
- **缓解方式**：AGENT_REGISTRY 用注册表模式集中管理 Agent 实例化（get_agent L21-25），新增 Agent 不需要改 orchestrator 主流程——这是模板方法 + 注册表的组合，扩展开销被控制在「新增文件 + 注册一行」。
- **未来演进**：README 列出的「LLM 路由替代关键词路由」正是为了治本——把硬编码关键词换成 LLM 分类或可配置规则文件。
- 项目链接：`[router.py AGENT_MAP](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/router.py#L72)`、`[agents/__init__.py AGENT_REGISTRY](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/__init__.py#L14)`、`[agents/base.py BaseAgent](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L15)`
- 外部来源：[小林 coding：2026 最全 AI 大模型面试题（Multi-Agent 协作与动态切换）](https://xiaolincoding.com/project/xiaolinnote.html) / [CSDN：万字详解面试题库-框架篇（LangChain/LangGraph 18 题）](https://blog.csdn.net/vaemusicsky/article/details/160689465)


## 维度 5：检索质量与评估

**Q9. 你怎么量化「检索质量」？RAG 评估为什么必须拆成检索侧和生成侧？你的项目怎么测 RRF 和路由的正确性？**

- **两层评估框架**：检索侧看 Recall@K（找没找到）、Precision@K（找得干不干净）、MRR（第一个对的排多前）、NDCG（排序质量）；生成侧看 Faithfulness（忠实度）、Answer Relevancy（相关性）、Context Recall/Relevancy（上下文利用率）。只看最终答案无法定位问题是检索坏还是生成坏。
- **本项目实践**：`tests/test_main.py` 按模块分 TestConfig/TestIngest/TestRouter/TestRRF/TestPrompts/TestGenerator/TestAgents 共 18 用例；TestRRF（L157）验证融合排序，TestRouter（L89）验证意图打分与 MIXED 阈值，TestAgents（L284）验证 category 过滤——把「检索链路」变成可回归的单测资产。
- **手动基线**：`docs/TESTING.md`（152 行）提供 8 大节手动测试清单，覆盖精确名、同义改写、混合意图、未知意图等真实问题形态，弥补自动化用例覆盖不足。
- **为什么重要**：没有评估的优化是「拿着锤子找钉子」——换了 Embedding、调了 k，效果升了降了不知道，面试官问「你怎么验证优化有效」就露馅。
- 项目链接：`[tests/test_main.py TestRRF](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L157)`、`[tests/test_main.py TestRouter](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L89)`、`[docs/TESTING.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/TESTING.md)`
- 外部来源：[卡码笔记：RAG评估体系——检索召回率、MRR、NDCG 与 RAGAS 框架详解](https://notes.kamacoder.com/llm/app/rag_evaluation.html) / [meko1：RAG 评估（RAGAS 与指标体系）](https://meko1.github.io/llm-interview-guide/rag/rag-evaluation) / [腾讯云：Recall@K、MRR、NDCG 三指标一文讲透](https://cloud.tencent.com/developer/article/2658116)

**Q10. 你的索引是用 `.opencode/context/` 真实知识库在运行时构建的，测试却要用固定 fixtures——怎么防止「测试集污染」和「测试数据与真实数据不一致」？**

- **隔离原则**：知识库（`.opencode/context/` 8 个文档）是**运行数据**，测试用**固定构造的文档**（TestIngest L39 构造 markdown/frontmatter 样本），两者物理隔离——测试不依赖真实知识库内容，避免「知识库更新导致测试漂移」。
- **防污染的另一层**：测试只验证**机制**（分块边界、RRF 排序性质、路由打分），不验证「特定文档的检索命中」——后者属于人工回归清单（TESTING.md），前者才是自动化该管的。
- **索引运行时生成**：`ingest.py` build_indexes（L200-219）从知识库生成 `faiss_index/` + `bm25_index.pkl`，`check_indexes`（L222-235）做存在性校验；测试环境不建真实索引，直接 mock/构造小型索引验证检索逻辑。
- **面试加分点**：能说出「评估集要代表真实分布、且与训练/索引数据隔离」——本项目的隔离策略对应 RAG 工程里的「黄金数据集（Golden Dataset）」思想：核心场景 50-200 条人工标注即可建立可信基线。
- 项目链接：`[tests/test_main.py TestIngest](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L39)`、`[ingest.py build_indexes](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L200)`、`[ingest.py check_indexes](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L222)`
- 外部来源：[Hugging Face Cookbook：RAG 评估（合成评估集 + LLM-as-a-judge）](https://huggingface.co/learn/cookbook/zh-CN/rag_evaluation) / [掘金：2026 年最新 AI agent 面试（06）RAG 文档与检索（Embedding 用自有数据 Hit@K，别盲信 MTEB）](https://juejin.cn/post/7662645689650511914)

## 维度 6：中文分块与分词优化

**Q11. 为什么选 RecursiveCharacterTextSplitter 且参数是 chunk_size=500 / chunk_overlap=50？中文文档分块最大的坑是什么？**

- **分隔符链**：RecursiveCharacterTextSplitter 按优先级递归切分——先按段落 `\n\n`、再 `\n`、然后中文句号 `。！？`、最后按字符兜底；它「尽可能保留语义结构」：先试大段落边界，切不动才降级到句子/字符，是生产环境默认推荐策略。
- **为什么 500/50**：chunk 太碎语义不足（向量表达弱）、太大信噪比恶化（一块含多个主题）；500 字符 + 10% overlap（50 字符）是中文场景常用甜点区间，overlap 兜住跨块语义截断（比如「GPU 显存不足导致……」被切到两块时，50 字符重叠保证关键信息至少完整出现在一块里）。
- **中文最大的坑**：中文无空格分词，按 token 数硬切极易从词中间截断（「向量检索」被切成「向量检」「索」）；本项目 markdown 源有 `##` 标题结构，递归分块沿段落边界走，天然比固定大小分块保留更多结构。
- **实现位置**：`ingest.py` split_documents（L86-110）统一做 frontmatter 剥离 + 递归分块；分块参数在 `config.py` 检索参数（L42-46）集中配置，改参数不用改代码。
- 项目链接：`[ingest.py split_documents](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L86)`、`[ingest.py load_markdown_files](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L26)`
- 外部来源：[掘金：RAG Chunking 全攻略——5 种策略（递归分块分隔符优先级详解）](https://juejin.cn/post/7631986773392736256) / [51CTO：LLM-RAG 的文本分块最佳实践：原理、细节与工程化落地](https://blog.51cto.com/u_15239532/14538510)

**Q12. 为什么 BM25 需要 jieba 分词而 FAISS 不需要？frontmatter 手写解析有什么风险？**

- **分词必要性**：BM25 是词袋模型——先切词（tokenize）再统计 TF/IDF；中文不分词，「GPU显存不足」整串当一个词，用户问「显存」就匹配不上。jieba 把「GPU/显存/不足」拆开，倒排索引才能命中。FAISS 是向量模型，分词已内化在 Embedding 里，不需要显式分词。
- **实现细节**：`build_bm25_index`（ingest.py L147-197）对每个 chunk 做 jieba 分词后建 BM25Okapi 索引；查询侧 `sparse_search`（retriever.py L57-101）同样 jieba 分词后再打分，保证训练/查询词表一致（不一致是中文 BM25 最容易踩的坑）。
- **frontmatter 风险**：`load_markdown_files`（L26-83）手写解析 `---` 头（ingest.py L51-59 附近），只支持简单 `key: value`；嵌套结构（列表、多行值）会解析错位，把 metadata 当正文或正文当 metadata——这是已知工程债，README 与 AGENTS.md 均已记录，未来应换完整 YAML 解析。
- **索引一致性**：FAISS 索引与 BM25 索引必须基于**同一批 chunk**（build_indexes L200-219 保证两路共享 split 结果），否则 RRF 融合时两路文档 ID 对不上——这是混合检索的隐性正确性前提。
- 项目链接：`[ingest.py build_bm25_index](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L147)`、`[ingest.py frontmatter 解析](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L51)`、`[retriever.py sparse_search](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L57)`
- 外部来源：[掘金：混合检索（稀疏+稠密）实战教程（jieba 分词 + 去停用词预处理）](https://juejin.cn/post/7613230154211721262) / [CSDN：RAG 混合检索核心 BM25 算法面试通关笔记](https://blog.csdn.net/m0_56800366/article/details/158573200)


## 维度 7：飞书/IM 集成与消息去重

**Q13. 飞书接入为什么选 WebSocket 长连接而不是 Webhook 回调？「免公网 IP」在实验室场景意味着什么？**

- **WebSocket vs Webhook 核心差异**：Webhook 是平台 → 你的公网 HTTP 地址主动 POST，**接收端必须有公网可达 IP** 且要处理签名校验、超时重试；WebSocket 是**你的程序主动连出去**（出站连接），服务器在实验室/内网也能用。
- **免公网 IP 的价值**：实验室服务器通常在 NAT 后面、没有固定公网 IP、无法开放端口——Webhook 直接不可用（或者要买公网服务器/内网穿透）；飞书 WebSocket 长连接（`feishu_ws.py` start L93-180）是出站连接，天然穿透 NAT，零额外成本。
- **实时性对比**：WebSocket 毫秒级双向、需要心跳保活（断线重连 on_message L124-160 处理）；Webhook 秒级单向、实现简单但依赖公网。Akamai 的对比结论：API 适合 CRUD、Webhook 适合实时事件、WebSocket 适合双向通信——本项目只需要「收消息→回消息」，双向长连接是顺手的选择。
- **兼容性细节**：`main.py` mode_feishu（L91）启动 WS 长连接；v1 遗留的 `main.py webhook` 别名实际也启动 WS（兼容别名），避免老用户习惯被打破。
- 项目链接：`[feishu_ws.py start](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L93)`、`[main.py mode_feishu](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L91)`
- 外部来源：[Akamai：API、Webhook 与 WebSocket 之间的区别是什么](https://www.akamai.com/zh/glossary/what-is-an-api-vs-webhook-vs-websocket) / [阿里云：搭建 websocket 消息推送服务必须要考虑的几个问题](https://developer.aliyun.com/article/759986) / [掘金：Webhook vs WebSocket 选型对比（企业云盘集成实战）](https://juejin.cn/post/7644484272205545513)

**Q14. 飞书 SDK 会重试推送，你的两层去重（message_id 集合 + 30 秒内容哈希窗口）分别防什么？19900 字符截断是什么？**

- **为什么需要去重**：飞书事件推送是 AT-LEAST-ONCE 语义——网络波动（确认包丢了）、服务重启（内存清空）、断线重连都可能导致同一条消息被推两次；不去重，用户会收到两条一模一样的回答。
- **第一层 message_id**：`_is_duplicate`（feishu_ws.py L28-56）用已处理 message_id 集合精确去重——同一消息 ID 只处理一次，O(1) 判重。
- **第二层 30 秒内容哈希**：message_id 在「用户编辑消息重新发送」场景会变，但内容相同；用内容哈希 + 30 秒窗口兜住「不同 ID 同内容」的重复（比如 SDK 重试时重新生成 ID），两层互补：ID 层防「同 ID 重推」，哈希层防「换 ID 重推」。
- **19900 截断**：`_send_message`（L78-90，L83 截断）——飞书单条消息长度有限制，LLM 长回答截到 19900 字符，避免发送被拒；截断而非丢弃，保证用户至少拿到主体内容。
- **工程类比**：这与消息队列消费幂等是同一类问题——「不丢和不重是矛盾的，消息重复有解决方案，消息丢失更麻烦」，所以补偿式去重是消费端标配。
- 项目链接：`[feishu_ws.py _is_duplicate](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L28)`、`[feishu_ws.py _send_message](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L78)`、`[feishu_ws.py on_message](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/feishu_ws.py#L124)`
- 外部来源：[博客园：飞书 .NET SDK 事件处理的幂等性与去重机制（WebSocket SeqID/Webhook Nonce 三层防护）](https://www.cnblogs.com/mudtools/p/19469184) / [阿里经典面试题：消息队列的消费幂等性如何保证](https://objcoding.com/2021/07/27/message-dedup/)

## 维度 8：双轨 LLM 架构（本地 + 云端）

**Q15. `LABQA_LLM_MODE=ollama|cloud` 双轨设计解决了什么问题？本地免费为什么还要保留云端？**

- **双轨动机**：实验室预算敏感 + 数据敏感（GPU 使用情况、项目进度不想外传）→ Ollama 本地跑（零 token 成本、数据不出内网）；但本地模型质量/速度可能不够（复杂解释、长文总结）→ 云端 DeepSeek/OpenAI 兜质量。`config.py` LLM_MODE（L25）+ `generator.py` create_llm（L99-118）按模式组装不同客户端。
- **成本权衡**：本地部署 = 高固定成本（GPU 已有）+ 近零边际成本；云端 = 零前期投入 + 线性 token 费用。20 人实验室调用量小，Ollama 边际成本为 0 是「默认免费」的底气；云端只在需要质量时启用，把费用控制在几元/月量级。
- **接口一致性**：`generator.py` 的 OllamaChat（L48-74）与云端 ChatOpenAI 都走 OpenAI 兼容协议，业务代码只换 base_url/key 不换逻辑——这是「本地↔云端切换只改一行配置」的工程红利（匠人学院的观点：路由层只是决定请求发给哪个 base_url）。
- **三级 Key 回退**：`LLM_API_KEY` 按 LABQA_LLM_API_KEY → DEEPSEEK_API_KEY → OPENAI_API_KEY 回退（config.py L32-35），换供应商不动 .env 变量名，兼容历史配置。
- 项目链接：`[config.py LLM_MODE](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L25)`、`[generator.py create_llm](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L99)`、`[generator.py OllamaChat](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L48)`、`[.env.example](https://github.com/BLYHFL/labqa-rag/blob/main/.env.example#L6)`
- 外部来源：[followbot：本地部署 vs API 调用——成本、性能与体验全面对比（2026-08 更新）](https://followbot.cn/local-llm-vs-api) / [知乎：本地部署 vs 云端调用——大模型使用成本的全链路对比](https://zhuanlan.zhihu.com/p/2053072912849036274) / [匠人学院：开源模型、本地部署与模型路由（下放清单 + 降级路径）](https://jiangren.com.au/learn/ai-engineer/open-weight-models-local-deploy)

**Q16. Ollama 挂了或者没装，系统会怎样？双轨之间有没有自动降级链路？**

- **降级链设计**：`create_llm`（generator.py L99-118）按 mode 构造 LLM；Ollama 模式依赖本地服务（OLLAMA_HOST），服务不可达时调用抛异常 → 上层 `generate_answer`（L123-199）捕获异常走兜底回答（"知识库索引正常，但 LLM 服务不可用"之类的提示），而不是崩溃。
- **降级 vs 故障转移**：当前实现是**故障降级**（失败 → 明确提示），不是自动故障转移（Ollama 挂 → 自动切云端）；后者需要健康探测 + 重试策略，是 README 待优化项——面试时能主动说出「这里只做了降级没做自动切换，是刻意的 v2 边界」是加分回答。
- **配置即开关**：.env 的 LABQA_LLM_MODE 一行切换双轨；LLM_API_KEY 三级回退让 cloud 模式换 Key 不换代码；手写 .env 加载器（main.py L26-L41）+ python-dotenv 双实现，保证任何环境下变量都能加载。
- **对用户体验**：实验室场景的取舍——「回答质量低」可接受（本地模型），「服务挂掉没反馈」不可接受，所以降级路径确保永远有响应（未知意图/错误都有兜底文案，orchestrator.py _unknown_intent_response L132-165）。
- 项目链接：`[generator.py generate_answer](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/generator.py#L123)`、`[config.py LLM_API_KEY 三级回退](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/config.py#L32)`、`[main.py .env 加载器](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L26)`
- 外部来源：[PromptQuorum：本地 LLM vs 云端 API 2026——隐私与成本全面对比](https://www.promptquorum.com/zh/local-llms/local-llms-vs-cloud-apis) / [CSDN：对比评测 Ollama vs 云端 API（本地模型效率优势）](https://blog.csdn.net/SunstoneLion34/article/details/156074953)


## 维度 9：离线兜底与降级设计

**Q17. 索引没构建、LLM 不可用、飞书断连——三种故障下系统分别怎么表现？「三级降级」的设计思路是什么？**

- **三级降级全景**（核心卖点，面试重点展开）：
  1. **索引未构建**：`ingest.py` check_indexes（L222-235）启动时校验 `faiss_index/` 与 `bm25_index.pkl`，缺失则提示先跑 `python3 main.py ingest`（main.py mode_ingest L52）；CLI 的 `/ingest` 命令（cli.py L123）可随时重建——系统不会带病运行。
  2. **LLM 不可用**：`create_llm`/`generate_answer` 捕获调用异常 → 返回兜底文案（见 Q16），检索层照常工作（`/stats` L110、`/reload` L120 等运维命令不依赖 LLM）。
  3. **飞书断连**：WS 长连接断线自动重连（feishu_ws.py on_message L124-160 处理重连逻辑）；离线期间用户消息暂无法应答，但服务进程不死。
- **设计原则**：降级 = 有损容错——核心功能（问答）在依赖缺失时明确失败并给指引，运维功能（stats/ingest/reload）永不依赖 LLM；这是「故障不可避免，把故障当作架构的一部分」的工程观。
- **CLI 永远是最后兜底**：`main.py` mode_cli（L46）提供纯本地交互（cli.py main L62-149，含 /help L108 /stats L110 /reload L120 /ingest L123 /debug L131），GUI/飞书全挂也能命令行问答——多入口架构本身就是容错。
- 项目链接：`[ingest.py check_indexes](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L222)`、`[cli.py main](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L62)`、`[main.py mode_cli](https://github.com/BLYHFL/labqa-rag/blob/main/main.py#L46)`
- 外部来源：[JavaGuide：高可用系统设计详解——限流熔断、降级容灾、缓存与灰度发布](https://javaguide.cn/high-availability/high-availability-system-design.html) / [JavaGuide：2026 最新高可用系统设计面试题总结](https://javaguide.cn/high-availability/high-availability-system-interview-questions.html) / [极客时间：从架构师角度回答系统容错、降级等高可用问题（熔断三态）](https://learn.lianglianglee.com/%E4%B8%93%E6%A0%8F/%E6%9E%B6%E6%9E%84%E8%AE%BE%E8%AE%A1%E9%9D%A2%E8%AF%95%E7%B2%BE%E8%AE%B2/16%20%20%E5%A6%82%E4%BD%95%E4%BB%8E%E6%9E%B6%E6%9E%84%E5%B8%88%E8%A7%92%E5%BA%A6%E5%9B%9E%E7%AD%94%E7%B3%BB%E7%BB%9F%E5%AE%B9%E9%94%99%E3%80%81%E9%99%8D%E7%BA%A7%E7%AD%89%E9%AB%98%E5%8F%AF%E7%94%A8%E9%97%AE%E9%A2%98%E5%90%97%EF%BC%9F.md)

**Q18. `LABQA_LLM_API_KEY="offline-mode"` 这个哨兵值是什么意思？未知意图和 LLM 拒绝回答时，系统怎么「优雅地说不知道」？**

- **offline-mode 哨兵**：CLI 离线模式（AGENTS.md L75 记录）——API Key 设为 `"offline-mode"` 时系统识别为「纯离线」：跳过 LLM 调用，只做检索并把检索到的原文片段直接返回（或返回未配置提示），保证在完全无网/无 Key 的实验室环境仍能用检索兜底。
- **未知意图兜底**：`recognize_intent` 三组关键词都打不出分 → 归为 UNKNOWN → `orchestrator.py` _unknown_intent_response（L132-165）返回引导文案（提示可问「设备」「项目」「知识库」）+ `prompts.py` UNKNOWN_INTENT_PROMPT（L81）定义生成措辞——把「答不上来」变成「引导用户」，而不是硬编答案。
- **三档「不知道」**：① 意图未知 → 引导文案；② 意图识别但检索无结果 → BaseAgent._no_result_response（L109-121）说明没找到；③ LLM 生成失败 → 兜底错误提示。每一档都有明确的用户体验，这是 RAG 幻觉治理的最朴素防线——**不硬编**。
- **调试入口**：CLI `/debug`（cli.py L131）打印路由/检索/生成全链路中间状态，故障时先看哪一档兜底被触发，再定位问题——可观测性设计让降级路径可排查。
- 项目链接：`[orchestrator.py _unknown_intent_response](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/orchestrator.py#L132)`、`[prompts.py UNKNOWN_INTENT_PROMPT](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/prompts.py#L81)`、`[cli.py /debug](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/cli.py#L131)`、`[agents/base.py _no_result_response](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/agents/base.py#L109)`
- 外部来源：[掘金：RAG 面试指南——如何减少幻觉（置信度控制、拒绝回答、兜底）](https://juejin.cn/post/7623685143834247168) / [知乎：RAG大厂面试题汇总——幻觉处理六种策略](https://zhuanlan.zhihu.com/p/2029999895302628181)

## 维度 10：LangChain 框架与工程化

**Q19. 索引为什么「已存在就不重建」？双执行面架构（Python 运行时 + OpenCode Agent 定义）为什么对一个 20 人实验室有价值？**

- **索引持久化**：`build_faiss_index`/`build_bm25_index` 先检查索引文件是否存在（ingest.py L125、L158），存在则直接加载（load_indexes retriever.py L147-172）——避免每次启动重新 Embedding 全部文档（本地 Embedding 慢且费时），同时保留「内容变更后手动 /ingest 重建」的显式更新路径；配合 `orchestrator.py` reload（L178-184）支持热加载新索引。
- **双执行面架构**（ADR-002，docs/ARCHITECTURE.md L235）：① Python 运行时 `src/labqa/` 给飞书机器人/CLI 用；② `.opencode/agent/`（lab-orchestrator + 3 个 subagents + commands/ + workflows/）让 OpenCode 这个 AI 编码助手在仓库里也用同一套知识库问答。两个执行面**共享 `.opencode/context/` 知识库**，知识只写一遍、多处消费。
- **为什么值得**：实验室文档（设备、项目、论文笔记）是团队公共资产——人类同事用飞书问、AI 编码助手在写代码时问、新人看新人指南，同一套知识库三处生效；这是「知识库单一事实来源」的工程红利，比多套文档维护省得多。
- **工程债并存**：双执行面代码有重复维护风险（AGENTS.md 记录），Python 逻辑与 OpenCode Agent 定义各自演化，需要定期对齐——面试中主动说出代价比只吹优点可信。
- 项目链接：`[ingest.py 索引持久化](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/ingest.py#L125)`、`[retriever.py load_indexes](https://github.com/BLYHFL/labqa-rag/blob/main/src/labqa/retriever.py#L147)`、`[.opencode/agent/lab-orchestrator.md](https://github.com/BLYHFL/labqa-rag/blob/main/.opencode/agent/lab-orchestrator.md#L96)`、`[docs/ARCHITECTURE.md ADR-002](https://github.com/BLYHFL/labqa-rag/blob/main/docs/ARCHITECTURE.md#L235)`
- 外部来源：[CSDN：万字详解面试题库-框架篇（LangChain/LangGraph——核心定位、组件、工程实践）](https://blog.csdn.net/vaemusicsky/article/details/160689465) / [dirjaker：LLM & Agent 面试题全集 05-RAG 系统设计（增量索引更新、异常监控）](https://dirjaker.github.io/llm_agent_interview/02-%E6%A0%B8%E5%BF%83%E6%8A%80%E8%83%BD/05-RAG%E7%B3%BB%E7%BB%9F%E8%AE%BE%E8%AE%A1.html)

**Q20. v1 是关键词匹配，v2 重构成混合检索 RAG——这次重构里你最值的决策和最想改的决策分别是什么？**

- **最值：先立评估再动手**：v2（commit `757df23`）重构同步落地 18 个 pytest 用例（TestConfig/TestIngest/TestRouter/TestRRF/TestPrompts/TestGenerator/TestAgents 7 个测试类）——先用单测锁住路由、分块、RRF 的正确性，再改实现，重构有安全网。
- **最值：领域先行拆分 Agent**：设备/项目/知识三 Agent + category 过滤，让「知识库目录结构 = 路由结构」——新文档进来按目录归类即可被正确检索，没有引入复杂编排框架。
- **最值：双轨 LLM + 离线兜底**：v2 一开始就设计 Ollama 默认、云端可选、CLI 兜底，实验室断网/没 Key 也能用——「先保证在任何环境下可用，再追求效果」。
- **最想改**：① 关键词路由硬编码（新增意图要改 5 处，AGENTS.md L77 自认工程债）→ 应早点做可配置规则或 LLM 路由；② 手写 frontmatter 解析（ingest.py L51-59）嵌套结构会坏 → 该用完整 YAML；③ 无 Reranker（RRF 后直接 Top-K）——README 已列扩展方向，检索精度天花板受限。
- **方法论沉淀**：`docs/ARCHITECTURE.md` 用 ADR（ADR-001 L221、ADR-002 L235）记录决策背景，`docs/WORKFLOW.md`（506 行）把问答流程写成可走读的教程——重构的「为什么」和「怎么用」都是仓库资产，新人能快速接手。
- 项目链接：`[tests/test_main.py](https://github.com/BLYHFL/labqa-rag/blob/main/tests/test_main.py#L284)`、`[docs/ARCHITECTURE.md ADR-001](https://github.com/BLYHFL/labqa-rag/blob/main/docs/ARCHITECTURE.md#L221)`、`[docs/WORKFLOW.md](https://github.com/BLYHFL/labqa-rag/blob/main/docs/WORKFLOW.md#L186)`
- 外部来源：[博客园：2026年RAG面试高频考点全解析（索引与检索生成、RRF 融合、向量库选型）](https://www.cnblogs.com/ycfenxi/p/20057801) / [GitHub：AgentGuide 04-interview 02-rag-questions（RAG 原理与工程化 20+ 问）](https://github.com/adongwanai/AgentGuide/blob/main/docs/04-interview/02-rag-questions.md)


---
# 附录：自检记录与使用建议

## A.1 结构校验说明

本 notebook 由多个 DocWriter 分节并行生成（各 section 文件），由主控组装为 `.ipynb` 并做 JSON 校验。组装完成后，建议在命令行执行一次最终校验：

```bash
python3 -c "import json; nb = json.load(open('项目全景说明书.ipynb')); print('OK', len(nb['cells']), 'cells')"
```

若输出 `OK N cells`（N 为全部单元格数）即通过；若报 JSONDecodeError，说明存在未转义的引号/换行，需回到对应 section 修复。

## A.2 使用与运行顺序建议

1. **索引构建先行**：首次使用必须先 `python3 main.py ingest` 构建 `faiss_index/` + `bm25_index.pkl`（main.py mode_ingest L52；索引未构建时系统会明确提示，见 check_indexes L222-235）。
2. **三种运行方式按环境选**：
   - 飞书机器人（推荐日常使用）：`python3 main.py feishu`（mode_feishu L91，WebSocket 长连接，免公网 IP）
   - CLI 交互：`python3 main.py`（mode_cli L46，内建 /help /stats /reload /ingest /debug）
   - 单次问答：`python3 main.py ask "问题"`（mode_ask L58）
3. **LLM 模式切换**：`.env` 中 `LABQA_LLM_MODE=ollama|cloud` 一行切换；无 Key/断网环境设 `LABQA_LLM_API_KEY="offline-mode"` 走纯离线检索兜底。
4. **知识库更新流程**：文档放入 `.opencode/context/` 对应目录（devices/projects/knowledge/guides）→ `python3 main.py ingest` 或 CLI `/ingest` 重建索引 → `/reload` 热加载（orchestrator.py reload L178-184）。
5. **测试**：`pytest tests/` 运行 18 个用例；手动回归按 `docs/TESTING.md` 8 大节清单执行。

## A.3 面试准备建议

1. **第九部分 20 题的使用方法**：每题先自己口头回答（要点 + 项目链接 + 外部来源），再对照本 notebook 检查遗漏；重点题是 Q1（为什么混合检索）、Q3（RRF 数学）、Q6（混合意图）、Q14（两层去重）、Q17（三级降级）——这 5 题最容易被追问细节。
2. **追问防御**：每个「为什么」都准备好「如果……会怎样」：如 k 从 60 改成 10 会怎样、Ollama 挂了要不要自动切云端、新增意图改几处、BM25 无结果怎么办（retriever.py L212-216 降级）。
3. **外部来源刷新**：附录引用的外部链接是 2026-08-08 检索快照，面试前 1-2 天重新打开验证内容与 URL 有效性，防止死链。
4. **诚实呈现工程债**：AGENTS.md 记录的硬编码路由、frontmatter 手写解析、无 Reranker、无 CI 等，面试时主动说出「已知局限 + 演进路径」比隐瞒更可信。

## A.4 文档信息

- 生成方式：对代码仓库全量扫描 + git 历史分析 + 联网检索高频面试题后自动生成
- 版本：v1.0；生成日期：2026-08-08
- 数据口径：源码行号以主控核验为准（context-bundle.md）；测试数量以 tests/test_main.py 实际为准
- 仓库：`https://github.com/BLYHFL/labqa-rag`（main 分支）
